# Qwen3-8B: Five-Fold Two-Head LoRA

Train five independent outer prompt folds and one full-data model from the official pretrained base. Each of two T4 workers handles three independent jobs; model weights are never split across GPUs. All jobs use three fixed epochs. Validation labels never select epochs, checkpoints, learning rates or blend weights.

The 24-layer backbone uses rank16/alpha32 LoRA on seven projection types. Eight normalized mean pools feed plain and prompt-centered heads. Prompt-aware triples, accumulated six at a time, preserve effective batch18. Each epoch covers every fitting answer, with explicitly recorded padding of incomplete triples and accumulation batches. The objective, dropout and clipping match the feasibility diagnostic. The full schedule adds10%warmup and cosine decay. AMP overflow retries use the same shared code.

This is an independent recipe inspired by the public24-layer checkpoint, not an exact reproduction of its unavailable training source. Public attribution adapters are forbidden as inputs. Before any worker imports PEFT, the exact reviewed smoke-v2 preflight runs in a fresh process and removes only incompatible unused torchao if necessary. Quantization is forbidden, no package is installed, and all core dependency versions must match the measured smoke.

Launch remains disabled until the notebook is rebuilt with the actual source-verified successful v2 run audit, its twelve successful-update report and its successful optional-dependency audit. Output hashes and helper/core versions are checked. The timing estimate must leave room for loading, validation and recorded overflow retries. Monitoring is split into30-minute cells, below the standard per-cell timeout, for at most ten hours.

After all six jobs finish, the five held-out feature banks are evaluated under the original sparse1/3/6 prompt pools. Lexical topic recovery and singleton fallback match the existing validation. A fixed50/50 log-probability mixture of plain and centered LoRA heads enters predeclared5%,10%and20% log pools with the calibrated Gemma baseline. These are descriptive comparisons on reused development data; no inner-OOF weight tuning or competition submission is performed. UltraBERT is excluded from validation.

In [ ]:
import ast,json,hashlib
from pathlib import Path
assert Path('/kaggle/input').is_dir()
sources=json.loads("{\"evaluate_lora24.py\":\"\\\"\\\"\\\"Fixed-mixture evaluation; no readout/ensemble fitting or outer-label tuning.\\\"\\\"\\\"\\nimport hashlib,json\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy.cluster.hierarchy import linkage,fcluster\\nfrom scipy.spatial.distance import squareform\\nfrom scipy.special import logsumexp\\nfrom sklearn.feature_extraction.text import TfidfVectorizer\\nfrom sklearn.metrics import balanced_accuracy_score\\nassert Path('/kaggle/input').is_dir()\\nroot=Path('/kaggle/working');inputs=Path('/kaggle/input')\\nspec=json.loads((root/'training_spec.json').read_text());plan=spec['plan']\\ntrain_path=[p for p in inputs.rglob('train.csv') if (p.parent/'test.csv').exists()]\\nassert len(train_path)==1 and hashlib.sha256(train_path[0].read_bytes()).hexdigest()==plan['train_sha256']\\ntrain=pd.read_csv(train_path[0],dtype=str);labels=np.asarray(plan['labels'],dtype=str)\\ny=np.searchsorted(labels,train.model.to_numpy());groups=train.prompt_id.to_numpy(dtype=str)\\ntexts=train.text.to_numpy(dtype=object)\\nreference={}\\nfor name,digest in spec['reference_hashes'].items():\\n    matches=list(inputs.rglob(name));assert len(matches)==1\\n    path=matches[0];assert hashlib.sha256(path.read_bytes()).hexdigest()==digest,name\\n    reference['report' if name.endswith('.json') else 'oof']=path\\nbaseline_report=json.loads(reference['report'].read_text())\\nassert baseline_report['train_sha256']==plan['train_sha256']\\nwith np.load(reference['oof'],allow_pickle=False) as z:\\n    assert np.array_equal(z['ids'],train.id.to_numpy(dtype=str)) and np.array_equal(z['labels'],labels)\\n    assert np.array_equal(z['y'],y) and np.array_equal(z['groups'],groups)\\n    baseline=z['nested'].copy()\\nassert baseline.shape==(3,len(train),18) and np.isfinite(baseline).all()\\ndef scores(values):return [float(balanced_accuracy_score(y,part.argmax(1))) for part in values]\\nnp.testing.assert_allclose(scores(baseline),[baseline_report['scenario_scores']['nested_candidate'][s] for s in ('1','3','6')],rtol=0,atol=1e-12)\\ndef clusters(rows):\\n    if len(rows)<2:return np.zeros(len(rows),int)\\n    x=TfidfVectorizer(stop_words='english',sublinear_tf=True).fit_transform(texts[rows])\\n    distance=np.clip(1-(x@x.T).toarray(),0,2);np.fill_diagonal(distance,0)\\n    return fcluster(linkage(squareform(distance,checks=False),method='average'),.9,criterion='distance')\\ndef context(rows,scenario,name):\\n    rng=np.random.default_rng(20261015+int(hashlib.sha256(name.encode()).hexdigest()[:6],16))\\n    batches=np.empty(len(rows),int)\\n    for group in np.unique(groups[rows]):\\n        members=rng.permutation(np.flatnonzero(groups[rows]==group))\\n        batches[members]=np.arange(len(members))//scenario\\n    lab=np.empty(len(rows),int);offset=0\\n    for batch in np.unique(batches):\\n        members=np.flatnonzero(batches==batch);recovered=clusters(rows[members])\\n        lab[members]=recovered+offset;offset+=int(recovered.max())+1\\n    return lab\\ndef affine(x,coef,bias):\\n    z=np.asarray(x,dtype=np.float64)@np.asarray(coef,dtype=np.float64).T+np.asarray(bias,dtype=np.float64)\\n    return z-logsumexp(z,axis=-1,keepdims=True)\\nplain=np.empty((3,len(train),18));center=np.empty_like(plain);coverage=np.zeros(len(train),int)\\nfor name in spec['jobs']:\\n    checkpoint=root/'lora24'/name\\n    completion=json.loads((checkpoint/'COMPLETE.json').read_text())\\n    assert completion['job']==name and completion['train_sha256']==plan['train_sha256']\\n    for file,digest in completion['files'].items():\\n        assert hashlib.sha256((checkpoint/file).read_bytes()).hexdigest()==digest,(name,file)\\n    if name=='final':continue\\n    rows=np.asarray(plan['jobs'][name]['valid'],int);coverage[rows]+=1\\n    with np.load(checkpoint/'validation_features.npz',allow_pickle=False) as z:\\n        assert np.array_equal(z['ids'],train.id.iloc[rows].to_numpy(dtype=str))\\n        assert np.array_equal(z['classes'],labels) and z['train_sha256'].item()==plan['train_sha256']\\n        features=z['features'].copy()\\n    with np.load(checkpoint/'readouts.npz',allow_pickle=False) as z:\\n        assert np.array_equal(z['classes'],labels) and z['train_sha256'].item()==plan['train_sha256']\\n        head={key:z[key].copy() for key in ('plain_coef','plain_intercept','center_coef','center_intercept')}\\n    p=affine(features,head['plain_coef'],head['plain_intercept'])\\n    for index,scenario in enumerate((1,3,6)):\\n        recovered=context(rows,scenario,name);sizes=np.bincount(recovered)[recovered]\\n        residual=features.copy()\\n        for group in np.unique(recovered):\\n            members=np.flatnonzero(recovered==group);residual[members]-=features[members].mean(0)\\n        c=affine(residual,head['center_coef'],head['center_intercept']);c[sizes<2]=p[sizes<2]\\n        plain[index,rows]=p;center[index,rows]=c\\nassert np.all(coverage==1) and np.isfinite(plain).all() and np.isfinite(center).all()\\nhalf=.5*plain+.5*center\\nunique=np.unique(groups)\\nden=np.stack([np.bincount(y[groups==g],minlength=18) for g in unique])\\ndraws=np.random.default_rng(20261015).integers(len(unique),size=(3000,len(unique)))\\ndef compare(logits):\\n    diff=((logits.argmax(-1)==y[None]).astype(float)-(baseline.argmax(-1)==y[None]).astype(float)).mean(0)\\n    num=np.stack([np.bincount(y[groups==g],weights=diff[groups==g],minlength=18) for g in unique])\\n    bootstrap=[float((num[d].sum(0)/den[d].sum(0).clip(1)).mean()) for d in draws]\\n    return dict(scores=dict(zip(('1','3','6'),scores(logits))),mean_gain=float(np.mean(scores(logits))-np.mean(scores(baseline))),\\n                paired_prompt_gain_interval=np.quantile(bootstrap,[.025,.975]).tolist())\\nassert spec['fixed_mixture_weights']==[.05,.1,.2]\\nmixtures={str(weight):(1-weight)*baseline+weight*half for weight in spec['fixed_mixture_weights']}\\nreport=dict(train_sha256=plan['train_sha256'],classes=labels.tolist(),\\n            training='Five independent outer fits and one final fit, official pretrained base only, fixed3epochs.',\\n            baseline_tag='gemma_tail_half_current_calibrated',baseline_scores=scores(baseline),\\n            standalone=dict(plain=scores(plain),center=scores(center),half=scores(half)),\\n            fixed_mixtures={weight:compare(values) for weight,values in mixtures.items()},\\n            inner_weight_fitting=False,outer_label_model_selection=False,submitted=False,\\n            limitations=['Development data reused; three fixed mixtures are descriptive comparisons.',\\n                         'Own two-head sampler/loss does not exactly reproduce unavailable public training source.',\\n                         'Lexical context recovery .9; singleton plain fallback; no class uniqueness assumption.',\\n                         'UltraBERT excluded from this validation. No competition submission created.'])\\n(root/'lora24_comparison.json').write_text(json.dumps(report,indent=2))\\nnp.savez_compressed(root/'lora24_oof.npz',ids=train.id.to_numpy(dtype=str),labels=labels,y=y,groups=groups,\\n                    plain=plain,center=center,half=half,**{f'mixture_{key}':value for key,value in mixtures.items()})\\nprint(json.dumps(report,indent=2),flush=True)\\n\",\"amp_update.py\":\"\\\"\\\"\\\"Bounded AMP updates shared by the smoke and full training workers.\\\"\\\"\\\"\\nimport time\\nimport torch\\n\\n\\ndef successful_update(optimizer,scaler,parameters,adapter_parameters,forward_batch,\\n                      batches,number,configuration,on_attempt,device):\\n    \\\"\\\"\\\"Repeat only scaled-gradient overflows; never step the optimizer on them.\\n\\n    Batches and dropout RNG are replayed identically. Nonfinite forward values\\n    fail immediately. Every overflow/successful attempt is emitted; other\\n    forward/backward/clipping exceptions propagate to the caller's error report.\\n    \\\"\\\"\\\"\\n    assert len(batches)==configuration['accumulation']==6\\n    max_overflows=configuration['max_overflows_per_update']\\n    minimum_scale=configuration['minimum_scale']\\n    cpu_rng=torch.get_rng_state()\\n    cuda_rng=torch.cuda.get_rng_state(device)\\n    total_started=time.monotonic()\\n    retry_seconds=0.;overflows=0;attempt=0\\n    while True:\\n        attempt+=1\\n        torch.set_rng_state(cpu_rng)\\n        torch.cuda.set_rng_state(cuda_rng,device)\\n        optimizer.zero_grad(set_to_none=True)\\n        torch.cuda.synchronize(device)\\n        started=time.monotonic()\\n        scale_before=float(scaler.get_scale())\\n        losses=[];plain_losses=[];center_losses=[];tokens=0;padded_tokens=0\\n        for batch in batches:\\n            loss,metrics=forward_batch(batch)\\n            assert bool(torch.isfinite(loss)),'Nonfinite forward loss; no AMP retry.'\\n            losses.append(float(loss.detach()))\\n            plain_losses.append(metrics['plain_loss'])\\n            center_losses.append(metrics['center_loss'])\\n            tokens+=metrics['tokens'];padded_tokens+=metrics['padded_tokens']\\n            scaler.scale(loss/len(batches)).backward()\\n            del loss,metrics\\n        scaler.unscale_(optimizer)\\n        finite=all(p.grad is None or bool(torch.isfinite(p.grad).all()) for p in parameters)\\n        if not finite:\\n            overflows+=1\\n            next_scale=scale_before*.5\\n            terminal=(overflows>max_overflows or next_scale<minimum_scale or scale_before<=minimum_scale)\\n            # Explicit public GradScaler.update(new_scale=...) resets its\\n            # optimizer stage without ever calling optimizer.step on overflow.\\n            if not terminal:scaler.update(new_scale=next_scale)\\n            optimizer.zero_grad(set_to_none=True)\\n            torch.cuda.synchronize(device)\\n            elapsed=time.monotonic()-started\\n            retry_seconds+=elapsed\\n            event=dict(update=number,attempt=attempt,status='TERMINAL_OVERFLOW' if terminal else 'OVERFLOW_RETRY',\\n                       finite_gradients=False,optimizer_step_performed=False,seconds=elapsed,\\n                       overflow_count=overflows,scale_before=scale_before,\\n                       scale_after=float(scaler.get_scale()),minimum_scale=minimum_scale)\\n            on_attempt(event)\\n            if terminal:\\n                raise FloatingPointError(f'Update{number}: overflow limit/minimum scale reached: {event}')\\n            continue\\n        assert any(p.grad is not None and bool(p.grad.abs().max()>0) for p in adapter_parameters),'Adapter gradient missing.'\\n        gradient_norm=float(torch.nn.utils.clip_grad_norm_(parameters,1.,error_if_nonfinite=True))\\n        scaler.step(optimizer);scaler.update()\\n        torch.cuda.synchronize(device)\\n        elapsed=time.monotonic()-started\\n        event=dict(update=number,attempt=attempt,status='SUCCESS',finite_gradients=True,\\n                   optimizer_step_performed=True,seconds=elapsed,overflow_count=overflows,\\n                   scale_before=scale_before,scale_after=float(scaler.get_scale()))\\n        on_attempt(event)\\n        return dict(update=number,seconds=elapsed,successful_attempt_seconds=elapsed,\\n                    retry_seconds=retry_seconds,total_update_wall_seconds=time.monotonic()-total_started,\\n                    attempts=attempt,skipped_overflows=overflows,loss=sum(losses)/len(losses),\\n                    plain_loss=sum(plain_losses)/len(plain_losses),center_loss=sum(center_losses)/len(center_losses),\\n                    gradient_norm=gradient_norm,finite_gradients=True,\\n                    grad_scaler_before=scale_before,grad_scaler_after=float(scaler.get_scale()),\\n                    tokens=tokens,padded_tokens=padded_tokens,tokens_per_second=tokens/elapsed,\\n                    peak_allocated_bytes=torch.cuda.max_memory_allocated(device),\\n                    peak_reserved_bytes=torch.cuda.max_memory_reserved(device))\\n\",\"optional_torchao.py\":\"\\\"\\\"\\\"Remove only the incompatible, unused torchao package in a Kaggle smoke VM.\\n\\nPEFT 0.20.0 probes this optional dispatcher even for ordinary FP16 Linear\\nlayers. No quantization is used by this experiment. No packages are installed.\\n\\\"\\\"\\\"\\nimport importlib.metadata as metadata\\nimport importlib.util\\nimport json\\nimport subprocess\\nimport sys\\nfrom pathlib import Path\\n\\nfrom packaging.version import Version\\n\\nassert Path('/kaggle/input').is_dir(), 'Kaggle-only environment preflight.'\\nassert not any(name.split('.')[0] in {'torch', 'transformers', 'peft'}\\n               for name in sys.modules), 'Run before ML library imports.'\\nout = Path('/kaggle/working/optional_torchao_audit.json')\\ncore = ('torch', 'transformers', 'peft', 'accelerate', 'safetensors', 'numpy', 'pandas')\\nbefore = {name: metadata.version(name) for name in core}\\nreport = dict(status='INITIALIZING', core_versions_before=before,\\n              network_install_attempted=False, packages_installed=[],\\n              packages_removed=[], ordinary_unquantized_base_required=True)\\n\\ntry:\\n    configs = []\\n    for path in Path('/kaggle/input').rglob('config.json'):\\n        try:\\n            config = json.loads(path.read_text())\\n        except (OSError, ValueError):\\n            continue\\n        if (config.get('model_type') == 'qwen3' and config.get('hidden_size') == 4096\\n                and config.get('num_hidden_layers') == 36\\n                and list(path.parent.glob('*.safetensors'))):\\n            configs.append(config)\\n    assert len(configs) == 1, 'Require the sole official Qwen3-8B base.'\\n    assert not configs[0].get('quantization_config'), 'Never disable a required quantizer.'\\n    assert not list(Path('/kaggle/input').rglob('adapter_model.safetensors'))\\n    try:\\n        installed = metadata.version('torchao')\\n    except metadata.PackageNotFoundError:\\n        installed = None\\n    report['torchao_version_before'] = installed\\n    if installed is not None and Version(installed) < Version('0.16.0'):\\n        assert before['peft'] == '0.20.0', 'Review the dispatcher for another PEFT release.'\\n        process = subprocess.run(\\n            [sys.executable, '-m', 'pip', '--disable-pip-version-check',\\n             'uninstall', '--yes', 'torchao'],\\n            text=True, capture_output=True, timeout=120,\\n        )\\n        report['uninstall_returncode'] = process.returncode\\n        report['uninstall_stdout'] = process.stdout[-4000:]\\n        report['uninstall_stderr'] = process.stderr[-4000:]\\n        assert process.returncode == 0, 'Optional torchao uninstall failed.'\\n        try:\\n            metadata.version('torchao')\\n        except metadata.PackageNotFoundError:\\n            pass\\n        else:\\n            raise AssertionError('torchao distribution remains installed.')\\n        assert importlib.util.find_spec('torchao') is None, 'torchao import remains available.'\\n        report['packages_removed'] = ['torchao']\\n        report['status'] = 'REMOVED_INCOMPATIBLE_OPTIONAL_TORCHAO'\\n    else:\\n        report['status'] = 'NO_CHANGE_REQUIRED'\\n    after = {name: metadata.version(name) for name in core}\\n    report['core_versions_after'] = after\\n    assert before == after, 'Core dependency versions changed.'\\n    report['core_versions_unchanged'] = True\\nexcept Exception as error:\\n    report.update(status='ERROR', error=repr(error))\\n    raise\\nfinally:\\n    out.write_text(json.dumps(report, indent=2))\\n    print(json.dumps(report, indent=2), flush=True)\\n\",\"train_lora24.py\":\"\\\"\\\"\\\"Own five-outer-fold plus final Qwen24 LoRA; fixed three-epoch recipe.\\n\\nRuns only on Kaggle. Each process sees one independent GPU. Public attribution\\nadapters are forbidden inputs. Outer labels never select epochs or checkpoints.\\n\\\"\\\"\\\"\\nimport os\\nfor key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):\\n    os.environ[key]='2'\\nos.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1'\\nos.environ['TOKENIZERS_PARALLELISM']='false'\\nimport argparse,gc,hashlib,json,math,random,time,traceback\\nfrom pathlib import Path\\nfrom importlib.metadata import version\\nassert Path('/kaggle/input').is_dir(),'Kaggle only.'\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nfrom torch import nn\\nfrom transformers import AutoConfig,AutoModel,AutoTokenizer\\nfrom peft import LoraConfig,get_peft_model\\nfrom safetensors.torch import save_file\\nfrom amp_update import successful_update\\n\\nparser=argparse.ArgumentParser();parser.add_argument('--worker',type=int,choices=[0,1],required=True)\\nargs=parser.parse_args()\\ntorch.set_num_threads(2)\\nassert torch.cuda.is_available() and torch.cuda.device_count()==1\\nassert torch.cuda.get_device_capability(0)==(7,5)\\ndevice=torch.device('cuda:0')\\nroot=Path('/kaggle/working');inputs=Path('/kaggle/input')\\nspec=json.loads((root/'training_spec.json').read_text())\\nassert spec['authorized_diagnostic_smoke_required'] is True\\nplan=spec['plan'];labels=np.asarray(plan['labels'],dtype=str)\\nassert len(labels)==18 and spec['amp']==dict(initial_scale=256.,minimum_scale=1.,max_overflows_per_update=3,accumulation=6,growth_interval=2000)\\ndata=[p for p in inputs.rglob('train.csv') if (p.parent/'test.csv').exists()]\\nassert len(data)==1 and hashlib.sha256(data[0].read_bytes()).hexdigest()==plan['train_sha256']\\ntrain=pd.read_csv(data[0],dtype=str)\\nassert len(train)==2699 and train.id.is_unique and train.notna().all().all()\\nassert not list(inputs.rglob('adapter_model.safetensors')),'Public attribution adapters forbidden.'\\nmodel_paths=[]\\nfor path in inputs.rglob('config.json'):\\n    try:config_json=json.loads(path.read_text())\\n    except (OSError,ValueError):continue\\n    if (config_json.get('model_type')=='qwen3' and config_json.get('hidden_size')==4096\\n            and config_json.get('num_hidden_layers')==36 and config_json.get('intermediate_size')==12288\\n            and list(path.parent.glob('*.safetensors'))):model_paths.append(path.parent)\\nassert len(model_paths)==1\\nmodel_dir=model_paths[0]\\nassert hashlib.sha256((model_dir/'config.json').read_bytes()).hexdigest()==spec['smoke_evidence']['base_config_sha256']\\ntokenizer=AutoTokenizer.from_pretrained(model_dir,local_files_only=True)\\ntokenizer.padding_side='right'\\nif tokenizer.pad_token is None:tokenizer.pad_token=tokenizer.eos_token\\ntokens=tokenizer(train.text.tolist(),add_special_tokens=False,truncation=True,max_length=447)['input_ids']\\ntokens=[ids+[tokenizer.eos_token_id] for ids in tokens]\\nassert min(map(len,tokens))>=2 and max(map(len,tokens))<=448\\nlayers=[2,4,6,8,12,16,20,24]\\ntarget=np.searchsorted(labels,train.model.to_numpy())\\ngroups=train.prompt_id.to_numpy(dtype=str)\\nall_jobs=list(spec['jobs'])\\nassigned=all_jobs[args.worker::2]\\nfolder=root/'lora24';folder.mkdir(exist_ok=True)\\nworker_status=dict(worker=args.worker,jobs=assigned,completed=[],status='INITIALIZING',\\n                   dependencies={name:version(name) for name in ('torch','transformers','peft','accelerate','safetensors')},\\n                   cuda_visible_devices=os.environ.get('CUDA_VISIBLE_DEVICES'),public_adapters_loaded=False)\\nstatus_path=folder/f'worker{args.worker}_status.json'\\ndef write_status():status_path.write_text(json.dumps(worker_status,indent=2))\\nwrite_status()\\n\\ndef epoch_batches(fit,rng):\\n    batches=[];coverage=[]\\n    for group in np.unique(groups[fit]):\\n        members=rng.permutation(fit[groups[fit]==group])\\n        assert len(members)>=3\\n        for start in range(0,len(members),3):\\n            batch=members[start:start+3]\\n            if len(batch)<3:\\n                more=rng.choice(np.setdiff1d(members,batch),3-len(batch),replace=False)\\n                batch=np.concatenate([batch,more])\\n            assert len(batch)==3 and len(np.unique(batch))==3 and len(np.unique(groups[batch]))==1\\n            batches.append(batch);coverage.extend(batch.tolist())\\n    assert set(coverage)==set(fit),'Every fitting answer must occur in each epoch.'\\n    batches=[batches[i] for i in rng.permutation(len(batches))]\\n    original_count=len(batches)\\n    while len(batches)%6:batches.append(batches[int(rng.integers(original_count))].copy())\\n    return batches,dict(original_microbatches=original_count,padded_microbatches=len(batches)-original_count,\\n                        example_exposures=len(batches)*3,unique_fitting_examples=len(fit))\\n\\ntry:\\n    for name in assigned:\\n        fit=np.asarray(plan['jobs'][name]['fit'],int);valid=np.asarray(plan['jobs'][name]['valid'],int)\\n        assert not set(fit)&set(valid) and not set(groups[fit])&set(groups[valid])\\n        assert set(fit)|set(valid)==set(range(len(train)))\\n        assert np.array_equal(np.unique(target[fit]),np.arange(18))\\n        checkpoint=folder/name\\n        assert not checkpoint.exists(),'No implicit resume or reuse of a prior fitted checkpoint.'\\n        checkpoint.mkdir()\\n        worker_status.update(status='TRAINING',current_job=name);write_status()\\n        random.seed(42);np.random.seed(42);torch.manual_seed(42);torch.cuda.manual_seed_all(42)\\n        rng=np.random.default_rng(42)\\n        planned=[epoch_batches(fit,rng) for _ in range(3)]\\n        total_updates=sum(len(batches)//6 for batches,_ in planned)\\n        assert total_updates<spec['amp']['growth_interval'],'Manual-scale growth-tracker caveat requires review.'\\n        config=AutoConfig.from_pretrained(model_dir,local_files_only=True)\\n        config.num_hidden_layers=24;config.use_cache=False\\n        if getattr(config,'layer_types',None) is not None:config.layer_types=config.layer_types[:24]\\n        load_started=time.monotonic()\\n        base=AutoModel.from_pretrained(model_dir,config=config,dtype=torch.float16,\\n                local_files_only=True,low_cpu_mem_usage=True,device_map={'':0},attn_implementation='sdpa')\\n        assert not hasattr(base,'lm_head') and all(p.device==device for p in base.parameters())\\n        base.requires_grad_(False)\\n        model=get_peft_model(base,LoraConfig(r=16,lora_alpha=32,lora_dropout=.05,bias='none',\\n                    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']))\\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})\\n        model.enable_input_require_grads()\\n        for p in model.parameters():\\n            if p.requires_grad:p.data=p.data.float()\\n        norms=nn.ModuleList([nn.LayerNorm(4096) for _ in layers]).to(device).float()\\n        plain=nn.Linear(4096*len(layers),18).to(device).float()\\n        centered=nn.Linear(4096*len(layers),18).to(device).float()\\n        dropout=nn.Dropout(.1)\\n        adapter_parameters=[p for p in model.parameters() if p.requires_grad]\\n        head_parameters=list(norms.parameters())+list(plain.parameters())+list(centered.parameters())\\n        parameters=adapter_parameters+head_parameters\\n        assert all(p.dtype==torch.float32 for p in parameters)\\n        assert all('lora_' in key for key,p in model.named_parameters() if p.requires_grad)\\n        optimizer=torch.optim.AdamW([{'params':adapter_parameters,'lr':1e-4},\\n                                     {'params':head_parameters,'lr':1e-3}],weight_decay=.01)\\n        warmup=max(1,int(total_updates*.1))\\n        def schedule(step):\\n            if step<warmup:return (step+1)/warmup\\n            fraction=min(1.,(step-warmup)/max(1,total_updates-warmup))\\n            return .5*(1+math.cos(math.pi*fraction))\\n        scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,schedule)\\n        scaler=torch.amp.GradScaler('cuda',init_scale=256.,growth_interval=2000)\\n        records=[];attempts=[];skipped=[]\\n        job_report=dict(job=name,train_sha256=plan['train_sha256'],classes=labels.tolist(),\\n                        fit_rows=len(fit),valid_rows=len(valid),fit_ids=train.id.iloc[fit].tolist(),\\n                        epochs=3,total_updates=total_updates,planned_epochs=[record for _,record in planned],\\n                        load_seconds=time.monotonic()-load_started,attempts=attempts,skipped_scales=skipped,\\n                        updates=records,public_adapters_loaded=False,pretrained_base_only=True,\\n                        config_sha256=hashlib.sha256((model_dir/'config.json').read_bytes()).hexdigest(),\\n                        training_spec_sha256=hashlib.sha256((root/'training_spec.json').read_bytes()).hexdigest())\\n        report_path=checkpoint/'training_report.json'\\n        def persist_attempt(event):\\n            attempts.append(event)\\n            if 'OVERFLOW' in event['status']:skipped.append(event.copy())\\n            report_path.write_text(json.dumps(job_report,indent=2))\\n            if event['status']!='SUCCESS':print(name,'AMP',json.dumps(event),flush=True)\\n        def representation(indices):\\n            batch=tokenizer.pad({'input_ids':[tokens[i] for i in indices]},return_tensors='pt').to(device)\\n            with torch.autocast('cuda',dtype=torch.float16):\\n                hidden=model(**batch,output_hidden_states=True,use_cache=False).hidden_states\\n            mask=batch.attention_mask.unsqueeze(-1).float();mask[:,0]=0\\n            with torch.autocast('cuda',enabled=False):\\n                z=torch.cat([norm((hidden[layer].float()*mask).sum(1)/mask.sum(1))\\n                             for layer,norm in zip(layers,norms)],-1)\\n            assert bool(torch.isfinite(z).all())\\n            return z,int(batch.attention_mask.sum()),int(batch.input_ids.numel())\\n        def forward_batch(indices):\\n            assert len(indices)==3 and len(np.unique(groups[indices]))==1\\n            z,n_tokens,n_padded=representation(indices)\\n            y=torch.as_tensor(target[indices],device=device,dtype=torch.long)\\n            with torch.autocast('cuda',enabled=False):\\n                loss_plain=nn.functional.cross_entropy(plain(dropout(z)),y)\\n                loss_center=nn.functional.cross_entropy(centered(dropout(z-z.mean(0,keepdim=True))),y)\\n            return loss_plain+loss_center,dict(plain_loss=float(loss_plain.detach()),\\n                        center_loss=float(loss_center.detach()),tokens=n_tokens,padded_tokens=n_padded)\\n        torch.cuda.reset_peak_memory_stats(device)\\n        model.train();norms.train();plain.train();centered.train()\\n        for epoch,(batches,_) in enumerate(planned):\\n            for start in range(0,len(batches),6):\\n                record=successful_update(optimizer,scaler,parameters,adapter_parameters,forward_batch,\\n                        batches[start:start+6],len(records)+1,spec['amp'],persist_attempt,device)\\n                record.update(epoch=epoch+1,learning_rates=[group['lr'] for group in optimizer.param_groups])\\n                records.append(record);scheduler.step()\\n                if len(records)%10==0:\\n                    report_path.write_text(json.dumps(job_report,indent=2))\\n                    print(json.dumps(dict(job=name,worker=args.worker,completed=len(records),\\n                                         total=total_updates,epoch=epoch+1,last=record)),flush=True)\\n        assert len(records)==total_updates\\n        # Fixed final epoch only; no outer-label checkpoint selection.\\n        model.eval();norms.eval();plain.eval();centered.eval()\\n        model.gradient_checkpointing_disable()\\n        def extract(rows):\\n            values=np.empty((len(rows),4096*len(layers)),np.float32)\\n            order=np.argsort([len(tokens[i]) for i in rows],kind='stable')\\n            with torch.inference_mode():\\n                for start in range(0,len(order),3):\\n                    positions=order[start:start+3]\\n                    z,_,_=representation(rows[positions])\\n                    values[positions]=z.cpu().numpy()\\n            assert np.isfinite(values).all()\\n            return values\\n        valid_z=extract(valid)\\n        # Stable portable affine heads; CPU matrix multiplication uses these\\n        # saved arrays later, without any original-label fitting.\\n        np.savez_compressed(checkpoint/'readouts.npz',classes=labels,\\n            plain_coef=plain.weight.detach().cpu().numpy(),plain_intercept=plain.bias.detach().cpu().numpy(),\\n            center_coef=centered.weight.detach().cpu().numpy(),center_intercept=centered.bias.detach().cpu().numpy(),\\n            train_sha256=plan['train_sha256'],pool_layers=np.array(layers))\\n        np.savez_compressed(checkpoint/'validation_features.npz',ids=train.id.iloc[valid].to_numpy(dtype=str),\\n                            classes=labels,features=valid_z,train_sha256=plan['train_sha256'])\\n        model.save_pretrained(checkpoint/'adapter',safe_serialization=True,save_embedding_layers=False)\\n        tensors={**{f'norms.{key}':value.detach().cpu().contiguous() for key,value in norms.state_dict().items()},\\n                 **{f'plain.{key}':value.detach().cpu().contiguous() for key,value in plain.state_dict().items()},\\n                 **{f'centered.{key}':value.detach().cpu().contiguous() for key,value in centered.state_dict().items()}}\\n        save_file(tensors,checkpoint/'heads.safetensors')\\n        if name=='final':\\n            anchors=np.arange(3)\\n            np.savez_compressed(checkpoint/'anchor_features.npz',ids=train.id.iloc[anchors].to_numpy(dtype=str),\\n                                classes=labels,features=extract(anchors),train_sha256=plan['train_sha256'])\\n            tokenizer.save_pretrained(checkpoint/'tokenizer');config.save_pretrained(checkpoint/'base_config')\\n        job_report.update(status='COMPLETE',successful_updates=len(records),\\n                          peak_allocated_bytes=torch.cuda.max_memory_allocated(device),\\n                          peak_reserved_bytes=torch.cuda.max_memory_reserved(device),\\n                          successful_attempt_seconds=sum(row['seconds'] for row in records),\\n                          overflow_retry_seconds=sum(row['seconds'] for row in skipped),\\n                          checkpoints_selected_by='Fixed epoch3; no held-out labels')\\n        report_path.write_text(json.dumps(job_report,indent=2))\\n        (checkpoint/'COMPLETE.json').write_text(json.dumps(dict(job=name,train_sha256=plan['train_sha256'],\\n                files={str(path.relative_to(checkpoint)):hashlib.sha256(path.read_bytes()).hexdigest()\\n                       for path in checkpoint.rglob('*') if path.is_file()}),indent=2))\\n        worker_status['completed'].append(name);write_status()\\n        print('JOB COMPLETE',name,flush=True)\\n        del base,model,norms,plain,centered,optimizer,scheduler,scaler,parameters,adapter_parameters,head_parameters,tensors,valid_z\\n        gc.collect();torch.cuda.empty_cache()\\n    worker_status.update(status='COMPLETE');write_status()\\nexcept Exception as error:\\n    worker_status.update(status='ERROR',error=repr(error),traceback=traceback.format_exc())\\n    write_status()\\n    raise\\n\"}")
for name,source in sources.items():
    ast.parse(source,filename=name)
    Path('/kaggle/working',name).write_text(source)
spec=json.loads("{\"plan\":{\"train_sha256\":\"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\",\"labels\":[\"claude-fable-5\",\"claude-opus-4-6\",\"claude-opus-4-8\",\"claude-opus-5\",\"claude-sonnet-5\",\"gemini-3.1-pro-preview\",\"gemini-3.6-flash\",\"gemini-3.7-flash\",\"gemini-3.8-flash\",\"glm-5.3\",\"glm-5.3-flash\",\"gpt-5.5\",\"gpt-5.6-luna\",\"gpt-5.6-sol\",\"gpt-5.6-terra\",\"gpt-6-astra\",\"kimi-k3\",\"qwen3.8-max\"],\"outer_folds\":5,\"inner_folds\":3,\"method\":\"Nested prompt-disjoint validation; blend weights fit only to inner OOF within each outer training split.\",\"jobs\":{\"outer0\":{\"fit\":[0,1,2,3,4,5,6,7,9,14,15,17,18,20,22,24,25,27,28,29,30,31,33,34,35,37,39,40,41,42,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,60,61,62,63,64,65,68,69,70,71,72,74,75,76,77,78,79,80,82,83,85,86,87,88,90,91,92,93,94,95,96,97,99,100,102,103,104,105,106,107,108,109,110,111,112,113,114,116,117,119,120,121,123,124,126,127,128,130,131,132,134,135,137,139,141,143,144,145,146,147,148,149,150,151,154,155,157,158,159,160,161,164,165,166,169,170,171,173,174,175,176,177,178,179,180,181,182,183,184,187,188,190,191,192,193,194,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,223,224,225,226,227,230,231,233,236,237,238,239,240,241,242,244,245,246,247,248,250,251,252,253,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,275,277,278,279,280,281,283,284,285,286,287,288,289,290,291,292,293,295,296,298,299,300,302,303,304,305,306,307,308,310,311,312,315,316,318,319,320,321,322,323,325,326,328,329,330,331,332,333,334,335,336,337,338,339,341,342,345,346,347,348,349,352,353,355,356,357,358,359,361,365,366,367,368,369,370,371,372,373,374,375,377,378,379,380,381,382,383,384,386,387,389,392,393,394,395,396,397,398,399,400,401,402,404,405,406,407,408,410,412,413,414,415,416,418,419,420,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,440,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,465,466,467,468,469,471,472,473,474,475,476,477,478,479,480,481,482,484,485,487,488,489,490,491,492,494,495,496,497,499,500,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,522,523,525,526,527,528,529,530,531,532,535,536,537,538,539,540,541,542,543,544,545,547,548,550,551,552,553,554,555,556,557,559,560,562,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,584,585,586,587,588,590,592,593,594,595,598,601,603,604,605,606,607,608,610,611,612,613,614,615,617,618,620,621,622,623,624,628,629,630,631,632,633,634,635,636,637,639,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,658,660,661,663,664,665,666,667,668,669,671,672,675,676,677,678,679,680,681,682,683,684,685,686,688,689,690,691,692,694,695,696,697,698,699,701,703,704,705,706,707,708,709,710,712,713,714,715,716,717,718,719,720,723,724,725,726,727,728,729,730,731,732,734,736,737,738,739,740,741,742,743,747,748,749,750,751,752,753,754,755,757,759,760,761,762,763,764,766,767,768,769,771,773,774,776,777,778,781,782,783,784,785,787,789,790,791,792,793,794,796,797,799,800,801,802,803,805,807,808,809,811,812,814,816,817,818,820,821,822,823,824,825,826,827,828,830,831,832,834,835,836,838,841,842,843,845,847,849,850,851,852,853,854,855,856,857,858,859,860,861,863,864,865,866,867,868,869,871,872,874,875,876,877,878,879,880,881,882,883,884,885,886,887,889,891,892,893,894,895,896,897,898,899,901,902,903,905,907,908,909,910,912,913,914,918,919,921,922,923,924,925,926,927,928,929,930,931,932,933,934,935,936,937,938,939,940,941,942,943,944,945,946,947,950,953,954,955,956,957,958,961,963,964,966,967,968,970,973,975,976,977,978,979,980,981,982,984,985,986,987,988,989,990,991,993,994,996,998,999,1001,1002,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1016,1017,1018,1019,1021,1023,1024,1025,1026,1028,1030,1031,1032,1036,1037,1039,1040,1041,1042,1043,1044,1045,1046,1047,1049,1053,1054,1055,1056,1058,1059,1060,1061,1062,1063,1064,1065,1067,1068,1069,1070,1071,1073,1074,1076,1077,1078,1079,1080,1081,1082,1083,1084,1087,1090,1091,1093,1094,1095,1096,1097,1098,1099,1101,1103,1104,1105,1106,1108,1110,1111,1112,1114,1115,1116,1117,1118,1119,1121,1122,1123,1124,1125,1126,1127,1128,1129,1130,1131,1133,1134,1135,1136,1137,1138,1139,1140,1141,1143,1144,1146,1147,1148,1149,1151,1152,1154,1155,1156,1159,1160,1163,1164,1166,1167,1168,1169,1171,1172,1173,1174,1175,1176,1177,1179,1180,1181,1182,1183,1184,1185,1186,1188,1189,1191,1193,1194,1196,1197,1199,1201,1203,1204,1205,1206,1207,1208,1209,1210,1212,1213,1214,1215,1216,1217,1218,1219,1220,1221,1222,1223,1225,1226,1227,1228,1229,1230,1231,1233,1234,1235,1236,1237,1238,1239,1241,1242,1243,1244,1245,1246,1247,1248,1249,1250,1251,1252,1253,1255,1258,1259,1260,1261,1262,1263,1264,1265,1266,1267,1268,1269,1270,1271,1272,1273,1274,1275,1276,1278,1279,1281,1282,1283,1284,1285,1286,1287,1288,1290,1291,1292,1293,1295,1297,1298,1299,1300,1301,1302,1303,1304,1305,1306,1307,1308,1309,1310,1311,1312,1313,1314,1315,1316,1317,1318,1319,1320,1321,1322,1323,1325,1326,1327,1328,1329,1330,1331,1332,1333,1335,1336,1337,1338,1339,1340,1341,1342,1343,1345,1346,1347,1348,1352,1354,1355,1356,1357,1358,1360,1361,1362,1363,1367,1368,1369,1371,1373,1374,1375,1376,1378,1379,1380,1381,1382,1383,1384,1385,1386,1387,1388,1389,1390,1391,1392,1393,1395,1396,1397,1398,1399,1400,1401,1402,1403,1404,1406,1407,1408,1410,1411,1412,1413,1415,1416,1417,1418,1419,1420,1422,1423,1426,1427,1429,1431,1432,1433,1434,1435,1436,1437,1438,1439,1440,1442,1443,1446,1447,1448,1450,1451,1452,1453,1454,1455,1456,1458,1459,1461,1462,1464,1465,1466,1467,1469,1470,1471,1473,1474,1475,1477,1478,1481,1482,1486,1487,1489,1491,1492,1494,1495,1496,1497,1499,1500,1501,1502,1503,1504,1505,1506,1507,1509,1510,1511,1512,1513,1514,1516,1517,1520,1521,1523,1524,1526,1527,1528,1529,1530,1532,1533,1535,1538,1539,1541,1542,1543,1544,1545,1546,1548,1550,1551,1552,1553,1554,1555,1557,1558,1559,1561,1562,1563,1564,1565,1566,1567,1568,1569,1570,1571,1572,1573,1574,1575,1576,1577,1579,1580,1582,1583,1585,1586,1587,1588,1589,1590,1591,1592,1593,1594,1595,1596,1597,1601,1603,1604,1605,1606,1608,1609,1610,1611,1612,1614,1615,1616,1617,1618,1619,1620,1621,1622,1623,1625,1626,1627,1628,1630,1631,1632,1633,1634,1635,1636,1639,1640,1642,1643,1645,1646,1648,1649,1650,1651,1654,1656,1657,1658,1660,1661,1662,1663,1665,1667,1669,1670,1671,1672,1673,1678,1679,1682,1684,1685,1687,1688,1689,1690,1691,1692,1693,1694,1695,1696,1697,1698,1699,1700,1702,1703,1704,1705,1706,1707,1708,1711,1712,1715,1717,1718,1719,1720,1721,1722,1724,1725,1728,1729,1730,1731,1732,1733,1734,1735,1736,1737,1739,1740,1741,1742,1743,1744,1745,1746,1747,1748,1750,1751,1752,1753,1754,1755,1756,1757,1758,1760,1761,1762,1763,1764,1765,1766,1767,1768,1769,1770,1771,1772,1773,1775,1776,1778,1779,1781,1782,1784,1787,1789,1790,1791,1792,1793,1794,1795,1796,1797,1798,1799,1800,1801,1803,1804,1805,1807,1808,1809,1810,1811,1813,1814,1817,1818,1819,1820,1821,1823,1826,1828,1829,1831,1832,1833,1834,1835,1836,1837,1838,1839,1840,1841,1842,1843,1844,1845,1846,1847,1848,1849,1850,1851,1852,1853,1854,1855,1856,1857,1858,1859,1860,1861,1862,1863,1864,1866,1867,1868,1870,1871,1872,1874,1875,1876,1877,1878,1879,1880,1882,1883,1884,1885,1886,1887,1888,1890,1891,1892,1893,1894,1895,1896,1898,1899,1900,1901,1902,1903,1904,1905,1906,1908,1909,1911,1913,1914,1915,1917,1918,1920,1921,1923,1925,1926,1928,1929,1930,1931,1932,1933,1934,1935,1937,1938,1940,1941,1942,1943,1944,1945,1946,1947,1948,1949,1950,1951,1952,1953,1954,1955,1956,1958,1959,1962,1963,1964,1965,1966,1967,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1983,1984,1986,1987,1988,1989,1990,1991,1992,1994,1996,1997,1998,1999,2000,2001,2002,2003,2005,2006,2007,2008,2009,2010,2011,2012,2014,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2053,2054,2055,2056,2058,2059,2061,2062,2064,2065,2066,2067,2068,2069,2070,2072,2073,2074,2075,2076,2077,2078,2080,2081,2082,2083,2084,2085,2087,2088,2089,2090,2091,2094,2097,2098,2100,2101,2102,2103,2104,2105,2106,2107,2108,2109,2111,2113,2114,2115,2116,2118,2119,2121,2122,2125,2127,2129,2130,2131,2132,2134,2135,2136,2137,2138,2139,2140,2141,2143,2144,2145,2146,2147,2149,2150,2151,2152,2153,2154,2155,2156,2158,2160,2161,2162,2163,2165,2166,2167,2168,2169,2171,2172,2173,2174,2175,2176,2177,2178,2179,2180,2181,2182,2183,2184,2186,2187,2188,2190,2191,2193,2195,2196,2199,2200,2201,2202,2204,2205,2206,2207,2208,2209,2210,2211,2212,2213,2215,2216,2217,2219,2222,2224,2225,2227,2229,2230,2231,2234,2236,2237,2240,2242,2243,2244,2245,2246,2247,2248,2249,2250,2251,2252,2253,2254,2255,2256,2258,2260,2261,2262,2263,2264,2266,2267,2268,2269,2271,2272,2273,2274,2275,2277,2279,2280,2281,2282,2283,2284,2285,2286,2287,2288,2289,2290,2291,2292,2293,2294,2296,2298,2299,2300,2301,2302,2303,2304,2306,2307,2309,2310,2311,2316,2317,2318,2319,2320,2321,2322,2323,2324,2326,2327,2329,2330,2331,2332,2333,2334,2335,2336,2337,2338,2340,2341,2342,2343,2344,2345,2346,2347,2348,2349,2350,2351,2352,2354,2355,2356,2360,2361,2363,2364,2365,2366,2368,2370,2371,2372,2373,2374,2375,2377,2379,2380,2381,2383,2384,2385,2386,2388,2389,2390,2391,2392,2394,2395,2396,2397,2398,2399,2401,2402,2403,2405,2406,2407,2408,2409,2410,2412,2413,2414,2416,2417,2418,2420,2421,2422,2423,2425,2426,2427,2428,2429,2431,2432,2433,2435,2436,2437,2438,2439,2440,2441,2443,2444,2445,2446,2447,2448,2449,2450,2451,2452,2454,2455,2456,2457,2458,2459,2460,2461,2463,2464,2466,2468,2469,2470,2472,2473,2474,2475,2476,2477,2478,2479,2481,2482,2483,2484,2486,2487,2489,2491,2493,2497,2499,2500,2501,2502,2503,2504,2507,2508,2511,2513,2514,2515,2516,2517,2518,2519,2520,2521,2522,2523,2524,2525,2526,2528,2529,2531,2532,2535,2536,2537,2538,2539,2540,2541,2542,2543,2544,2545,2548,2549,2550,2553,2554,2555,2556,2557,2558,2559,2560,2561,2563,2564,2565,2566,2567,2568,2570,2571,2572,2574,2575,2577,2578,2579,2580,2581,2583,2584,2585,2587,2588,2590,2591,2593,2594,2595,2596,2597,2599,2601,2602,2603,2605,2606,2607,2608,2609,2610,2611,2612,2613,2614,2615,2616,2617,2618,2619,2620,2621,2623,2624,2625,2627,2628,2629,2630,2631,2632,2633,2634,2635,2636,2637,2639,2640,2641,2642,2644,2645,2646,2647,2648,2651,2653,2654,2655,2656,2658,2659,2660,2661,2662,2664,2665,2666,2667,2668,2669,2672,2673,2674,2675,2676,2677,2678,2679,2680,2682,2683,2684,2685,2686,2687,2689,2690,2691,2692,2693,2694,2695,2696,2697,2698],\"valid\":[8,10,11,12,13,16,19,21,23,26,32,36,38,43,59,66,67,73,81,84,89,98,101,115,118,122,125,129,133,136,138,140,142,152,153,156,162,163,167,168,172,185,186,189,195,196,221,222,228,229,232,234,235,243,249,254,274,276,282,294,297,301,309,313,314,317,324,327,340,343,344,350,351,354,360,362,363,364,376,385,388,390,391,403,409,411,417,421,439,441,464,470,483,486,493,498,501,521,524,533,534,546,549,558,561,563,564,583,589,591,596,597,599,600,602,609,616,619,625,626,627,638,640,657,659,662,670,673,674,687,693,700,702,711,721,722,733,735,744,745,746,756,758,765,770,772,775,779,780,786,788,795,798,804,806,810,813,815,819,829,833,837,839,840,844,846,848,862,870,873,888,890,900,904,906,911,915,916,917,920,948,949,951,952,959,960,962,965,969,971,972,974,983,992,995,997,1000,1003,1015,1020,1022,1027,1029,1033,1034,1035,1038,1048,1050,1051,1052,1057,1066,1072,1075,1085,1086,1088,1089,1092,1100,1102,1107,1109,1113,1120,1132,1142,1145,1150,1153,1157,1158,1161,1162,1165,1170,1178,1187,1190,1192,1195,1198,1200,1202,1211,1224,1232,1240,1254,1256,1257,1277,1280,1289,1294,1296,1324,1334,1344,1349,1350,1351,1353,1359,1364,1365,1366,1370,1372,1377,1394,1405,1409,1414,1421,1424,1425,1428,1430,1441,1444,1445,1449,1457,1460,1463,1468,1472,1476,1479,1480,1483,1484,1485,1488,1490,1493,1498,1508,1515,1518,1519,1522,1525,1531,1534,1536,1537,1540,1547,1549,1556,1560,1578,1581,1584,1598,1599,1600,1602,1607,1613,1624,1629,1637,1638,1641,1644,1647,1652,1653,1655,1659,1664,1666,1668,1674,1675,1676,1677,1680,1681,1683,1686,1701,1709,1710,1713,1714,1716,1723,1726,1727,1738,1749,1759,1774,1777,1780,1783,1785,1786,1788,1802,1806,1812,1815,1816,1822,1824,1825,1827,1830,1865,1869,1873,1881,1889,1897,1907,1910,1912,1916,1919,1922,1924,1927,1936,1939,1957,1960,1961,1968,1982,1985,1993,1995,2004,2013,2015,2031,2032,2052,2057,2060,2063,2071,2079,2086,2092,2093,2095,2096,2099,2110,2112,2117,2120,2123,2124,2126,2128,2133,2142,2148,2157,2159,2164,2170,2185,2189,2192,2194,2197,2198,2203,2214,2218,2220,2221,2223,2226,2228,2232,2233,2235,2238,2239,2241,2257,2259,2265,2270,2276,2278,2295,2297,2305,2308,2312,2313,2314,2315,2325,2328,2339,2353,2357,2358,2359,2362,2367,2369,2376,2378,2382,2387,2393,2400,2404,2411,2415,2419,2424,2430,2434,2442,2453,2462,2465,2467,2471,2480,2485,2488,2490,2492,2494,2495,2496,2498,2505,2506,2509,2510,2512,2527,2530,2533,2534,2546,2547,2551,2552,2562,2569,2573,2576,2582,2586,2589,2592,2598,2600,2604,2622,2626,2638,2643,2649,2650,2652,2657,2663,2670,2671,2681,2688]},\"outer0_inner0\":{\"fit\":[2,4,5,6,14,17,25,27,28,31,33,34,35,37,41,42,44,45,46,47,49,50,51,53,54,55,56,57,61,62,64,65,68,69,70,71,77,78,80,82,83,85,86,87,90,91,92,95,96,97,100,103,104,105,107,109,116,119,120,123,126,128,131,134,135,137,141,144,145,146,148,149,150,151,154,155,157,158,161,164,166,171,173,174,176,177,178,180,181,182,183,187,192,193,197,199,202,203,206,208,209,210,211,213,214,215,216,219,220,223,226,230,231,233,236,237,240,241,245,246,248,250,251,252,253,255,257,258,260,263,265,266,267,268,269,270,271,272,273,275,277,279,280,283,284,285,287,288,289,291,292,295,299,302,303,304,308,310,312,315,318,319,320,325,326,328,331,332,333,336,337,338,339,342,345,346,352,353,355,356,358,359,361,365,368,371,372,373,375,377,379,380,383,384,387,389,392,393,396,397,398,399,400,401,402,404,405,406,407,408,412,414,415,416,419,420,423,424,425,427,428,429,431,433,434,435,436,438,440,443,445,446,447,449,452,453,455,456,457,458,459,460,461,462,465,466,469,471,472,473,474,475,478,479,480,484,485,487,488,489,491,494,499,500,502,503,505,506,507,508,511,512,513,514,515,518,519,520,522,523,526,527,528,529,531,535,536,537,538,539,542,543,544,545,547,548,550,551,552,554,556,557,565,566,567,568,569,570,571,572,573,574,575,576,578,580,582,587,590,592,593,594,595,598,603,605,606,607,610,611,612,613,614,615,617,618,622,623,628,629,630,636,639,641,643,646,647,648,649,650,651,652,653,654,655,656,658,660,661,663,666,667,671,672,676,677,680,681,684,685,686,688,689,690,691,692,694,695,696,697,698,699,701,703,704,705,706,709,710,712,713,716,717,718,720,723,724,725,727,729,732,734,736,737,738,741,742,743,748,749,751,753,755,757,759,760,761,762,763,764,768,769,771,773,774,776,777,778,781,782,783,784,785,787,789,790,791,792,794,796,797,800,801,807,808,809,811,812,816,817,820,822,824,825,828,830,831,834,835,836,841,842,843,847,849,850,853,854,855,856,857,859,860,861,865,867,868,871,876,878,879,880,881,882,885,887,891,893,894,896,897,898,899,901,902,903,908,910,912,914,918,919,921,923,925,926,930,931,932,933,934,935,937,939,940,941,942,944,945,947,953,954,955,961,963,966,973,975,978,979,980,981,986,988,989,990,993,994,996,999,1001,1004,1005,1006,1007,1009,1010,1012,1013,1014,1016,1017,1021,1023,1025,1026,1028,1030,1032,1036,1037,1041,1044,1045,1046,1047,1049,1053,1054,1055,1056,1058,1059,1060,1061,1062,1063,1064,1065,1069,1071,1073,1074,1077,1078,1079,1081,1083,1084,1087,1090,1091,1094,1095,1098,1099,1101,1103,1105,1106,1108,1110,1112,1115,1117,1118,1122,1124,1125,1126,1127,1133,1134,1135,1136,1137,1138,1139,1141,1143,1144,1146,1148,1149,1152,1154,1155,1156,1160,1163,1164,1166,1168,1171,1172,1173,1174,1177,1179,1180,1182,1183,1184,1185,1186,1188,1189,1191,1201,1203,1204,1205,1206,1207,1208,1209,1210,1213,1215,1216,1217,1218,1219,1220,1221,1222,1225,1226,1228,1229,1230,1234,1235,1236,1237,1238,1242,1243,1246,1247,1248,1249,1250,1253,1255,1261,1262,1264,1265,1266,1267,1268,1272,1273,1274,1278,1281,1282,1287,1290,1291,1292,1293,1295,1299,1301,1302,1303,1306,1309,1310,1311,1313,1314,1315,1316,1317,1318,1320,1322,1325,1327,1328,1329,1331,1332,1333,1335,1336,1338,1339,1340,1341,1342,1345,1348,1352,1354,1355,1358,1360,1361,1363,1367,1369,1371,1373,1378,1379,1381,1382,1385,1386,1387,1388,1389,1392,1395,1396,1398,1401,1402,1403,1406,1408,1416,1417,1418,1422,1423,1434,1435,1437,1439,1440,1442,1447,1448,1450,1451,1452,1453,1454,1455,1456,1458,1461,1462,1464,1466,1467,1469,1471,1473,1474,1475,1477,1481,1491,1492,1496,1497,1499,1500,1501,1502,1503,1504,1505,1507,1510,1512,1513,1514,1516,1517,1520,1521,1523,1524,1527,1529,1530,1532,1535,1538,1539,1542,1543,1544,1545,1546,1550,1551,1552,1553,1554,1558,1562,1565,1566,1569,1570,1571,1572,1575,1580,1583,1586,1587,1589,1590,1591,1592,1594,1597,1601,1603,1604,1606,1609,1610,1611,1612,1614,1615,1616,1619,1620,1622,1626,1627,1630,1631,1632,1634,1639,1643,1645,1646,1650,1651,1656,1658,1661,1662,1663,1667,1670,1671,1679,1682,1689,1690,1691,1692,1693,1694,1695,1696,1697,1700,1702,1703,1704,1706,1707,1708,1712,1717,1718,1719,1720,1721,1725,1729,1730,1731,1732,1733,1736,1737,1739,1741,1742,1743,1744,1746,1748,1751,1752,1753,1754,1755,1756,1757,1758,1761,1763,1764,1765,1766,1767,1768,1769,1770,1771,1773,1775,1776,1778,1779,1782,1784,1787,1790,1791,1794,1795,1796,1798,1799,1800,1801,1803,1805,1807,1808,1809,1810,1811,1813,1818,1819,1820,1821,1823,1826,1828,1829,1831,1832,1833,1834,1835,1837,1839,1840,1841,1842,1843,1844,1845,1846,1848,1850,1851,1852,1853,1854,1855,1856,1857,1858,1859,1861,1862,1863,1864,1868,1870,1871,1872,1874,1875,1877,1879,1883,1884,1885,1886,1887,1893,1894,1895,1896,1898,1899,1902,1903,1904,1906,1908,1909,1911,1913,1915,1918,1920,1923,1925,1929,1931,1932,1933,1934,1935,1937,1940,1941,1942,1943,1945,1948,1952,1953,1954,1955,1956,1963,1964,1965,1969,1971,1973,1974,1975,1977,1978,1980,1983,1984,1987,1988,1989,1990,1991,1992,1996,1997,2000,2003,2005,2006,2008,2010,2011,2016,2018,2022,2023,2024,2025,2026,2030,2033,2034,2035,2036,2038,2041,2042,2045,2046,2047,2049,2050,2051,2054,2056,2059,2066,2067,2072,2074,2076,2077,2081,2082,2083,2085,2087,2088,2089,2090,2097,2101,2104,2105,2106,2107,2109,2111,2114,2115,2116,2119,2121,2125,2127,2130,2132,2135,2136,2137,2144,2145,2146,2149,2152,2153,2155,2156,2158,2160,2161,2163,2167,2169,2171,2172,2173,2174,2176,2179,2180,2183,2187,2188,2190,2191,2193,2195,2196,2199,2201,2202,2204,2205,2207,2209,2211,2212,2213,2219,2222,2224,2225,2229,2230,2231,2234,2236,2243,2244,2245,2246,2247,2249,2252,2253,2260,2263,2266,2267,2268,2269,2271,2272,2273,2274,2277,2279,2281,2282,2285,2287,2288,2289,2291,2293,2294,2298,2299,2300,2301,2304,2306,2307,2309,2310,2311,2316,2317,2318,2319,2320,2323,2324,2326,2327,2330,2332,2333,2334,2335,2336,2337,2338,2341,2342,2344,2345,2346,2347,2350,2351,2352,2354,2355,2360,2361,2363,2364,2366,2368,2370,2373,2377,2379,2380,2383,2385,2386,2388,2389,2390,2392,2395,2396,2397,2398,2401,2402,2405,2407,2408,2409,2410,2412,2413,2414,2416,2418,2420,2421,2422,2426,2427,2429,2431,2432,2433,2435,2438,2439,2440,2441,2444,2445,2447,2448,2451,2452,2455,2456,2457,2458,2459,2463,2464,2468,2469,2473,2474,2475,2476,2477,2482,2483,2484,2486,2487,2489,2491,2493,2499,2502,2503,2508,2511,2513,2514,2517,2519,2520,2521,2522,2523,2524,2525,2526,2529,2531,2532,2536,2538,2539,2540,2542,2543,2544,2545,2548,2553,2554,2557,2563,2564,2565,2567,2570,2571,2572,2574,2575,2577,2578,2583,2584,2587,2591,2593,2594,2596,2599,2602,2605,2606,2608,2610,2611,2614,2615,2616,2617,2618,2619,2620,2621,2623,2624,2627,2629,2633,2634,2636,2637,2639,2641,2642,2646,2647,2648,2651,2653,2654,2655,2656,2658,2662,2664,2665,2666,2667,2668,2669,2672,2673,2675,2680,2682,2683,2685,2687,2689,2690,2691,2693,2695],\"valid\":[0,1,3,7,9,15,18,20,22,24,29,30,39,40,48,52,58,60,63,72,74,75,76,79,88,93,94,99,102,106,108,110,111,112,113,114,117,121,124,127,130,132,139,143,147,159,160,165,169,170,175,179,184,188,190,191,194,198,200,201,204,205,207,212,217,218,224,225,227,238,239,242,244,247,256,259,261,262,264,278,281,286,290,293,296,298,300,305,306,307,311,316,321,322,323,329,330,334,335,341,347,348,349,357,366,367,369,370,374,378,381,382,386,394,395,410,413,418,422,426,430,432,437,442,444,448,450,451,454,463,467,468,476,477,481,482,490,492,495,496,497,504,509,510,516,517,525,530,532,540,541,553,555,559,560,562,577,579,581,584,585,586,588,601,604,608,620,621,624,631,632,633,634,635,637,642,644,645,664,665,668,669,675,678,679,682,683,707,708,714,715,719,726,728,730,731,739,740,747,750,752,754,766,767,793,799,802,803,805,814,818,821,823,826,827,832,838,845,851,852,858,863,864,866,869,872,874,875,877,883,884,886,889,892,895,905,907,909,913,922,924,927,928,929,936,938,943,946,950,956,957,958,964,967,968,970,976,977,982,984,985,987,991,998,1002,1008,1011,1018,1019,1024,1031,1039,1040,1042,1043,1067,1068,1070,1076,1080,1082,1093,1096,1097,1104,1111,1114,1116,1119,1121,1123,1128,1129,1130,1131,1140,1147,1151,1159,1167,1169,1175,1176,1181,1193,1194,1196,1197,1199,1212,1214,1223,1227,1231,1233,1239,1241,1244,1245,1251,1252,1258,1259,1260,1263,1269,1270,1271,1275,1276,1279,1283,1284,1285,1286,1288,1297,1298,1300,1304,1305,1307,1308,1312,1319,1321,1323,1326,1330,1337,1343,1346,1347,1356,1357,1362,1368,1374,1375,1376,1380,1383,1384,1390,1391,1393,1397,1399,1400,1404,1407,1410,1411,1412,1413,1415,1419,1420,1426,1427,1429,1431,1432,1433,1436,1438,1443,1446,1459,1465,1470,1478,1482,1486,1487,1489,1494,1495,1506,1509,1511,1526,1528,1533,1541,1548,1555,1557,1559,1561,1563,1564,1567,1568,1573,1574,1576,1577,1579,1582,1585,1588,1593,1595,1596,1605,1608,1617,1618,1621,1623,1625,1628,1633,1635,1636,1640,1642,1648,1649,1654,1657,1660,1665,1669,1672,1673,1678,1684,1685,1687,1688,1698,1699,1705,1711,1715,1722,1724,1728,1734,1735,1740,1745,1747,1750,1760,1762,1772,1781,1789,1792,1793,1797,1804,1814,1817,1836,1838,1847,1849,1860,1866,1867,1876,1878,1880,1882,1888,1890,1891,1892,1900,1901,1905,1914,1917,1921,1926,1928,1930,1938,1944,1946,1947,1949,1950,1951,1958,1959,1962,1966,1967,1970,1972,1976,1979,1981,1986,1994,1998,1999,2001,2002,2007,2009,2012,2014,2017,2019,2020,2021,2027,2028,2029,2037,2039,2040,2043,2044,2048,2053,2055,2058,2061,2062,2064,2065,2068,2069,2070,2073,2075,2078,2080,2084,2091,2094,2098,2100,2102,2103,2108,2113,2118,2122,2129,2131,2134,2138,2139,2140,2141,2143,2147,2150,2151,2154,2162,2165,2166,2168,2175,2177,2178,2181,2182,2184,2186,2200,2206,2208,2210,2215,2216,2217,2227,2237,2240,2242,2248,2250,2251,2254,2255,2256,2258,2261,2262,2264,2275,2280,2283,2284,2286,2290,2292,2296,2302,2303,2321,2322,2329,2331,2340,2343,2348,2349,2356,2365,2371,2372,2374,2375,2381,2384,2391,2394,2399,2403,2406,2417,2423,2425,2428,2436,2437,2443,2446,2449,2450,2454,2460,2461,2466,2470,2472,2478,2479,2481,2497,2500,2501,2504,2507,2515,2516,2518,2528,2535,2537,2541,2549,2550,2555,2556,2558,2559,2560,2561,2566,2568,2579,2580,2581,2585,2588,2590,2595,2597,2601,2603,2607,2609,2612,2613,2625,2628,2630,2631,2632,2635,2640,2644,2645,2659,2660,2661,2674,2676,2677,2678,2679,2684,2686,2692,2694,2696,2697,2698]},\"outer0_inner1\":{\"fit\":[0,1,3,4,6,7,9,15,18,20,22,24,25,27,29,30,31,35,37,39,40,42,44,45,48,49,51,52,54,55,56,57,58,60,62,63,65,68,69,72,74,75,76,77,79,80,82,83,85,88,91,92,93,94,95,97,99,100,102,105,106,108,110,111,112,113,114,117,121,124,127,128,130,132,134,139,143,144,145,147,148,151,154,158,159,160,165,166,169,170,171,175,177,179,183,184,187,188,190,191,194,197,198,200,201,204,205,206,207,209,211,212,214,215,216,217,218,220,223,224,225,227,230,231,233,236,237,238,239,242,244,246,247,248,251,255,256,257,258,259,260,261,262,264,266,267,268,270,271,272,273,277,278,280,281,283,286,290,291,292,293,295,296,298,300,302,303,305,306,307,308,311,316,318,321,322,323,329,330,334,335,337,339,341,345,346,347,348,349,352,355,357,358,361,366,367,369,370,371,372,374,375,377,378,379,381,382,383,384,386,387,389,393,394,395,396,397,400,401,407,408,410,412,413,414,416,418,419,420,422,424,425,426,427,428,429,430,432,434,435,436,437,440,442,444,445,446,448,449,450,451,454,455,458,460,461,463,466,467,468,469,471,473,474,476,477,478,479,480,481,482,484,485,488,489,490,491,492,494,495,496,497,500,502,503,504,505,507,509,510,512,513,516,517,520,522,523,525,526,527,530,532,538,539,540,541,543,544,551,552,553,555,559,560,562,572,573,576,577,579,581,582,584,585,586,588,590,592,594,598,601,604,606,607,608,611,613,615,620,621,622,623,624,628,630,631,632,633,634,635,636,637,639,642,643,644,645,646,647,648,650,655,658,661,663,664,665,667,668,669,671,675,678,679,680,681,682,683,684,688,689,694,698,701,703,704,706,707,708,712,714,715,719,723,726,728,730,731,732,736,737,738,739,740,741,742,747,749,750,751,752,754,759,766,767,776,781,783,785,790,791,792,793,796,797,799,800,802,803,805,807,808,809,814,818,820,821,822,823,824,825,826,827,832,834,835,836,838,843,845,851,852,853,855,858,860,863,864,865,866,867,869,872,874,875,877,882,883,884,885,886,887,889,892,894,895,896,897,898,902,903,905,907,909,910,912,913,918,919,922,923,924,925,927,928,929,930,932,933,934,935,936,938,940,942,943,944,946,950,953,954,955,956,957,958,961,964,967,968,970,973,975,976,977,978,982,984,985,987,990,991,993,994,996,998,999,1001,1002,1004,1005,1008,1009,1011,1013,1014,1017,1018,1019,1023,1024,1025,1030,1031,1039,1040,1042,1043,1044,1045,1053,1054,1058,1060,1061,1067,1068,1070,1071,1076,1079,1080,1081,1082,1083,1087,1090,1093,1094,1095,1096,1097,1098,1103,1104,1111,1112,1114,1116,1118,1119,1121,1122,1123,1124,1125,1128,1129,1130,1131,1133,1134,1139,1140,1141,1144,1147,1148,1149,1151,1152,1156,1159,1160,1166,1167,1169,1171,1173,1175,1176,1179,1180,1181,1183,1184,1185,1188,1189,1191,1193,1194,1196,1197,1199,1201,1203,1205,1207,1208,1212,1213,1214,1216,1217,1218,1220,1222,1223,1225,1226,1227,1228,1231,1233,1234,1235,1238,1239,1241,1243,1244,1245,1248,1250,1251,1252,1253,1258,1259,1260,1263,1266,1269,1270,1271,1274,1275,1276,1278,1279,1281,1283,1284,1285,1286,1287,1288,1290,1292,1297,1298,1299,1300,1301,1304,1305,1306,1307,1308,1309,1310,1312,1314,1315,1316,1317,1318,1319,1320,1321,1322,1323,1325,1326,1327,1329,1330,1332,1333,1335,1337,1338,1342,1343,1345,1346,1347,1352,1354,1355,1356,1357,1360,1362,1367,1368,1369,1373,1374,1375,1376,1380,1381,1382,1383,1384,1385,1386,1390,1391,1392,1393,1397,1398,1399,1400,1404,1406,1407,1410,1411,1412,1413,1415,1418,1419,1420,1422,1423,1426,1427,1429,1431,1432,1433,1436,1437,1438,1439,1443,1446,1447,1450,1455,1459,1461,1465,1467,1469,1470,1471,1478,1482,1486,1487,1489,1494,1495,1504,1505,1506,1509,1511,1512,1516,1523,1524,1526,1527,1528,1530,1533,1535,1538,1541,1542,1544,1546,1548,1550,1551,1552,1555,1557,1559,1561,1563,1564,1565,1566,1567,1568,1570,1571,1572,1573,1574,1575,1576,1577,1579,1580,1582,1585,1586,1587,1588,1589,1593,1595,1596,1597,1604,1605,1608,1609,1610,1612,1617,1618,1619,1620,1621,1623,1625,1627,1628,1630,1633,1634,1635,1636,1639,1640,1642,1643,1648,1649,1651,1654,1656,1657,1660,1665,1667,1669,1670,1672,1673,1678,1679,1682,1684,1685,1687,1688,1689,1690,1691,1692,1693,1694,1695,1696,1698,1699,1700,1703,1705,1706,1711,1715,1717,1719,1721,1722,1724,1725,1728,1729,1732,1733,1734,1735,1736,1737,1739,1740,1741,1743,1744,1745,1747,1748,1750,1754,1756,1757,1760,1761,1762,1764,1765,1766,1767,1771,1772,1781,1784,1789,1790,1791,1792,1793,1795,1796,1797,1800,1801,1804,1808,1810,1811,1813,1814,1817,1818,1819,1820,1821,1823,1831,1832,1836,1838,1840,1847,1849,1852,1855,1858,1860,1861,1864,1866,1867,1868,1872,1874,1875,1876,1877,1878,1880,1882,1883,1888,1890,1891,1892,1896,1898,1899,1900,1901,1902,1903,1904,1905,1908,1909,1911,1914,1915,1917,1918,1921,1923,1925,1926,1928,1930,1931,1932,1934,1938,1940,1944,1946,1947,1949,1950,1951,1953,1955,1956,1958,1959,1962,1966,1967,1970,1972,1974,1975,1976,1977,1978,1979,1981,1984,1986,1987,1988,1990,1991,1994,1998,1999,2001,2002,2005,2007,2008,2009,2012,2014,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2027,2028,2029,2033,2035,2037,2039,2040,2043,2044,2045,2046,2047,2048,2049,2050,2053,2054,2055,2056,2058,2059,2061,2062,2064,2065,2067,2068,2069,2070,2073,2074,2075,2076,2077,2078,2080,2081,2082,2084,2090,2091,2094,2097,2098,2100,2101,2102,2103,2104,2105,2106,2107,2108,2109,2113,2114,2118,2122,2125,2129,2130,2131,2134,2135,2136,2138,2139,2140,2141,2143,2144,2146,2147,2150,2151,2153,2154,2155,2156,2160,2162,2163,2165,2166,2168,2171,2172,2175,2177,2178,2181,2182,2183,2184,2186,2187,2188,2191,2195,2200,2206,2208,2209,2210,2212,2215,2216,2217,2222,2227,2229,2234,2237,2240,2242,2247,2248,2250,2251,2252,2254,2255,2256,2258,2261,2262,2264,2266,2268,2271,2272,2274,2275,2279,2280,2283,2284,2285,2286,2287,2289,2290,2292,2293,2296,2298,2299,2300,2302,2303,2306,2307,2309,2317,2320,2321,2322,2323,2324,2326,2327,2329,2331,2334,2337,2338,2340,2343,2344,2345,2348,2349,2350,2354,2355,2356,2363,2364,2365,2366,2371,2372,2374,2375,2379,2381,2384,2388,2389,2390,2391,2392,2394,2396,2397,2398,2399,2403,2406,2407,2409,2413,2414,2416,2417,2420,2422,2423,2425,2427,2428,2431,2432,2435,2436,2437,2439,2440,2441,2443,2444,2446,2447,2448,2449,2450,2451,2454,2455,2456,2457,2459,2460,2461,2463,2466,2470,2472,2477,2478,2479,2481,2483,2486,2489,2491,2493,2497,2500,2501,2502,2504,2507,2511,2514,2515,2516,2517,2518,2519,2520,2521,2522,2524,2525,2526,2528,2532,2535,2537,2538,2540,2541,2543,2545,2549,2550,2553,2554,2555,2556,2557,2558,2559,2560,2561,2563,2564,2565,2566,2568,2575,2578,2579,2580,2581,2583,2584,2585,2588,2590,2591,2594,2595,2596,2597,2601,2602,2603,2607,2609,2610,2612,2613,2614,2615,2616,2623,2624,2625,2628,2630,2631,2632,2633,2634,2635,2636,2639,2640,2642,2644,2645,2648,2653,2654,2656,2658,2659,2660,2661,2664,2665,2667,2672,2673,2674,2676,2677,2678,2679,2680,2683,2684,2686,2687,2690,2692,2694,2696,2697,2698],\"valid\":[2,5,14,17,28,33,34,41,46,47,50,53,61,64,70,71,78,86,87,90,96,103,104,107,109,116,119,120,123,126,131,135,137,141,146,149,150,155,157,161,164,173,174,176,178,180,181,182,192,193,199,202,203,208,210,213,219,226,240,241,245,250,252,253,263,265,269,275,279,284,285,287,288,289,299,304,310,312,315,319,320,325,326,328,331,332,333,336,338,342,353,356,359,365,368,373,380,392,398,399,402,404,405,406,415,423,431,433,438,443,447,452,453,456,457,459,462,465,472,475,487,499,506,508,511,514,515,518,519,528,529,531,535,536,537,542,545,547,548,550,554,556,557,565,566,567,568,569,570,571,574,575,578,580,587,593,595,603,605,610,612,614,617,618,629,641,649,651,652,653,654,656,660,666,672,676,677,685,686,690,691,692,695,696,697,699,705,709,710,713,716,717,718,720,724,725,727,729,734,743,748,753,755,757,760,761,762,763,764,768,769,771,773,774,777,778,782,784,787,789,794,801,811,812,816,817,828,830,831,841,842,847,849,850,854,856,857,859,861,868,871,876,878,879,880,881,891,893,899,901,908,914,921,926,931,937,939,941,945,947,963,966,979,980,981,986,988,989,1006,1007,1010,1012,1016,1021,1026,1028,1032,1036,1037,1041,1046,1047,1049,1055,1056,1059,1062,1063,1064,1065,1069,1073,1074,1077,1078,1084,1091,1099,1101,1105,1106,1108,1110,1115,1117,1126,1127,1135,1136,1137,1138,1143,1146,1154,1155,1163,1164,1168,1172,1174,1177,1182,1186,1204,1206,1209,1210,1215,1219,1221,1229,1230,1236,1237,1242,1246,1247,1249,1255,1261,1262,1264,1265,1267,1268,1272,1273,1282,1291,1293,1295,1302,1303,1311,1313,1328,1331,1336,1339,1340,1341,1348,1358,1361,1363,1371,1378,1379,1387,1388,1389,1395,1396,1401,1402,1403,1408,1416,1417,1434,1435,1440,1442,1448,1451,1452,1453,1454,1456,1458,1462,1464,1466,1473,1474,1475,1477,1481,1491,1492,1496,1497,1499,1500,1501,1502,1503,1507,1510,1513,1514,1517,1520,1521,1529,1532,1539,1543,1545,1553,1554,1558,1562,1569,1583,1590,1591,1592,1594,1601,1603,1606,1611,1614,1615,1616,1622,1626,1631,1632,1645,1646,1650,1658,1661,1662,1663,1671,1697,1702,1704,1707,1708,1712,1718,1720,1730,1731,1742,1746,1751,1752,1753,1755,1758,1763,1768,1769,1770,1773,1775,1776,1778,1779,1782,1787,1794,1798,1799,1803,1805,1807,1809,1826,1828,1829,1833,1834,1835,1837,1839,1841,1842,1843,1844,1845,1846,1848,1850,1851,1853,1854,1856,1857,1859,1862,1863,1870,1871,1879,1884,1885,1886,1887,1893,1894,1895,1906,1913,1920,1929,1933,1935,1937,1941,1942,1943,1945,1948,1952,1954,1963,1964,1965,1969,1971,1973,1980,1983,1989,1992,1996,1997,2000,2003,2006,2010,2011,2026,2030,2034,2036,2038,2041,2042,2051,2066,2072,2083,2085,2087,2088,2089,2111,2115,2116,2119,2121,2127,2132,2137,2145,2149,2152,2158,2161,2167,2169,2173,2174,2176,2179,2180,2190,2193,2196,2199,2201,2202,2204,2205,2207,2211,2213,2219,2224,2225,2230,2231,2236,2243,2244,2245,2246,2249,2253,2260,2263,2267,2269,2273,2277,2281,2282,2288,2291,2294,2301,2304,2310,2311,2316,2318,2319,2330,2332,2333,2335,2336,2341,2342,2346,2347,2351,2352,2360,2361,2368,2370,2373,2377,2380,2383,2385,2386,2395,2401,2402,2405,2408,2410,2412,2418,2421,2426,2429,2433,2438,2445,2452,2458,2464,2468,2469,2473,2474,2475,2476,2482,2484,2487,2499,2503,2508,2513,2523,2529,2531,2536,2539,2542,2544,2548,2567,2570,2571,2572,2574,2577,2587,2593,2599,2605,2606,2608,2611,2617,2618,2619,2620,2621,2627,2629,2637,2641,2646,2647,2651,2655,2662,2666,2668,2669,2675,2682,2685,2689,2691,2693,2695]},\"outer0_inner2\":{\"fit\":[0,1,2,3,5,7,9,14,15,17,18,20,22,24,28,29,30,33,34,39,40,41,46,47,48,50,52,53,58,60,61,63,64,70,71,72,74,75,76,78,79,86,87,88,90,93,94,96,99,102,103,104,106,107,108,109,110,111,112,113,114,116,117,119,120,121,123,124,126,127,130,131,132,135,137,139,141,143,146,147,149,150,155,157,159,160,161,164,165,169,170,173,174,175,176,178,179,180,181,182,184,188,190,191,192,193,194,198,199,200,201,202,203,204,205,207,208,210,212,213,217,218,219,224,225,226,227,238,239,240,241,242,244,245,247,250,252,253,256,259,261,262,263,264,265,269,275,278,279,281,284,285,286,287,288,289,290,293,296,298,299,300,304,305,306,307,310,311,312,315,316,319,320,321,322,323,325,326,328,329,330,331,332,333,334,335,336,338,341,342,347,348,349,353,356,357,359,365,366,367,368,369,370,373,374,378,380,381,382,386,392,394,395,398,399,402,404,405,406,410,413,415,418,422,423,426,430,431,432,433,437,438,442,443,444,447,448,450,451,452,453,454,456,457,459,462,463,465,467,468,472,475,476,477,481,482,487,490,492,495,496,497,499,504,506,508,509,510,511,514,515,516,517,518,519,525,528,529,530,531,532,535,536,537,540,541,542,545,547,548,550,553,554,555,556,557,559,560,562,565,566,567,568,569,570,571,574,575,577,578,579,580,581,584,585,586,587,588,593,595,601,603,604,605,608,610,612,614,617,618,620,621,624,629,631,632,633,634,635,637,641,642,644,645,649,651,652,653,654,656,660,664,665,666,668,669,672,675,676,677,678,679,682,683,685,686,690,691,692,695,696,697,699,705,707,708,709,710,713,714,715,716,717,718,719,720,724,725,726,727,728,729,730,731,734,739,740,743,747,748,750,752,753,754,755,757,760,761,762,763,764,766,767,768,769,771,773,774,777,778,782,784,787,789,793,794,799,801,802,803,805,811,812,814,816,817,818,821,823,826,827,828,830,831,832,838,841,842,845,847,849,850,851,852,854,856,857,858,859,861,863,864,866,868,869,871,872,874,875,876,877,878,879,880,881,883,884,886,889,891,892,893,895,899,901,905,907,908,909,913,914,921,922,924,926,927,928,929,931,936,937,938,939,941,943,945,946,947,950,956,957,958,963,964,966,967,968,970,976,977,979,980,981,982,984,985,986,987,988,989,991,998,1002,1006,1007,1008,1010,1011,1012,1016,1018,1019,1021,1024,1026,1028,1031,1032,1036,1037,1039,1040,1041,1042,1043,1046,1047,1049,1055,1056,1059,1062,1063,1064,1065,1067,1068,1069,1070,1073,1074,1076,1077,1078,1080,1082,1084,1091,1093,1096,1097,1099,1101,1104,1105,1106,1108,1110,1111,1114,1115,1116,1117,1119,1121,1123,1126,1127,1128,1129,1130,1131,1135,1136,1137,1138,1140,1143,1146,1147,1151,1154,1155,1159,1163,1164,1167,1168,1169,1172,1174,1175,1176,1177,1181,1182,1186,1193,1194,1196,1197,1199,1204,1206,1209,1210,1212,1214,1215,1219,1221,1223,1227,1229,1230,1231,1233,1236,1237,1239,1241,1242,1244,1245,1246,1247,1249,1251,1252,1255,1258,1259,1260,1261,1262,1263,1264,1265,1267,1268,1269,1270,1271,1272,1273,1275,1276,1279,1282,1283,1284,1285,1286,1288,1291,1293,1295,1297,1298,1300,1302,1303,1304,1305,1307,1308,1311,1312,1313,1319,1321,1323,1326,1328,1330,1331,1336,1337,1339,1340,1341,1343,1346,1347,1348,1356,1357,1358,1361,1362,1363,1368,1371,1374,1375,1376,1378,1379,1380,1383,1384,1387,1388,1389,1390,1391,1393,1395,1396,1397,1399,1400,1401,1402,1403,1404,1407,1408,1410,1411,1412,1413,1415,1416,1417,1419,1420,1426,1427,1429,1431,1432,1433,1434,1435,1436,1438,1440,1442,1443,1446,1448,1451,1452,1453,1454,1456,1458,1459,1462,1464,1465,1466,1470,1473,1474,1475,1477,1478,1481,1482,1486,1487,1489,1491,1492,1494,1495,1496,1497,1499,1500,1501,1502,1503,1506,1507,1509,1510,1511,1513,1514,1517,1520,1521,1526,1528,1529,1532,1533,1539,1541,1543,1545,1548,1553,1554,1555,1557,1558,1559,1561,1562,1563,1564,1567,1568,1569,1573,1574,1576,1577,1579,1582,1583,1585,1588,1590,1591,1592,1593,1594,1595,1596,1601,1603,1605,1606,1608,1611,1614,1615,1616,1617,1618,1621,1622,1623,1625,1626,1628,1631,1632,1633,1635,1636,1640,1642,1645,1646,1648,1649,1650,1654,1657,1658,1660,1661,1662,1663,1665,1669,1671,1672,1673,1678,1684,1685,1687,1688,1697,1698,1699,1702,1704,1705,1707,1708,1711,1712,1715,1718,1720,1722,1724,1728,1730,1731,1734,1735,1740,1742,1745,1746,1747,1750,1751,1752,1753,1755,1758,1760,1762,1763,1768,1769,1770,1772,1773,1775,1776,1778,1779,1781,1782,1787,1789,1792,1793,1794,1797,1798,1799,1803,1804,1805,1807,1809,1814,1817,1826,1828,1829,1833,1834,1835,1836,1837,1838,1839,1841,1842,1843,1844,1845,1846,1847,1848,1849,1850,1851,1853,1854,1856,1857,1859,1860,1862,1863,1866,1867,1870,1871,1876,1878,1879,1880,1882,1884,1885,1886,1887,1888,1890,1891,1892,1893,1894,1895,1900,1901,1905,1906,1913,1914,1917,1920,1921,1926,1928,1929,1930,1933,1935,1937,1938,1941,1942,1943,1944,1945,1946,1947,1948,1949,1950,1951,1952,1954,1958,1959,1962,1963,1964,1965,1966,1967,1969,1970,1971,1972,1973,1976,1979,1980,1981,1983,1986,1989,1992,1994,1996,1997,1998,1999,2000,2001,2002,2003,2006,2007,2009,2010,2011,2012,2014,2017,2019,2020,2021,2026,2027,2028,2029,2030,2034,2036,2037,2038,2039,2040,2041,2042,2043,2044,2048,2051,2053,2055,2058,2061,2062,2064,2065,2066,2068,2069,2070,2072,2073,2075,2078,2080,2083,2084,2085,2087,2088,2089,2091,2094,2098,2100,2102,2103,2108,2111,2113,2115,2116,2118,2119,2121,2122,2127,2129,2131,2132,2134,2137,2138,2139,2140,2141,2143,2145,2147,2149,2150,2151,2152,2154,2158,2161,2162,2165,2166,2167,2168,2169,2173,2174,2175,2176,2177,2178,2179,2180,2181,2182,2184,2186,2190,2193,2196,2199,2200,2201,2202,2204,2205,2206,2207,2208,2210,2211,2213,2215,2216,2217,2219,2224,2225,2227,2230,2231,2236,2237,2240,2242,2243,2244,2245,2246,2248,2249,2250,2251,2253,2254,2255,2256,2258,2260,2261,2262,2263,2264,2267,2269,2273,2275,2277,2280,2281,2282,2283,2284,2286,2288,2290,2291,2292,2294,2296,2301,2302,2303,2304,2310,2311,2316,2318,2319,2321,2322,2329,2330,2331,2332,2333,2335,2336,2340,2341,2342,2343,2346,2347,2348,2349,2351,2352,2356,2360,2361,2365,2368,2370,2371,2372,2373,2374,2375,2377,2380,2381,2383,2384,2385,2386,2391,2394,2395,2399,2401,2402,2403,2405,2406,2408,2410,2412,2417,2418,2421,2423,2425,2426,2428,2429,2433,2436,2437,2438,2443,2445,2446,2449,2450,2452,2454,2458,2460,2461,2464,2466,2468,2469,2470,2472,2473,2474,2475,2476,2478,2479,2481,2482,2484,2487,2497,2499,2500,2501,2503,2504,2507,2508,2513,2515,2516,2518,2523,2528,2529,2531,2535,2536,2537,2539,2541,2542,2544,2548,2549,2550,2555,2556,2558,2559,2560,2561,2566,2567,2568,2570,2571,2572,2574,2577,2579,2580,2581,2585,2587,2588,2590,2593,2595,2597,2599,2601,2603,2605,2606,2607,2608,2609,2611,2612,2613,2617,2618,2619,2620,2621,2625,2627,2628,2629,2630,2631,2632,2635,2637,2640,2641,2644,2645,2646,2647,2651,2655,2659,2660,2661,2662,2666,2668,2669,2674,2675,2676,2677,2678,2679,2682,2684,2685,2686,2689,2691,2692,2693,2694,2695,2696,2697,2698],\"valid\":[4,6,25,27,31,35,37,42,44,45,49,51,54,55,56,57,62,65,68,69,77,80,82,83,85,91,92,95,97,100,105,128,134,144,145,148,151,154,158,166,171,177,183,187,197,206,209,211,214,215,216,220,223,230,231,233,236,237,246,248,251,255,257,258,260,266,267,268,270,271,272,273,277,280,283,291,292,295,302,303,308,318,337,339,345,346,352,355,358,361,371,372,375,377,379,383,384,387,389,393,396,397,400,401,407,408,412,414,416,419,420,424,425,427,428,429,434,435,436,440,445,446,449,455,458,460,461,466,469,471,473,474,478,479,480,484,485,488,489,491,494,500,502,503,505,507,512,513,520,522,523,526,527,538,539,543,544,551,552,572,573,576,582,590,592,594,598,606,607,611,613,615,622,623,628,630,636,639,643,646,647,648,650,655,658,661,663,667,671,680,681,684,688,689,694,698,701,703,704,706,712,723,732,736,737,738,741,742,749,751,759,776,781,783,785,790,791,792,796,797,800,807,808,809,820,822,824,825,834,835,836,843,853,855,860,865,867,882,885,887,894,896,897,898,902,903,910,912,918,919,923,925,930,932,933,934,935,940,942,944,953,954,955,961,973,975,978,990,993,994,996,999,1001,1004,1005,1009,1013,1014,1017,1023,1025,1030,1044,1045,1053,1054,1058,1060,1061,1071,1079,1081,1083,1087,1090,1094,1095,1098,1103,1112,1118,1122,1124,1125,1133,1134,1139,1141,1144,1148,1149,1152,1156,1160,1166,1171,1173,1179,1180,1183,1184,1185,1188,1189,1191,1201,1203,1205,1207,1208,1213,1216,1217,1218,1220,1222,1225,1226,1228,1234,1235,1238,1243,1248,1250,1253,1266,1274,1278,1281,1287,1290,1292,1299,1301,1306,1309,1310,1314,1315,1316,1317,1318,1320,1322,1325,1327,1329,1332,1333,1335,1338,1342,1345,1352,1354,1355,1360,1367,1369,1373,1381,1382,1385,1386,1392,1398,1406,1418,1422,1423,1437,1439,1447,1450,1455,1461,1467,1469,1471,1504,1505,1512,1516,1523,1524,1527,1530,1535,1538,1542,1544,1546,1550,1551,1552,1565,1566,1570,1571,1572,1575,1580,1586,1587,1589,1597,1604,1609,1610,1612,1619,1620,1627,1630,1634,1639,1643,1651,1656,1667,1670,1679,1682,1689,1690,1691,1692,1693,1694,1695,1696,1700,1703,1706,1717,1719,1721,1725,1729,1732,1733,1736,1737,1739,1741,1743,1744,1748,1754,1756,1757,1761,1764,1765,1766,1767,1771,1784,1790,1791,1795,1796,1800,1801,1808,1810,1811,1813,1818,1819,1820,1821,1823,1831,1832,1840,1852,1855,1858,1861,1864,1868,1872,1874,1875,1877,1883,1896,1898,1899,1902,1903,1904,1908,1909,1911,1915,1918,1923,1925,1931,1932,1934,1940,1953,1955,1956,1974,1975,1977,1978,1984,1987,1988,1990,1991,2005,2008,2016,2018,2022,2023,2024,2025,2033,2035,2045,2046,2047,2049,2050,2054,2056,2059,2067,2074,2076,2077,2081,2082,2090,2097,2101,2104,2105,2106,2107,2109,2114,2125,2130,2135,2136,2144,2146,2153,2155,2156,2160,2163,2171,2172,2183,2187,2188,2191,2195,2209,2212,2222,2229,2234,2247,2252,2266,2268,2271,2272,2274,2279,2285,2287,2289,2293,2298,2299,2300,2306,2307,2309,2317,2320,2323,2324,2326,2327,2334,2337,2338,2344,2345,2350,2354,2355,2363,2364,2366,2379,2388,2389,2390,2392,2396,2397,2398,2407,2409,2413,2414,2416,2420,2422,2427,2431,2432,2435,2439,2440,2441,2444,2447,2448,2451,2455,2456,2457,2459,2463,2477,2483,2486,2489,2491,2493,2502,2511,2514,2517,2519,2520,2521,2522,2524,2525,2526,2532,2538,2540,2543,2545,2553,2554,2557,2563,2564,2565,2575,2578,2583,2584,2591,2594,2596,2602,2610,2614,2615,2616,2623,2624,2633,2634,2636,2639,2642,2648,2653,2654,2656,2658,2664,2665,2667,2672,2673,2680,2683,2687,2690]},\"outer1\":{\"fit\":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,15,16,17,19,21,22,23,24,25,26,27,28,29,30,31,32,33,34,36,37,38,39,40,41,42,43,44,45,46,47,48,50,51,52,53,54,55,56,58,59,60,61,62,63,65,66,67,68,69,70,71,72,73,74,75,76,78,79,80,81,82,84,85,86,87,88,89,90,91,93,94,96,97,98,101,102,104,105,109,113,114,115,116,118,119,120,121,122,123,125,126,127,128,129,130,132,133,135,136,138,139,140,142,143,144,145,147,149,151,152,153,154,155,156,159,161,162,163,164,165,166,167,168,169,171,172,173,175,177,179,181,183,184,185,186,188,189,190,191,193,195,196,198,199,200,201,202,203,204,206,207,208,210,211,212,213,216,217,218,219,221,222,223,225,226,228,229,231,232,233,234,235,237,238,240,241,242,243,244,245,246,247,249,251,253,254,255,256,259,261,262,264,267,270,271,272,273,274,275,276,277,278,279,280,281,282,284,285,288,289,291,292,293,294,295,297,301,302,303,304,305,306,307,308,309,310,312,313,314,316,317,318,321,322,323,324,326,327,328,329,331,332,334,335,336,337,338,339,340,342,343,344,345,347,348,349,350,351,352,354,355,356,357,359,360,362,363,364,365,366,367,368,369,371,372,374,376,377,378,379,380,381,382,384,385,386,387,388,389,390,391,392,393,394,395,396,398,399,401,402,403,405,406,407,408,409,411,412,413,414,416,417,418,421,422,423,424,425,426,427,429,430,431,432,434,435,436,437,438,439,440,441,444,445,446,447,448,449,450,451,454,455,457,458,459,461,462,464,465,466,467,468,469,470,471,472,473,474,475,476,477,479,480,481,483,485,486,487,488,489,490,491,493,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,512,513,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,544,545,546,547,549,550,553,554,555,556,557,558,559,560,561,562,563,564,565,567,568,569,571,574,576,577,578,579,580,581,583,585,587,588,589,590,591,592,593,595,596,597,599,600,601,602,604,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,647,648,650,651,652,653,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,673,674,675,678,680,681,682,683,684,685,686,687,688,689,691,692,693,694,696,697,698,699,700,701,702,703,704,705,706,707,708,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,728,730,731,733,735,736,737,739,740,741,744,745,746,747,749,750,751,753,754,756,757,758,759,761,762,763,765,766,767,770,771,772,775,776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793,794,795,796,797,798,800,802,803,804,805,806,807,808,809,810,811,812,813,814,815,816,817,818,819,820,821,822,824,825,828,829,830,831,832,833,834,835,837,838,839,840,841,843,844,846,847,848,849,851,852,853,855,856,857,859,860,861,862,863,864,866,867,868,869,870,871,872,873,874,875,876,877,878,879,880,881,882,883,884,886,888,890,891,892,895,896,900,901,902,903,904,905,906,907,910,911,912,913,914,915,916,917,918,920,921,922,923,924,925,926,927,928,930,932,933,934,935,936,938,940,941,943,945,946,947,948,949,950,951,952,953,954,956,957,958,959,960,961,962,963,964,965,967,968,969,970,971,972,973,974,975,976,979,980,981,982,983,984,985,986,987,988,989,990,991,992,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1024,1025,1026,1027,1028,1029,1030,1032,1033,1034,1035,1037,1038,1040,1043,1045,1046,1047,1048,1050,1051,1052,1053,1054,1055,1056,1057,1058,1059,1060,1061,1062,1063,1064,1065,1066,1067,1068,1069,1070,1071,1072,1073,1074,1075,1076,1077,1078,1079,1080,1081,1082,1083,1084,1085,1086,1088,1089,1090,1092,1093,1095,1096,1097,1098,1099,1100,1101,1102,1105,1107,1108,1109,1110,1111,1112,1113,1115,1116,1117,1118,1119,1120,1121,1122,1123,1124,1125,1126,1127,1128,1129,1131,1132,1133,1134,1135,1138,1139,1140,1141,1142,1145,1146,1147,1148,1150,1151,1153,1154,1156,1157,1158,1160,1161,1162,1163,1164,1165,1166,1167,1168,1169,1170,1172,1174,1175,1178,1179,1180,1181,1182,1184,1185,1186,1187,1188,1189,1190,1191,1192,1193,1194,1195,1196,1197,1198,1199,1200,1201,1202,1203,1204,1206,1207,1208,1210,1211,1212,1213,1214,1216,1217,1218,1219,1220,1221,1222,1223,1224,1225,1227,1228,1229,1232,1234,1235,1236,1237,1238,1239,1240,1241,1245,1246,1247,1250,1251,1252,1254,1255,1256,1257,1260,1263,1265,1266,1267,1268,1269,1270,1271,1272,1273,1274,1275,1276,1277,1278,1279,1280,1281,1282,1283,1284,1285,1286,1287,1288,1289,1290,1292,1294,1296,1297,1298,1299,1300,1301,1303,1304,1306,1307,1308,1309,1310,1312,1313,1314,1315,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325,1326,1328,1330,1331,1333,1334,1335,1336,1340,1342,1343,1344,1345,1346,1347,1348,1349,1350,1351,1352,1353,1354,1355,1357,1358,1359,1360,1362,1363,1364,1365,1366,1367,1369,1370,1371,1372,1373,1374,1375,1376,1377,1378,1379,1380,1381,1383,1384,1386,1387,1388,1389,1390,1391,1392,1393,1394,1395,1396,1398,1399,1400,1401,1402,1405,1408,1409,1411,1413,1414,1415,1416,1417,1419,1420,1421,1422,1423,1424,1425,1426,1427,1428,1429,1430,1431,1432,1434,1436,1437,1438,1439,1440,1441,1443,1444,1445,1447,1448,1449,1450,1451,1452,1456,1457,1460,1461,1462,1463,1465,1466,1467,1468,1470,1471,1472,1474,1475,1476,1478,1479,1480,1482,1483,1484,1485,1486,1487,1488,1489,1490,1492,1493,1494,1497,1498,1501,1502,1503,1504,1505,1506,1507,1508,1509,1510,1511,1513,1514,1515,1517,1518,1519,1521,1522,1523,1524,1525,1526,1527,1528,1530,1531,1532,1533,1534,1536,1537,1538,1539,1540,1541,1542,1546,1547,1548,1549,1550,1551,1552,1554,1555,1556,1558,1559,1560,1561,1563,1564,1565,1567,1568,1569,1570,1572,1573,1574,1575,1576,1577,1578,1579,1580,1581,1582,1583,1584,1588,1590,1591,1592,1595,1596,1597,1598,1599,1600,1602,1603,1604,1605,1607,1609,1610,1611,1612,1613,1614,1615,1616,1617,1620,1621,1622,1623,1624,1625,1626,1627,1629,1630,1631,1632,1633,1634,1636,1637,1638,1639,1640,1641,1642,1643,1644,1646,1647,1648,1649,1650,1651,1652,1653,1654,1655,1657,1659,1660,1661,1662,1663,1664,1665,1666,1667,1668,1669,1670,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682,1683,1684,1686,1687,1688,1692,1693,1694,1695,1697,1698,1700,1701,1703,1704,1705,1707,1709,1710,1712,1713,1714,1715,1716,1717,1721,1722,1723,1724,1725,1726,1727,1729,1730,1731,1732,1733,1735,1737,1738,1740,1741,1742,1743,1744,1745,1746,1747,1748,1749,1750,1753,1754,1756,1758,1759,1760,1762,1768,1769,1770,1771,1772,1774,1776,1777,1778,1779,1780,1781,1782,1783,1784,1785,1786,1787,1788,1789,1790,1792,1793,1794,1795,1796,1797,1798,1799,1800,1802,1803,1804,1805,1806,1807,1808,1809,1810,1811,1812,1814,1815,1816,1818,1819,1820,1821,1822,1823,1824,1825,1826,1827,1828,1829,1830,1831,1832,1833,1834,1836,1838,1839,1840,1841,1845,1846,1847,1848,1849,1850,1851,1852,1853,1854,1855,1856,1858,1859,1860,1861,1862,1863,1864,1865,1866,1867,1869,1871,1872,1873,1874,1875,1876,1877,1878,1879,1881,1882,1884,1885,1886,1887,1888,1889,1890,1891,1893,1894,1896,1897,1898,1899,1900,1901,1902,1904,1905,1906,1907,1908,1910,1911,1912,1913,1914,1915,1916,1917,1918,1919,1920,1921,1922,1923,1924,1925,1927,1928,1929,1931,1933,1935,1936,1937,1938,1939,1940,1941,1942,1943,1944,1945,1946,1947,1949,1950,1951,1953,1954,1955,1956,1957,1958,1959,1960,1961,1962,1963,1966,1967,1968,1970,1972,1973,1974,1975,1976,1977,1979,1980,1981,1982,1983,1984,1985,1988,1989,1991,1992,1993,1995,1996,1997,1999,2001,2002,2004,2005,2006,2009,2010,2011,2012,2013,2015,2017,2018,2019,2020,2022,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2035,2039,2040,2042,2043,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057,2060,2062,2063,2064,2065,2066,2068,2069,2070,2071,2074,2075,2076,2078,2079,2081,2082,2083,2085,2086,2087,2090,2091,2092,2093,2094,2095,2096,2097,2098,2099,2102,2103,2104,2105,2106,2108,2109,2110,2112,2113,2117,2118,2120,2121,2122,2123,2124,2125,2126,2127,2128,2129,2130,2131,2132,2133,2134,2136,2137,2138,2139,2141,2142,2143,2144,2145,2146,2147,2148,2149,2150,2151,2152,2153,2157,2158,2159,2160,2162,2163,2164,2165,2167,2168,2169,2170,2172,2173,2175,2176,2177,2178,2179,2180,2181,2182,2183,2184,2185,2186,2187,2188,2189,2190,2191,2192,2194,2196,2197,2198,2199,2200,2203,2204,2205,2206,2207,2208,2209,2210,2213,2214,2216,2217,2218,2219,2220,2221,2222,2223,2224,2225,2226,2227,2228,2229,2230,2232,2233,2234,2235,2236,2237,2238,2239,2240,2241,2242,2244,2245,2246,2248,2250,2252,2253,2255,2256,2257,2258,2259,2260,2261,2262,2263,2264,2265,2266,2270,2271,2273,2274,2275,2276,2277,2278,2279,2280,2281,2282,2283,2284,2285,2286,2287,2288,2290,2291,2292,2293,2294,2295,2296,2297,2298,2300,2302,2303,2305,2306,2307,2308,2312,2313,2314,2315,2317,2318,2320,2321,2322,2323,2324,2325,2328,2329,2331,2332,2333,2334,2335,2336,2337,2338,2339,2340,2341,2345,2346,2347,2350,2352,2353,2354,2357,2358,2359,2360,2361,2362,2363,2365,2366,2367,2369,2372,2376,2378,2380,2381,2382,2383,2384,2385,2386,2387,2389,2390,2391,2392,2393,2394,2395,2396,2399,2400,2403,2404,2405,2406,2408,2410,2411,2412,2413,2415,2416,2417,2419,2420,2421,2422,2423,2424,2425,2426,2427,2428,2430,2434,2436,2437,2440,2441,2442,2443,2444,2446,2449,2450,2453,2454,2456,2458,2461,2462,2463,2465,2466,2467,2468,2469,2470,2471,2472,2474,2476,2477,2478,2479,2480,2481,2482,2483,2484,2485,2486,2487,2488,2490,2492,2494,2495,2496,2497,2498,2500,2501,2502,2504,2505,2506,2507,2508,2509,2510,2511,2512,2513,2514,2515,2516,2517,2518,2519,2524,2525,2527,2528,2529,2530,2532,2533,2534,2535,2536,2537,2538,2539,2541,2542,2544,2546,2547,2550,2551,2552,2554,2555,2556,2558,2559,2560,2561,2562,2563,2564,2566,2567,2568,2569,2570,2571,2572,2573,2574,2575,2576,2577,2579,2581,2582,2584,2585,2586,2587,2588,2589,2590,2591,2592,2593,2594,2595,2597,2598,2599,2600,2601,2602,2604,2605,2606,2607,2609,2610,2611,2613,2614,2619,2621,2622,2624,2625,2626,2627,2628,2629,2630,2631,2632,2633,2634,2635,2636,2637,2638,2639,2640,2641,2642,2643,2644,2647,2648,2649,2650,2651,2652,2653,2654,2656,2657,2659,2660,2662,2663,2665,2666,2667,2669,2670,2671,2672,2673,2674,2675,2676,2677,2678,2679,2681,2682,2685,2686,2688,2689,2690,2691,2692,2693,2694,2695,2697,2698],\"valid\":[14,18,20,35,49,57,64,77,83,92,95,99,100,103,106,107,108,110,111,112,117,124,131,134,137,141,146,148,150,157,158,160,170,174,176,178,180,182,187,192,194,197,205,209,214,215,220,224,227,230,236,239,248,250,252,257,258,260,263,265,266,268,269,283,286,287,290,296,298,299,300,311,315,319,320,325,330,333,341,346,353,358,361,370,373,375,383,397,400,404,410,415,419,420,428,433,442,443,452,453,456,460,463,478,482,484,492,494,495,511,514,515,542,543,548,551,552,566,570,572,573,575,582,584,586,594,598,603,605,622,646,649,654,655,672,676,677,679,690,695,709,710,726,727,729,732,734,738,742,743,748,752,755,760,764,768,769,773,774,799,801,823,826,827,836,842,845,850,854,858,865,885,887,889,893,894,897,898,899,908,909,919,929,931,937,939,942,944,955,966,977,978,993,994,1006,1023,1031,1036,1039,1041,1042,1044,1049,1087,1091,1094,1103,1104,1106,1114,1130,1136,1137,1143,1144,1149,1152,1155,1159,1171,1173,1176,1177,1183,1205,1209,1215,1226,1230,1231,1233,1242,1243,1244,1248,1249,1253,1258,1259,1261,1262,1264,1291,1293,1295,1302,1305,1311,1327,1329,1332,1337,1338,1339,1341,1356,1361,1368,1382,1385,1397,1403,1404,1406,1407,1410,1412,1418,1433,1435,1442,1446,1453,1454,1455,1458,1459,1464,1469,1473,1477,1481,1491,1495,1496,1499,1500,1512,1516,1520,1529,1535,1543,1544,1545,1553,1557,1562,1566,1571,1585,1586,1587,1589,1593,1594,1601,1606,1608,1618,1619,1628,1635,1645,1656,1658,1671,1685,1689,1690,1691,1696,1699,1702,1706,1708,1711,1718,1719,1720,1728,1734,1736,1739,1751,1752,1755,1757,1761,1763,1764,1765,1766,1767,1773,1775,1791,1801,1813,1817,1835,1837,1842,1843,1844,1857,1868,1870,1880,1883,1892,1895,1903,1909,1926,1930,1932,1934,1948,1952,1964,1965,1969,1971,1978,1986,1987,1990,1994,1998,2000,2003,2007,2008,2014,2016,2021,2023,2034,2036,2037,2038,2041,2044,2058,2059,2061,2067,2072,2073,2077,2080,2084,2088,2089,2100,2101,2107,2111,2114,2115,2116,2119,2135,2140,2154,2155,2156,2161,2166,2171,2174,2193,2195,2201,2202,2211,2212,2215,2231,2243,2247,2249,2251,2254,2267,2268,2269,2272,2289,2299,2301,2304,2309,2310,2311,2316,2319,2326,2327,2330,2342,2343,2344,2348,2349,2351,2355,2356,2364,2368,2370,2371,2373,2374,2375,2377,2379,2388,2397,2398,2401,2402,2407,2409,2414,2418,2429,2431,2432,2433,2435,2438,2439,2445,2447,2448,2451,2452,2455,2457,2459,2460,2464,2473,2475,2489,2491,2493,2499,2503,2520,2521,2522,2523,2526,2531,2540,2543,2545,2548,2549,2553,2557,2565,2578,2580,2583,2596,2603,2608,2612,2615,2616,2617,2618,2620,2623,2645,2646,2655,2658,2661,2664,2668,2680,2683,2684,2687,2696]},\"outer1_inner0\":{\"fit\":[0,7,12,13,15,17,19,21,22,23,24,25,26,27,30,31,33,36,37,38,40,42,45,47,50,52,53,55,56,58,59,62,63,65,66,69,70,74,75,76,79,80,81,82,84,87,89,90,91,93,94,96,97,98,101,104,109,113,114,121,122,123,125,127,129,130,132,133,135,139,140,142,143,145,147,149,154,156,159,162,163,164,165,166,168,169,172,175,177,179,181,183,184,186,189,190,191,193,195,196,198,199,200,202,204,207,208,211,216,217,218,222,223,225,228,229,232,233,234,235,238,240,241,242,244,245,247,249,254,255,256,259,261,262,264,267,270,271,272,274,276,277,278,280,282,284,291,292,293,294,295,301,302,303,305,306,307,309,310,317,318,323,324,326,327,329,332,334,336,338,339,342,344,345,347,348,351,352,357,360,366,367,368,369,371,372,374,378,379,380,381,382,384,385,386,387,388,389,390,391,392,393,395,396,399,401,402,403,405,407,408,409,411,413,414,416,417,421,422,423,424,426,427,429,430,432,435,436,437,438,439,440,441,444,445,446,449,450,451,454,455,457,458,464,466,467,468,469,470,471,473,477,479,480,481,483,485,488,490,493,496,497,498,500,501,502,503,504,507,508,510,512,513,516,517,518,521,522,523,524,525,526,527,533,534,535,538,539,540,541,544,545,546,553,558,559,560,562,564,569,571,576,577,578,579,580,581,585,587,588,590,591,592,593,595,596,597,600,601,602,604,606,607,609,613,615,616,619,621,624,625,626,627,628,631,632,633,634,635,636,638,641,642,643,644,645,647,648,650,653,656,657,658,659,661,662,664,665,667,669,670,673,678,680,681,682,683,685,686,687,689,694,698,700,704,705,706,707,708,711,712,713,714,715,716,719,723,728,730,731,733,736,739,741,744,745,746,747,750,754,757,758,759,762,765,766,767,770,772,775,776,780,781,782,783,786,790,793,796,797,798,802,804,805,806,808,809,811,814,815,817,819,820,821,822,824,825,829,830,832,833,834,835,837,838,841,843,844,846,847,848,849,851,853,856,857,860,864,866,867,870,872,874,875,876,879,880,881,883,884,886,890,891,895,896,900,904,905,907,910,911,912,913,914,920,921,922,923,925,926,927,928,930,932,934,935,940,941,943,946,948,950,951,953,954,956,957,959,961,962,964,965,967,968,970,971,972,973,974,975,976,979,981,982,983,984,986,990,991,996,997,1001,1002,1003,1004,1008,1009,1010,1011,1013,1015,1018,1019,1020,1022,1024,1025,1027,1029,1030,1032,1033,1034,1037,1038,1043,1045,1046,1047,1048,1050,1051,1052,1053,1054,1055,1056,1057,1058,1060,1061,1062,1064,1066,1068,1071,1072,1075,1077,1078,1079,1080,1081,1082,1083,1089,1090,1095,1096,1097,1098,1099,1100,1107,1108,1109,1111,1113,1115,1117,1120,1122,1123,1124,1125,1128,1131,1132,1133,1134,1135,1138,1139,1140,1142,1145,1148,1150,1151,1154,1156,1157,1158,1161,1162,1163,1165,1166,1167,1168,1169,1170,1172,1175,1178,1180,1181,1182,1184,1187,1188,1189,1190,1191,1192,1193,1194,1196,1197,1198,1199,1202,1203,1204,1206,1207,1211,1212,1213,1214,1217,1218,1221,1222,1223,1225,1228,1229,1234,1235,1237,1238,1239,1240,1241,1250,1251,1252,1254,1256,1257,1260,1263,1265,1266,1267,1269,1272,1274,1275,1276,1277,1278,1279,1281,1283,1284,1285,1286,1287,1288,1290,1292,1296,1297,1299,1300,1301,1303,1304,1306,1307,1308,1309,1312,1313,1314,1315,1316,1318,1320,1323,1324,1325,1326,1328,1330,1333,1335,1336,1340,1342,1343,1344,1345,1346,1347,1351,1352,1353,1354,1357,1359,1362,1364,1365,1367,1369,1371,1373,1374,1376,1377,1379,1380,1381,1384,1386,1387,1388,1390,1391,1392,1393,1394,1398,1399,1400,1401,1402,1405,1411,1413,1414,1415,1417,1419,1420,1421,1422,1423,1424,1425,1426,1427,1428,1431,1432,1437,1438,1439,1441,1443,1444,1447,1449,1450,1451,1452,1457,1460,1461,1463,1465,1466,1470,1471,1472,1474,1478,1480,1483,1484,1485,1486,1487,1488,1489,1490,1494,1504,1505,1506,1507,1508,1510,1514,1517,1519,1522,1524,1526,1527,1528,1530,1531,1532,1533,1536,1537,1540,1541,1542,1549,1550,1551,1552,1554,1556,1559,1560,1561,1564,1565,1567,1568,1569,1570,1573,1575,1576,1577,1578,1579,1580,1581,1583,1584,1591,1595,1597,1598,1599,1602,1603,1609,1610,1612,1613,1617,1620,1621,1622,1625,1626,1630,1631,1633,1634,1640,1641,1643,1646,1647,1648,1649,1650,1651,1652,1653,1654,1655,1657,1660,1662,1664,1665,1667,1668,1678,1679,1680,1682,1683,1684,1687,1688,1693,1694,1695,1700,1701,1703,1704,1705,1709,1714,1717,1722,1725,1726,1729,1730,1731,1732,1733,1737,1740,1741,1742,1744,1746,1749,1750,1754,1760,1762,1768,1769,1771,1772,1774,1776,1781,1783,1784,1785,1786,1787,1788,1790,1792,1794,1795,1796,1797,1798,1799,1800,1802,1804,1806,1808,1810,1811,1812,1814,1816,1821,1822,1823,1824,1826,1827,1828,1831,1832,1836,1838,1839,1846,1847,1849,1852,1853,1854,1856,1861,1862,1864,1867,1869,1872,1873,1874,1876,1877,1878,1882,1886,1887,1888,1889,1891,1893,1896,1899,1900,1902,1904,1905,1907,1908,1910,1912,1914,1915,1916,1917,1918,1920,1921,1922,1923,1925,1927,1929,1931,1933,1936,1938,1939,1940,1942,1943,1945,1949,1950,1953,1954,1956,1957,1958,1959,1960,1961,1962,1963,1966,1967,1968,1972,1973,1974,1976,1979,1981,1982,1983,1984,1985,1988,1989,1991,1995,1996,1997,1999,2001,2002,2009,2010,2011,2012,2015,2017,2018,2019,2020,2022,2024,2025,2026,2027,2028,2029,2031,2032,2033,2035,2039,2042,2045,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057,2062,2063,2064,2065,2066,2069,2070,2075,2076,2078,2079,2083,2085,2090,2091,2092,2097,2098,2099,2102,2103,2104,2105,2108,2109,2110,2112,2118,2120,2121,2122,2123,2125,2126,2127,2130,2131,2133,2134,2136,2138,2139,2143,2144,2145,2146,2147,2148,2150,2151,2153,2158,2159,2162,2165,2167,2170,2172,2175,2177,2181,2182,2184,2186,2187,2188,2189,2190,2191,2194,2197,2200,2203,2204,2207,2209,2214,2216,2217,2218,2219,2220,2221,2222,2226,2227,2230,2232,2239,2240,2241,2242,2245,2248,2250,2252,2255,2256,2257,2258,2259,2260,2263,2264,2266,2270,2271,2274,2275,2276,2278,2280,2281,2282,2283,2284,2286,2287,2291,2292,2293,2296,2298,2300,2302,2303,2305,2306,2312,2313,2314,2317,2318,2320,2321,2323,2324,2329,2331,2333,2334,2337,2338,2339,2340,2341,2345,2350,2353,2354,2360,2361,2362,2363,2365,2367,2369,2372,2376,2378,2381,2382,2384,2386,2387,2390,2391,2393,2394,2395,2396,2399,2403,2404,2406,2408,2411,2412,2415,2416,2417,2419,2420,2422,2423,2425,2426,2427,2428,2430,2436,2437,2440,2443,2444,2450,2456,2462,2463,2465,2466,2467,2470,2472,2477,2478,2479,2481,2482,2483,2485,2486,2487,2488,2490,2492,2495,2498,2500,2501,2502,2505,2506,2507,2508,2509,2510,2511,2512,2513,2517,2518,2524,2525,2527,2528,2529,2530,2532,2534,2535,2536,2537,2538,2541,2546,2547,2550,2554,2555,2556,2558,2559,2561,2562,2563,2564,2567,2568,2569,2570,2572,2573,2575,2584,2585,2586,2587,2588,2589,2591,2593,2594,2595,2598,2599,2600,2602,2604,2606,2609,2614,2621,2622,2624,2626,2630,2631,2633,2634,2636,2638,2639,2640,2641,2644,2648,2649,2650,2651,2653,2654,2657,2660,2663,2665,2669,2670,2671,2672,2673,2674,2676,2677,2678,2679,2681,2682,2685,2686,2688,2690,2691,2692,2694,2695],\"valid\":[1,2,3,4,5,6,8,9,10,11,16,28,29,32,34,39,41,43,44,46,48,51,54,60,61,67,68,71,72,73,78,85,86,88,102,105,115,116,118,119,120,126,128,136,138,144,151,152,153,155,161,167,171,173,185,188,201,203,206,210,212,213,219,221,226,231,237,243,246,251,253,273,275,279,281,285,288,289,297,304,308,312,313,314,316,321,322,328,331,335,337,340,343,349,350,354,355,356,359,362,363,364,365,376,377,394,398,406,412,418,425,431,434,447,448,459,461,462,465,472,474,475,476,486,487,489,491,499,505,506,509,519,520,528,529,530,531,532,536,537,547,549,550,554,555,556,557,561,563,565,567,568,574,583,589,599,608,610,611,612,614,617,618,620,623,629,630,637,639,640,651,652,660,663,666,668,671,674,675,684,688,691,692,693,696,697,699,701,702,703,717,718,720,721,722,724,725,735,737,740,749,751,753,756,761,763,771,777,778,779,784,785,787,788,789,791,792,794,795,800,803,807,810,812,813,816,818,828,831,839,840,852,855,859,861,862,863,868,869,871,873,877,878,882,888,892,901,902,903,906,915,916,917,918,924,933,936,938,945,947,949,952,958,960,963,969,980,985,987,988,989,992,995,998,999,1000,1005,1007,1012,1014,1016,1017,1021,1026,1028,1035,1040,1059,1063,1065,1067,1069,1070,1073,1074,1076,1084,1085,1086,1088,1092,1093,1101,1102,1105,1110,1112,1116,1118,1119,1121,1126,1127,1129,1141,1146,1147,1153,1160,1164,1174,1179,1185,1186,1195,1200,1201,1208,1210,1216,1219,1220,1224,1227,1232,1236,1245,1246,1247,1255,1268,1270,1271,1273,1280,1282,1289,1294,1298,1310,1317,1319,1321,1322,1331,1334,1348,1349,1350,1355,1358,1360,1363,1366,1370,1372,1375,1378,1383,1389,1395,1396,1408,1409,1416,1429,1430,1434,1436,1440,1445,1448,1456,1462,1467,1468,1475,1476,1479,1482,1492,1493,1497,1498,1501,1502,1503,1509,1511,1513,1515,1518,1521,1523,1525,1534,1538,1539,1546,1547,1548,1555,1558,1563,1572,1574,1582,1588,1590,1592,1596,1600,1604,1605,1607,1611,1614,1615,1616,1623,1624,1627,1629,1632,1636,1637,1638,1639,1642,1644,1659,1661,1663,1666,1669,1670,1672,1673,1674,1675,1676,1677,1681,1686,1692,1697,1698,1707,1710,1712,1713,1715,1716,1721,1723,1724,1727,1735,1738,1743,1745,1747,1748,1753,1756,1758,1759,1770,1777,1778,1779,1780,1782,1789,1793,1803,1805,1807,1809,1815,1818,1819,1820,1825,1829,1830,1833,1834,1840,1841,1845,1848,1850,1851,1855,1858,1859,1860,1863,1865,1866,1871,1875,1879,1881,1884,1885,1890,1894,1897,1898,1901,1906,1911,1913,1919,1924,1928,1935,1937,1941,1944,1946,1947,1951,1955,1970,1975,1977,1980,1992,1993,2004,2005,2006,2013,2030,2040,2043,2046,2047,2060,2068,2071,2074,2081,2082,2086,2087,2093,2094,2095,2096,2106,2113,2117,2124,2128,2129,2132,2137,2141,2142,2149,2152,2157,2160,2163,2164,2168,2169,2173,2176,2178,2179,2180,2183,2185,2192,2196,2198,2199,2205,2206,2208,2210,2213,2223,2224,2225,2228,2229,2233,2234,2235,2236,2237,2238,2244,2246,2253,2261,2262,2265,2273,2277,2279,2285,2288,2290,2294,2295,2297,2307,2308,2315,2322,2325,2328,2332,2335,2336,2346,2347,2352,2357,2358,2359,2366,2380,2383,2385,2389,2392,2400,2405,2410,2413,2421,2424,2434,2441,2442,2446,2449,2453,2454,2458,2461,2468,2469,2471,2474,2476,2480,2484,2494,2496,2497,2504,2514,2515,2516,2519,2533,2539,2542,2544,2551,2552,2560,2566,2571,2574,2576,2577,2579,2581,2582,2590,2592,2597,2601,2605,2607,2610,2611,2613,2619,2625,2627,2628,2629,2632,2635,2637,2642,2643,2647,2652,2656,2659,2662,2666,2667,2675,2689,2693,2697,2698]},\"outer1_inner1\":{\"fit\":[1,2,3,4,5,6,8,9,10,11,12,15,16,17,19,21,23,27,28,29,32,34,38,39,41,43,44,46,47,48,51,53,54,55,60,61,66,67,68,71,72,73,74,76,78,79,80,81,85,86,88,90,97,98,102,105,115,116,118,119,120,122,123,126,127,128,130,132,133,136,138,142,143,144,151,152,153,155,156,161,162,164,165,166,167,169,171,173,175,177,183,185,188,191,193,196,198,200,201,202,203,206,210,212,213,219,221,223,226,229,231,233,234,237,238,243,244,245,246,251,253,261,262,267,271,273,274,275,277,279,281,282,285,288,289,291,297,302,303,304,308,310,312,313,314,316,317,318,321,322,324,328,331,332,335,337,340,342,343,344,345,347,349,350,352,354,355,356,359,362,363,364,365,366,367,369,371,374,376,377,379,385,386,387,389,390,391,392,394,396,398,406,407,409,412,414,416,417,418,424,425,430,431,434,435,437,438,440,441,444,445,446,447,448,449,458,459,461,462,464,465,466,467,468,470,471,472,474,475,476,480,481,483,486,487,489,490,491,496,498,499,504,505,506,508,509,510,512,513,516,517,519,520,521,522,524,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,544,545,547,549,550,554,555,556,557,558,560,561,562,563,564,565,567,568,571,574,576,577,580,583,585,587,589,590,597,599,601,602,606,608,610,611,612,613,614,616,617,618,619,620,623,624,626,629,630,632,635,636,637,639,640,641,642,643,644,645,647,648,650,651,652,653,656,659,660,661,662,663,666,668,671,674,675,680,684,688,691,692,693,696,697,699,701,702,703,704,706,707,711,712,713,714,717,718,719,720,721,722,724,725,735,736,737,740,741,744,746,747,749,751,753,754,756,758,761,762,763,765,766,767,771,772,775,777,778,779,781,784,785,786,787,788,789,791,792,793,794,795,798,800,803,805,807,809,810,811,812,813,814,815,816,817,818,819,822,828,829,830,831,833,835,837,838,839,840,841,843,844,846,848,851,852,853,855,857,859,861,862,863,866,867,868,869,871,872,873,874,875,877,878,880,882,884,886,888,891,892,896,900,901,902,903,906,910,911,912,913,915,916,917,918,921,923,924,926,928,930,933,936,938,940,941,943,945,946,947,948,949,952,954,956,957,958,960,961,963,967,969,970,971,980,981,982,983,984,985,987,988,989,991,992,995,997,998,999,1000,1003,1004,1005,1007,1008,1009,1012,1014,1015,1016,1017,1019,1020,1021,1022,1024,1026,1027,1028,1029,1032,1034,1035,1037,1038,1040,1043,1046,1048,1050,1052,1053,1055,1056,1057,1058,1059,1061,1062,1063,1064,1065,1067,1068,1069,1070,1071,1072,1073,1074,1076,1080,1084,1085,1086,1088,1089,1092,1093,1095,1097,1099,1101,1102,1105,1107,1108,1110,1111,1112,1113,1115,1116,1118,1119,1120,1121,1122,1123,1124,1126,1127,1129,1132,1133,1134,1135,1139,1141,1142,1146,1147,1148,1150,1153,1154,1156,1160,1161,1163,1164,1165,1168,1169,1170,1172,1174,1179,1180,1182,1184,1185,1186,1187,1189,1190,1192,1193,1195,1197,1199,1200,1201,1202,1203,1204,1208,1210,1211,1212,1216,1217,1218,1219,1220,1221,1222,1224,1227,1228,1229,1232,1236,1237,1238,1239,1241,1245,1246,1247,1254,1255,1256,1257,1263,1265,1266,1267,1268,1269,1270,1271,1273,1274,1277,1280,1281,1282,1284,1285,1287,1289,1290,1292,1294,1298,1300,1301,1304,1306,1307,1308,1310,1312,1313,1316,1317,1318,1319,1320,1321,1322,1326,1330,1331,1333,1334,1336,1342,1343,1345,1346,1347,1348,1349,1350,1352,1353,1354,1355,1358,1360,1363,1366,1367,1370,1371,1372,1375,1376,1377,1378,1380,1381,1383,1384,1387,1388,1389,1392,1395,1396,1398,1401,1405,1408,1409,1413,1415,1416,1419,1420,1421,1423,1424,1425,1428,1429,1430,1431,1434,1436,1439,1440,1443,1445,1447,1448,1449,1450,1452,1456,1457,1460,1462,1465,1467,1468,1471,1474,1475,1476,1478,1479,1482,1483,1484,1485,1486,1492,1493,1497,1498,1501,1502,1503,1506,1507,1508,1509,1511,1513,1515,1517,1518,1521,1522,1523,1524,1525,1530,1531,1532,1533,1534,1537,1538,1539,1540,1546,1547,1548,1549,1550,1554,1555,1558,1560,1561,1563,1567,1568,1570,1572,1573,1574,1576,1578,1580,1582,1584,1588,1590,1592,1596,1597,1598,1600,1602,1604,1605,1607,1611,1613,1614,1615,1616,1620,1623,1624,1625,1627,1629,1631,1632,1636,1637,1638,1639,1641,1642,1643,1644,1647,1648,1649,1650,1651,1652,1653,1654,1655,1657,1659,1660,1661,1663,1666,1667,1669,1670,1672,1673,1674,1675,1676,1677,1678,1681,1682,1686,1692,1693,1694,1697,1698,1700,1703,1704,1705,1707,1709,1710,1712,1713,1714,1715,1716,1721,1722,1723,1724,1726,1727,1733,1735,1738,1740,1741,1743,1745,1746,1747,1748,1750,1753,1754,1756,1758,1759,1762,1770,1772,1774,1776,1777,1778,1779,1780,1781,1782,1783,1784,1785,1789,1790,1792,1793,1796,1798,1800,1802,1803,1805,1806,1807,1808,1809,1812,1814,1815,1816,1818,1819,1820,1825,1829,1830,1833,1834,1839,1840,1841,1845,1847,1848,1850,1851,1855,1856,1858,1859,1860,1863,1865,1866,1869,1871,1874,1875,1876,1877,1879,1881,1884,1885,1886,1889,1890,1894,1896,1897,1898,1900,1901,1904,1906,1908,1910,1911,1912,1913,1915,1916,1917,1918,1919,1920,1921,1923,1924,1927,1928,1935,1936,1937,1938,1939,1940,1941,1944,1945,1946,1947,1949,1950,1951,1955,1957,1958,1960,1961,1962,1963,1968,1970,1972,1973,1974,1975,1976,1977,1980,1981,1983,1985,1988,1989,1991,1992,1993,1996,1999,2001,2002,2004,2005,2006,2009,2010,2013,2015,2019,2024,2025,2028,2030,2031,2033,2035,2040,2042,2043,2046,2047,2051,2052,2054,2056,2057,2060,2063,2065,2068,2069,2071,2074,2081,2082,2085,2086,2087,2092,2093,2094,2095,2096,2097,2099,2102,2106,2108,2109,2113,2117,2118,2120,2121,2122,2123,2124,2126,2127,2128,2129,2130,2132,2134,2136,2137,2141,2142,2143,2146,2149,2152,2157,2158,2160,2162,2163,2164,2165,2168,2169,2173,2176,2177,2178,2179,2180,2183,2184,2185,2190,2191,2192,2196,2197,2198,2199,2200,2203,2205,2206,2208,2209,2210,2213,2216,2217,2219,2220,2221,2222,2223,2224,2225,2226,2227,2228,2229,2233,2234,2235,2236,2237,2238,2240,2242,2244,2245,2246,2250,2253,2256,2259,2261,2262,2263,2265,2273,2275,2276,2277,2278,2279,2285,2287,2288,2290,2294,2295,2296,2297,2298,2305,2306,2307,2308,2312,2313,2315,2322,2325,2328,2329,2331,2332,2334,2335,2336,2337,2338,2345,2346,2347,2350,2352,2353,2354,2357,2358,2359,2360,2361,2366,2367,2369,2372,2376,2380,2381,2382,2383,2385,2386,2389,2390,2391,2392,2394,2400,2403,2404,2405,2406,2408,2410,2411,2412,2413,2416,2417,2419,2420,2421,2423,2424,2425,2428,2434,2436,2440,2441,2442,2443,2446,2449,2453,2454,2456,2458,2461,2466,2468,2469,2470,2471,2474,2476,2478,2480,2481,2484,2487,2488,2492,2494,2495,2496,2497,2501,2504,2505,2507,2511,2512,2514,2515,2516,2518,2519,2530,2533,2534,2538,2539,2542,2544,2546,2547,2551,2552,2554,2555,2556,2560,2564,2566,2567,2568,2571,2574,2575,2576,2577,2579,2581,2582,2588,2589,2590,2592,2597,2598,2601,2602,2605,2606,2607,2610,2611,2613,2619,2622,2625,2626,2627,2628,2629,2632,2635,2636,2637,2639,2640,2641,2642,2643,2644,2647,2651,2652,2653,2654,2656,2659,2660,2662,2663,2665,2666,2667,2670,2672,2673,2675,2676,2678,2679,2681,2689,2691,2692,2693,2697,2698],\"valid\":[0,7,13,22,24,25,26,30,31,33,36,37,40,42,45,50,52,56,58,59,62,63,65,69,70,75,82,84,87,89,91,93,94,96,101,104,109,113,114,121,125,129,135,139,140,145,147,149,154,159,163,168,172,179,181,184,186,189,190,195,199,204,207,208,211,216,217,218,222,225,228,232,235,240,241,242,247,249,254,255,256,259,264,270,272,276,278,280,284,292,293,294,295,301,305,306,307,309,323,326,327,329,334,336,338,339,348,351,357,360,368,372,378,380,381,382,384,388,393,395,399,401,402,403,405,408,411,413,421,422,423,426,427,429,432,436,439,450,451,454,455,457,469,473,477,479,485,488,493,497,500,501,502,503,507,518,523,525,526,546,553,559,569,578,579,581,588,591,592,593,595,596,600,604,607,609,615,621,625,627,628,631,633,634,638,657,658,664,665,667,669,670,673,678,681,682,683,685,686,687,689,694,698,700,705,708,715,716,723,728,730,731,733,739,745,750,757,759,770,776,780,782,783,790,796,797,802,804,806,808,820,821,824,825,832,834,847,849,856,860,864,870,876,879,881,883,890,895,904,905,907,914,920,922,925,927,932,934,935,950,951,953,959,962,964,965,968,972,973,974,975,976,979,986,990,996,1001,1002,1010,1011,1013,1018,1025,1030,1033,1045,1047,1051,1054,1060,1066,1075,1077,1078,1079,1081,1082,1083,1090,1096,1098,1100,1109,1117,1125,1128,1131,1138,1140,1145,1151,1157,1158,1162,1166,1167,1175,1178,1181,1188,1191,1194,1196,1198,1206,1207,1213,1214,1223,1225,1234,1235,1240,1250,1251,1252,1260,1272,1275,1276,1278,1279,1283,1286,1288,1296,1297,1299,1303,1309,1314,1315,1323,1324,1325,1328,1335,1340,1344,1351,1357,1359,1362,1364,1365,1369,1373,1374,1379,1386,1390,1391,1393,1394,1399,1400,1402,1411,1414,1417,1422,1426,1427,1432,1437,1438,1441,1444,1451,1461,1463,1466,1470,1472,1480,1487,1488,1489,1490,1494,1504,1505,1510,1514,1519,1526,1527,1528,1536,1541,1542,1551,1552,1556,1559,1564,1565,1569,1575,1577,1579,1581,1583,1591,1595,1599,1603,1609,1610,1612,1617,1621,1622,1626,1630,1633,1634,1640,1646,1662,1664,1665,1668,1679,1680,1683,1684,1687,1688,1695,1701,1717,1725,1729,1730,1731,1732,1737,1742,1744,1749,1760,1768,1769,1771,1786,1787,1788,1794,1795,1797,1799,1804,1810,1811,1821,1822,1823,1824,1826,1827,1828,1831,1832,1836,1838,1846,1849,1852,1853,1854,1861,1862,1864,1867,1872,1873,1878,1882,1887,1888,1891,1893,1899,1902,1905,1907,1914,1922,1925,1929,1931,1933,1942,1943,1953,1954,1956,1959,1966,1967,1979,1982,1984,1995,1997,2011,2012,2017,2018,2020,2022,2026,2027,2029,2032,2039,2045,2048,2049,2050,2053,2055,2062,2064,2066,2070,2075,2076,2078,2079,2083,2090,2091,2098,2103,2104,2105,2110,2112,2125,2131,2133,2138,2139,2144,2145,2147,2148,2150,2151,2153,2159,2167,2170,2172,2175,2181,2182,2186,2187,2188,2189,2194,2204,2207,2214,2218,2230,2232,2239,2241,2248,2252,2255,2257,2258,2260,2264,2266,2270,2271,2274,2280,2281,2282,2283,2284,2286,2291,2292,2293,2300,2302,2303,2314,2317,2318,2320,2321,2323,2324,2333,2339,2340,2341,2362,2363,2365,2378,2384,2387,2393,2395,2396,2399,2415,2422,2426,2427,2430,2437,2444,2450,2462,2463,2465,2467,2472,2477,2479,2482,2483,2485,2486,2490,2498,2500,2502,2506,2508,2509,2510,2513,2517,2524,2525,2527,2528,2529,2532,2535,2536,2537,2541,2550,2558,2559,2561,2562,2563,2569,2570,2572,2573,2584,2585,2586,2587,2591,2593,2594,2595,2599,2600,2604,2609,2614,2621,2624,2630,2631,2633,2634,2638,2648,2649,2650,2657,2669,2671,2674,2677,2682,2685,2686,2688,2690,2694,2695]},\"outer1_inner2\":{\"fit\":[0,1,2,3,4,5,6,7,8,9,10,11,13,16,22,24,25,26,28,29,30,31,32,33,34,36,37,39,40,41,42,43,44,45,46,48,50,51,52,54,56,58,59,60,61,62,63,65,67,68,69,70,71,72,73,75,78,82,84,85,86,87,88,89,91,93,94,96,101,102,104,105,109,113,114,115,116,118,119,120,121,125,126,128,129,135,136,138,139,140,144,145,147,149,151,152,153,154,155,159,161,163,167,168,171,172,173,179,181,184,185,186,188,189,190,195,199,201,203,204,206,207,208,210,211,212,213,216,217,218,219,221,222,225,226,228,231,232,235,237,240,241,242,243,246,247,249,251,253,254,255,256,259,264,270,272,273,275,276,278,279,280,281,284,285,288,289,292,293,294,295,297,301,304,305,306,307,308,309,312,313,314,316,321,322,323,326,327,328,329,331,334,335,336,337,338,339,340,343,348,349,350,351,354,355,356,357,359,360,362,363,364,365,368,372,376,377,378,380,381,382,384,388,393,394,395,398,399,401,402,403,405,406,408,411,412,413,418,421,422,423,425,426,427,429,431,432,434,436,439,447,448,450,451,454,455,457,459,461,462,465,469,472,473,474,475,476,477,479,485,486,487,488,489,491,493,497,499,500,501,502,503,505,506,507,509,518,519,520,523,525,526,528,529,530,531,532,536,537,546,547,549,550,553,554,555,556,557,559,561,563,565,567,568,569,574,578,579,581,583,588,589,591,592,593,595,596,599,600,604,607,608,609,610,611,612,614,615,617,618,620,621,623,625,627,628,629,630,631,633,634,637,638,639,640,651,652,657,658,660,663,664,665,666,667,668,669,670,671,673,674,675,678,681,682,683,684,685,686,687,688,689,691,692,693,694,696,697,698,699,700,701,702,703,705,708,715,716,717,718,720,721,722,723,724,725,728,730,731,733,735,737,739,740,745,749,750,751,753,756,757,759,761,763,770,771,776,777,778,779,780,782,783,784,785,787,788,789,790,791,792,794,795,796,797,800,802,803,804,806,807,808,810,812,813,816,818,820,821,824,825,828,831,832,834,839,840,847,849,852,855,856,859,860,861,862,863,864,868,869,870,871,873,876,877,878,879,881,882,883,888,890,892,895,901,902,903,904,905,906,907,914,915,916,917,918,920,922,924,925,927,932,933,934,935,936,938,945,947,949,950,951,952,953,958,959,960,962,963,964,965,968,969,972,973,974,975,976,979,980,985,986,987,988,989,990,992,995,996,998,999,1000,1001,1002,1005,1007,1010,1011,1012,1013,1014,1016,1017,1018,1021,1025,1026,1028,1030,1033,1035,1040,1045,1047,1051,1054,1059,1060,1063,1065,1066,1067,1069,1070,1073,1074,1075,1076,1077,1078,1079,1081,1082,1083,1084,1085,1086,1088,1090,1092,1093,1096,1098,1100,1101,1102,1105,1109,1110,1112,1116,1117,1118,1119,1121,1125,1126,1127,1128,1129,1131,1138,1140,1141,1145,1146,1147,1151,1153,1157,1158,1160,1162,1164,1166,1167,1174,1175,1178,1179,1181,1185,1186,1188,1191,1194,1195,1196,1198,1200,1201,1206,1207,1208,1210,1213,1214,1216,1219,1220,1223,1224,1225,1227,1232,1234,1235,1236,1240,1245,1246,1247,1250,1251,1252,1255,1260,1268,1270,1271,1272,1273,1275,1276,1278,1279,1280,1282,1283,1286,1288,1289,1294,1296,1297,1298,1299,1303,1309,1310,1314,1315,1317,1319,1321,1322,1323,1324,1325,1328,1331,1334,1335,1340,1344,1348,1349,1350,1351,1355,1357,1358,1359,1360,1362,1363,1364,1365,1366,1369,1370,1372,1373,1374,1375,1378,1379,1383,1386,1389,1390,1391,1393,1394,1395,1396,1399,1400,1402,1408,1409,1411,1414,1416,1417,1422,1426,1427,1429,1430,1432,1434,1436,1437,1438,1440,1441,1444,1445,1448,1451,1456,1461,1462,1463,1466,1467,1468,1470,1472,1475,1476,1479,1480,1482,1487,1488,1489,1490,1492,1493,1494,1497,1498,1501,1502,1503,1504,1505,1509,1510,1511,1513,1514,1515,1518,1519,1521,1523,1525,1526,1527,1528,1534,1536,1538,1539,1541,1542,1546,1547,1548,1551,1552,1555,1556,1558,1559,1563,1564,1565,1569,1572,1574,1575,1577,1579,1581,1582,1583,1588,1590,1591,1592,1595,1596,1599,1600,1603,1604,1605,1607,1609,1610,1611,1612,1614,1615,1616,1617,1621,1622,1623,1624,1626,1627,1629,1630,1632,1633,1634,1636,1637,1638,1639,1640,1642,1644,1646,1659,1661,1662,1663,1664,1665,1666,1668,1669,1670,1672,1673,1674,1675,1676,1677,1679,1680,1681,1683,1684,1686,1687,1688,1692,1695,1697,1698,1701,1707,1710,1712,1713,1715,1716,1717,1721,1723,1724,1725,1727,1729,1730,1731,1732,1735,1737,1738,1742,1743,1744,1745,1747,1748,1749,1753,1756,1758,1759,1760,1768,1769,1770,1771,1777,1778,1779,1780,1782,1786,1787,1788,1789,1793,1794,1795,1797,1799,1803,1804,1805,1807,1809,1810,1811,1815,1818,1819,1820,1821,1822,1823,1824,1825,1826,1827,1828,1829,1830,1831,1832,1833,1834,1836,1838,1840,1841,1845,1846,1848,1849,1850,1851,1852,1853,1854,1855,1858,1859,1860,1861,1862,1863,1864,1865,1866,1867,1871,1872,1873,1875,1878,1879,1881,1882,1884,1885,1887,1888,1890,1891,1893,1894,1897,1898,1899,1901,1902,1905,1906,1907,1911,1913,1914,1919,1922,1924,1925,1928,1929,1931,1933,1935,1937,1941,1942,1943,1944,1946,1947,1951,1953,1954,1955,1956,1959,1966,1967,1970,1975,1977,1979,1980,1982,1984,1992,1993,1995,1997,2004,2005,2006,2011,2012,2013,2017,2018,2020,2022,2026,2027,2029,2030,2032,2039,2040,2043,2045,2046,2047,2048,2049,2050,2053,2055,2060,2062,2064,2066,2068,2070,2071,2074,2075,2076,2078,2079,2081,2082,2083,2086,2087,2090,2091,2093,2094,2095,2096,2098,2103,2104,2105,2106,2110,2112,2113,2117,2124,2125,2128,2129,2131,2132,2133,2137,2138,2139,2141,2142,2144,2145,2147,2148,2149,2150,2151,2152,2153,2157,2159,2160,2163,2164,2167,2168,2169,2170,2172,2173,2175,2176,2178,2179,2180,2181,2182,2183,2185,2186,2187,2188,2189,2192,2194,2196,2198,2199,2204,2205,2206,2207,2208,2210,2213,2214,2218,2223,2224,2225,2228,2229,2230,2232,2233,2234,2235,2236,2237,2238,2239,2241,2244,2246,2248,2252,2253,2255,2257,2258,2260,2261,2262,2264,2265,2266,2270,2271,2273,2274,2277,2279,2280,2281,2282,2283,2284,2285,2286,2288,2290,2291,2292,2293,2294,2295,2297,2300,2302,2303,2307,2308,2314,2315,2317,2318,2320,2321,2322,2323,2324,2325,2328,2332,2333,2335,2336,2339,2340,2341,2346,2347,2352,2357,2358,2359,2362,2363,2365,2366,2378,2380,2383,2384,2385,2387,2389,2392,2393,2395,2396,2399,2400,2405,2410,2413,2415,2421,2422,2424,2426,2427,2430,2434,2437,2441,2442,2444,2446,2449,2450,2453,2454,2458,2461,2462,2463,2465,2467,2468,2469,2471,2472,2474,2476,2477,2479,2480,2482,2483,2484,2485,2486,2490,2494,2496,2497,2498,2500,2502,2504,2506,2508,2509,2510,2513,2514,2515,2516,2517,2519,2524,2525,2527,2528,2529,2532,2533,2535,2536,2537,2539,2541,2542,2544,2550,2551,2552,2558,2559,2560,2561,2562,2563,2566,2569,2570,2571,2572,2573,2574,2576,2577,2579,2581,2582,2584,2585,2586,2587,2590,2591,2592,2593,2594,2595,2597,2599,2600,2601,2604,2605,2607,2609,2610,2611,2613,2614,2619,2621,2624,2625,2627,2628,2629,2630,2631,2632,2633,2634,2635,2637,2638,2642,2643,2647,2648,2649,2650,2652,2656,2657,2659,2662,2666,2667,2669,2671,2674,2675,2677,2682,2685,2686,2688,2689,2690,2693,2694,2695,2697,2698],\"valid\":[12,15,17,19,21,23,27,38,47,53,55,66,74,76,79,80,81,90,97,98,122,123,127,130,132,133,142,143,156,162,164,165,166,169,175,177,183,191,193,196,198,200,202,223,229,233,234,238,244,245,261,262,267,271,274,277,282,291,302,303,310,317,318,324,332,342,344,345,347,352,366,367,369,371,374,379,385,386,387,389,390,391,392,396,407,409,414,416,417,424,430,435,437,438,440,441,444,445,446,449,458,464,466,467,468,470,471,480,481,483,490,496,498,504,508,510,512,513,516,517,521,522,524,527,533,534,535,538,539,540,541,544,545,558,560,562,564,571,576,577,580,585,587,590,597,601,602,606,613,616,619,624,626,632,635,636,641,642,643,644,645,647,648,650,653,656,659,661,662,680,704,706,707,711,712,713,714,719,736,741,744,746,747,754,758,762,765,766,767,772,775,781,786,793,798,805,809,811,814,815,817,819,822,829,830,833,835,837,838,841,843,844,846,848,851,853,857,866,867,872,874,875,880,884,886,891,896,900,910,911,912,913,921,923,926,928,930,940,941,943,946,948,954,956,957,961,967,970,971,981,982,983,984,991,997,1003,1004,1008,1009,1015,1019,1020,1022,1024,1027,1029,1032,1034,1037,1038,1043,1046,1048,1050,1052,1053,1055,1056,1057,1058,1061,1062,1064,1068,1071,1072,1080,1089,1095,1097,1099,1107,1108,1111,1113,1115,1120,1122,1123,1124,1132,1133,1134,1135,1139,1142,1148,1150,1154,1156,1161,1163,1165,1168,1169,1170,1172,1180,1182,1184,1187,1189,1190,1192,1193,1197,1199,1202,1203,1204,1211,1212,1217,1218,1221,1222,1228,1229,1237,1238,1239,1241,1254,1256,1257,1263,1265,1266,1267,1269,1274,1277,1281,1284,1285,1287,1290,1292,1300,1301,1304,1306,1307,1308,1312,1313,1316,1318,1320,1326,1330,1333,1336,1342,1343,1345,1346,1347,1352,1353,1354,1367,1371,1376,1377,1380,1381,1384,1387,1388,1392,1398,1401,1405,1413,1415,1419,1420,1421,1423,1424,1425,1428,1431,1439,1443,1447,1449,1450,1452,1457,1460,1465,1471,1474,1478,1483,1484,1485,1486,1506,1507,1508,1517,1522,1524,1530,1531,1532,1533,1537,1540,1549,1550,1554,1560,1561,1567,1568,1570,1573,1576,1578,1580,1584,1597,1598,1602,1613,1620,1625,1631,1641,1643,1647,1648,1649,1650,1651,1652,1653,1654,1655,1657,1660,1667,1678,1682,1693,1694,1700,1703,1704,1705,1709,1714,1722,1726,1733,1740,1741,1746,1750,1754,1762,1772,1774,1776,1781,1783,1784,1785,1790,1792,1796,1798,1800,1802,1806,1808,1812,1814,1816,1839,1847,1856,1869,1874,1876,1877,1886,1889,1896,1900,1904,1908,1910,1912,1915,1916,1917,1918,1920,1921,1923,1927,1936,1938,1939,1940,1945,1949,1950,1957,1958,1960,1961,1962,1963,1968,1972,1973,1974,1976,1981,1983,1985,1988,1989,1991,1996,1999,2001,2002,2009,2010,2015,2019,2024,2025,2028,2031,2033,2035,2042,2051,2052,2054,2056,2057,2063,2065,2069,2085,2092,2097,2099,2102,2108,2109,2118,2120,2121,2122,2123,2126,2127,2130,2134,2136,2143,2146,2158,2162,2165,2177,2184,2190,2191,2197,2200,2203,2209,2216,2217,2219,2220,2221,2222,2226,2227,2240,2242,2245,2250,2256,2259,2263,2275,2276,2278,2287,2296,2298,2305,2306,2312,2313,2329,2331,2334,2337,2338,2345,2350,2353,2354,2360,2361,2367,2369,2372,2376,2381,2382,2386,2390,2391,2394,2403,2404,2406,2408,2411,2412,2416,2417,2419,2420,2423,2425,2428,2436,2440,2443,2456,2466,2470,2478,2481,2487,2488,2492,2495,2501,2505,2507,2511,2512,2518,2530,2534,2538,2546,2547,2554,2555,2556,2564,2567,2568,2575,2588,2589,2598,2602,2606,2622,2626,2636,2639,2640,2641,2644,2651,2653,2654,2660,2663,2665,2670,2672,2673,2676,2678,2679,2681,2691,2692]},\"outer2\":{\"fit\":[1,2,3,4,5,6,7,8,9,10,11,12,13,14,16,17,18,19,20,21,22,23,26,27,32,34,35,36,38,39,40,41,42,43,44,45,48,49,50,51,52,54,55,56,57,59,62,63,64,65,66,67,68,69,70,71,72,73,74,76,77,79,80,81,83,84,85,87,88,89,92,95,96,97,98,99,100,101,103,104,105,106,107,108,109,110,111,112,113,115,117,118,119,122,123,124,125,127,128,129,131,132,133,134,136,137,138,140,141,142,144,145,146,147,148,149,150,151,152,153,154,156,157,158,159,160,162,163,164,165,166,167,168,170,171,172,173,174,175,176,177,178,179,180,182,183,185,186,187,189,190,191,192,194,195,196,197,198,199,200,203,204,205,206,207,208,209,210,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,246,247,248,249,250,251,252,253,254,257,258,259,260,263,265,266,267,268,269,270,272,273,274,275,276,277,279,280,281,282,283,284,286,287,288,289,290,291,293,294,296,297,298,299,300,301,302,303,304,305,306,308,309,311,313,314,315,317,318,319,320,321,322,323,324,325,326,327,328,329,330,333,335,336,337,339,340,341,342,343,344,345,346,347,348,350,351,352,353,354,355,357,358,360,361,362,363,364,367,368,369,370,371,372,373,374,375,376,377,379,383,385,386,387,388,389,390,391,392,393,396,397,398,399,400,401,402,403,404,405,406,408,409,410,411,412,413,414,415,417,418,419,420,421,422,423,425,428,430,433,434,435,436,437,439,440,441,442,443,444,445,446,447,448,449,451,452,453,455,456,457,458,459,460,461,463,464,465,466,467,469,470,471,472,474,475,477,478,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,517,519,520,521,522,524,525,527,528,530,532,533,534,535,536,537,539,540,541,542,543,544,545,546,547,548,549,551,552,554,555,556,558,559,561,563,564,566,567,569,570,572,573,574,575,576,577,580,582,583,584,585,586,588,589,590,591,593,594,595,596,597,598,599,600,602,603,605,606,607,609,610,611,612,614,615,616,617,619,621,622,623,624,625,626,627,628,629,630,634,635,636,637,638,639,640,642,643,644,645,646,648,649,650,651,654,655,656,657,659,661,662,663,664,666,668,670,671,672,673,674,675,676,677,679,680,684,685,687,688,690,691,693,694,695,697,699,700,701,702,703,704,705,707,709,710,711,714,716,717,718,719,720,721,722,726,727,728,729,730,731,732,733,734,735,736,737,738,742,743,744,745,746,747,748,749,750,751,752,753,755,756,758,759,760,761,762,763,764,765,766,767,768,769,770,772,773,774,775,776,777,779,780,781,782,784,785,786,787,788,790,791,792,793,794,795,796,797,798,799,800,801,802,804,806,807,808,809,810,811,813,814,815,816,817,818,819,820,821,822,823,826,827,828,829,830,833,834,835,836,837,838,839,840,842,843,844,845,846,847,848,849,850,852,854,855,856,857,858,859,862,864,865,866,868,870,871,873,875,876,877,878,880,881,882,883,885,886,887,888,889,890,891,892,893,894,896,897,898,899,900,901,902,903,904,906,908,909,910,911,913,915,916,917,918,919,920,921,922,923,928,929,930,931,932,933,935,936,937,939,940,941,942,944,945,948,949,950,951,952,953,954,955,956,958,959,960,961,962,963,964,965,966,967,969,971,972,973,974,975,977,978,981,983,985,986,987,988,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1005,1006,1008,1009,1011,1014,1015,1016,1017,1018,1020,1021,1022,1023,1025,1026,1027,1029,1030,1031,1032,1033,1034,1035,1036,1038,1039,1041,1042,1044,1045,1046,1047,1048,1049,1050,1051,1052,1053,1054,1055,1056,1057,1059,1060,1061,1066,1068,1069,1070,1071,1072,1073,1074,1075,1077,1081,1082,1083,1084,1085,1086,1087,1088,1089,1090,1091,1092,1094,1095,1096,1099,1100,1102,1103,1104,1106,1107,1108,1109,1110,1111,1112,1113,1114,1115,1118,1120,1121,1123,1124,1125,1126,1127,1128,1129,1130,1132,1133,1135,1136,1137,1138,1139,1141,1142,1143,1144,1145,1146,1147,1148,1149,1150,1152,1153,1154,1155,1157,1158,1159,1160,1161,1162,1163,1164,1165,1166,1167,1169,1170,1171,1172,1173,1174,1175,1176,1177,1178,1179,1180,1183,1184,1185,1186,1187,1188,1189,1190,1191,1192,1195,1197,1198,1199,1200,1201,1202,1203,1204,1205,1206,1208,1209,1211,1212,1213,1215,1216,1220,1221,1222,1224,1226,1227,1228,1229,1230,1231,1232,1233,1234,1235,1236,1237,1238,1240,1241,1242,1243,1244,1246,1247,1248,1249,1251,1253,1254,1256,1257,1258,1259,1260,1261,1262,1263,1264,1265,1267,1268,1271,1272,1273,1274,1276,1277,1279,1280,1284,1285,1287,1289,1290,1291,1292,1293,1294,1295,1296,1298,1301,1302,1303,1304,1305,1306,1307,1308,1309,1310,1311,1312,1314,1316,1317,1318,1319,1320,1322,1324,1327,1329,1330,1332,1334,1336,1337,1338,1339,1340,1341,1342,1343,1344,1345,1346,1347,1348,1349,1350,1351,1353,1354,1355,1356,1358,1359,1360,1361,1362,1363,1364,1365,1366,1367,1368,1369,1370,1371,1372,1375,1376,1377,1378,1379,1381,1382,1383,1384,1385,1386,1387,1388,1389,1393,1394,1396,1397,1398,1401,1402,1403,1404,1405,1406,1407,1408,1409,1410,1411,1412,1414,1415,1417,1418,1420,1421,1424,1425,1428,1429,1430,1431,1432,1433,1434,1435,1436,1437,1438,1439,1440,1441,1442,1443,1444,1445,1446,1448,1449,1450,1451,1452,1453,1454,1455,1456,1457,1458,1459,1460,1461,1462,1463,1464,1465,1467,1468,1469,1472,1473,1475,1476,1477,1479,1480,1481,1483,1484,1485,1488,1489,1490,1491,1492,1493,1494,1495,1496,1497,1498,1499,1500,1502,1503,1504,1505,1506,1507,1508,1509,1510,1511,1512,1513,1514,1515,1516,1517,1518,1519,1520,1521,1522,1523,1524,1525,1527,1528,1529,1530,1531,1532,1534,1535,1536,1537,1538,1539,1540,1542,1543,1544,1545,1546,1547,1549,1550,1551,1552,1553,1554,1556,1557,1560,1562,1563,1564,1566,1567,1569,1570,1571,1572,1573,1575,1577,1578,1579,1580,1581,1583,1584,1585,1586,1587,1589,1590,1592,1593,1594,1595,1596,1598,1599,1600,1601,1602,1604,1605,1606,1607,1608,1611,1613,1616,1617,1618,1619,1620,1621,1624,1625,1626,1627,1628,1629,1630,1632,1634,1635,1637,1638,1639,1641,1643,1644,1645,1646,1647,1648,1649,1650,1651,1652,1653,1654,1655,1656,1657,1658,1659,1660,1663,1664,1665,1666,1667,1668,1670,1671,1672,1673,1674,1675,1676,1677,1678,1680,1681,1682,1683,1684,1685,1686,1687,1688,1689,1690,1691,1692,1694,1695,1696,1697,1698,1699,1701,1702,1705,1706,1707,1708,1709,1710,1711,1713,1714,1716,1717,1718,1719,1720,1721,1722,1723,1724,1725,1726,1727,1728,1729,1732,1733,1734,1735,1736,1738,1739,1740,1741,1743,1744,1745,1746,1747,1748,1749,1751,1752,1753,1754,1755,1756,1757,1759,1760,1761,1762,1763,1764,1765,1766,1767,1768,1769,1771,1772,1773,1774,1775,1776,1777,1778,1780,1781,1783,1784,1785,1786,1787,1788,1789,1791,1792,1793,1794,1795,1796,1797,1801,1802,1803,1804,1805,1806,1809,1812,1813,1814,1815,1816,1817,1818,1819,1820,1822,1823,1824,1825,1826,1827,1829,1830,1834,1835,1836,1837,1839,1840,1841,1842,1843,1844,1845,1846,1848,1850,1851,1853,1854,1855,1856,1857,1858,1859,1860,1862,1865,1867,1868,1869,1870,1873,1874,1875,1877,1878,1880,1881,1882,1883,1885,1886,1888,1889,1891,1892,1893,1894,1895,1896,1897,1898,1899,1900,1901,1902,1903,1904,1905,1906,1907,1909,1910,1911,1912,1913,1915,1916,1917,1918,1919,1920,1921,1922,1923,1924,1926,1927,1928,1929,1930,1931,1932,1933,1934,1936,1937,1938,1939,1942,1944,1945,1946,1947,1948,1950,1952,1953,1955,1957,1960,1961,1962,1964,1965,1967,1968,1969,1971,1972,1973,1974,1975,1977,1978,1980,1982,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1997,1998,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2019,2021,2022,2023,2025,2029,2030,2031,2032,2033,2034,2036,2037,2038,2039,2040,2041,2043,2044,2046,2047,2048,2052,2055,2056,2057,2058,2059,2060,2061,2063,2064,2065,2066,2067,2068,2069,2070,2071,2072,2073,2074,2076,2077,2079,2080,2081,2082,2083,2084,2085,2086,2087,2088,2089,2092,2093,2095,2096,2097,2098,2099,2100,2101,2102,2103,2105,2106,2107,2108,2109,2110,2111,2112,2113,2114,2115,2116,2117,2119,2120,2123,2124,2125,2126,2127,2128,2129,2131,2132,2133,2134,2135,2136,2137,2138,2139,2140,2141,2142,2143,2144,2147,2148,2149,2150,2151,2152,2153,2154,2155,2156,2157,2158,2159,2160,2161,2162,2163,2164,2165,2166,2169,2170,2171,2173,2174,2175,2178,2179,2180,2183,2184,2185,2189,2190,2191,2192,2193,2194,2195,2197,2198,2199,2201,2202,2203,2205,2206,2207,2208,2209,2210,2211,2212,2213,2214,2215,2216,2218,2219,2220,2221,2222,2223,2224,2225,2226,2227,2228,2229,2230,2231,2232,2233,2234,2235,2236,2237,2238,2239,2241,2242,2243,2244,2245,2247,2248,2249,2250,2251,2253,2254,2256,2257,2259,2260,2262,2263,2264,2265,2267,2268,2269,2270,2272,2274,2276,2278,2279,2280,2281,2282,2283,2284,2285,2286,2287,2288,2289,2290,2291,2292,2294,2295,2297,2299,2301,2303,2304,2305,2306,2307,2308,2309,2310,2311,2312,2313,2314,2315,2316,2318,2319,2320,2322,2324,2325,2326,2327,2328,2329,2330,2331,2333,2335,2336,2339,2341,2342,2343,2344,2345,2348,2349,2350,2351,2353,2354,2355,2356,2357,2358,2359,2361,2362,2364,2365,2366,2367,2368,2369,2370,2371,2373,2374,2375,2376,2377,2378,2379,2381,2382,2384,2385,2386,2387,2388,2389,2391,2392,2393,2394,2395,2396,2397,2398,2400,2401,2402,2403,2404,2405,2406,2407,2408,2409,2410,2411,2413,2414,2415,2416,2417,2418,2419,2420,2421,2422,2423,2424,2426,2427,2428,2429,2430,2431,2432,2433,2434,2435,2438,2439,2440,2441,2442,2443,2445,2446,2447,2448,2450,2451,2452,2453,2455,2456,2457,2458,2459,2460,2461,2462,2464,2465,2466,2467,2469,2470,2471,2472,2473,2474,2475,2476,2478,2479,2480,2481,2483,2484,2485,2487,2488,2489,2490,2491,2492,2493,2494,2495,2496,2497,2498,2499,2500,2501,2503,2504,2505,2506,2507,2508,2509,2510,2512,2513,2514,2515,2516,2517,2519,2520,2521,2522,2523,2524,2525,2526,2527,2528,2530,2531,2532,2533,2534,2536,2537,2539,2540,2543,2544,2545,2546,2547,2548,2549,2550,2551,2552,2553,2554,2556,2557,2558,2561,2562,2564,2565,2569,2570,2571,2572,2573,2576,2577,2578,2580,2581,2582,2583,2585,2586,2587,2589,2590,2592,2593,2596,2598,2599,2600,2602,2603,2604,2606,2607,2608,2609,2610,2611,2612,2613,2615,2616,2617,2618,2619,2620,2621,2622,2623,2624,2625,2626,2629,2632,2635,2637,2638,2641,2642,2643,2644,2645,2646,2647,2649,2650,2651,2652,2654,2655,2656,2657,2658,2661,2662,2663,2664,2665,2667,2668,2669,2670,2671,2672,2675,2677,2678,2680,2681,2682,2683,2684,2686,2687,2688,2690,2692,2693,2695,2696,2697,2698],\"valid\":[0,15,24,25,28,29,30,31,33,37,46,47,53,58,60,61,75,78,82,86,90,91,93,94,102,114,116,120,121,126,130,135,139,143,155,161,169,181,184,188,193,201,202,211,212,245,255,256,261,262,264,271,278,285,292,295,307,310,312,316,331,332,334,338,349,356,359,365,366,378,380,381,382,384,394,395,407,416,424,426,427,429,431,432,438,450,454,462,468,473,476,479,497,516,518,523,526,529,531,538,550,553,557,560,562,565,568,571,578,579,581,587,592,601,604,608,613,618,620,631,632,633,641,647,652,653,658,660,665,667,669,678,681,682,683,686,689,692,696,698,706,708,712,713,715,723,724,725,739,740,741,754,757,771,778,783,789,803,805,812,824,825,831,832,841,851,853,860,861,863,867,869,872,874,879,884,895,905,907,912,914,924,925,926,927,934,938,943,946,947,957,968,970,976,979,980,982,984,989,1004,1007,1010,1012,1013,1019,1024,1028,1037,1040,1043,1058,1062,1063,1064,1065,1067,1076,1078,1079,1080,1093,1097,1098,1101,1105,1116,1117,1119,1122,1131,1134,1140,1151,1156,1168,1181,1182,1193,1194,1196,1207,1210,1214,1217,1218,1219,1223,1225,1239,1245,1250,1252,1255,1266,1269,1270,1275,1278,1281,1282,1283,1286,1288,1297,1299,1300,1313,1315,1321,1323,1325,1326,1328,1331,1333,1335,1352,1357,1373,1374,1380,1390,1391,1392,1395,1399,1400,1413,1416,1419,1422,1423,1426,1427,1447,1466,1470,1471,1474,1478,1482,1486,1487,1501,1526,1533,1541,1548,1555,1558,1559,1561,1565,1568,1574,1576,1582,1588,1591,1597,1603,1609,1610,1612,1614,1615,1622,1623,1631,1633,1636,1640,1642,1661,1662,1669,1679,1693,1700,1703,1704,1712,1715,1730,1731,1737,1742,1750,1758,1770,1779,1782,1790,1798,1799,1800,1807,1808,1810,1811,1821,1828,1831,1832,1833,1838,1847,1849,1852,1861,1863,1864,1866,1871,1872,1876,1879,1884,1887,1890,1908,1914,1925,1935,1940,1941,1943,1949,1951,1954,1956,1958,1959,1963,1966,1970,1976,1979,1981,1983,1984,1996,1999,2017,2018,2020,2024,2026,2027,2028,2035,2042,2045,2049,2050,2051,2053,2054,2062,2075,2078,2090,2091,2094,2104,2118,2121,2122,2130,2145,2146,2167,2168,2172,2176,2177,2181,2182,2186,2187,2188,2196,2200,2204,2217,2240,2246,2252,2255,2258,2261,2266,2271,2273,2275,2277,2293,2296,2298,2300,2302,2317,2321,2323,2332,2334,2337,2338,2340,2346,2347,2352,2360,2363,2372,2380,2383,2390,2399,2412,2425,2436,2437,2444,2449,2454,2463,2468,2477,2482,2486,2502,2511,2518,2529,2535,2538,2541,2542,2555,2559,2560,2563,2566,2567,2568,2574,2575,2579,2584,2588,2591,2594,2595,2597,2601,2605,2614,2627,2628,2630,2631,2633,2634,2636,2639,2640,2648,2653,2659,2660,2666,2673,2674,2676,2679,2685,2689,2691,2694]},\"outer2_inner0\":{\"fit\":[1,2,3,4,5,6,7,8,9,11,12,14,19,27,32,39,41,42,43,45,48,50,51,54,55,56,57,62,63,65,66,67,68,69,71,72,73,76,77,79,80,81,87,88,92,96,97,99,100,101,103,107,109,113,115,122,123,125,128,132,134,136,140,145,146,148,149,150,151,152,154,157,158,163,164,165,166,172,173,174,175,176,177,179,182,183,185,186,187,189,191,192,196,199,200,203,206,207,209,210,213,215,216,217,218,219,222,223,230,232,233,234,235,236,241,244,247,248,253,254,257,258,267,268,269,270,272,273,274,275,276,277,279,280,281,282,283,284,286,287,289,291,293,297,299,301,302,303,304,305,306,308,311,314,315,317,318,319,321,322,324,325,326,327,328,333,335,336,339,341,343,345,346,347,353,355,357,360,361,362,363,364,367,368,371,372,373,375,376,377,379,383,386,387,389,390,391,393,396,399,400,401,402,403,404,405,408,409,411,413,414,415,417,418,419,421,423,425,430,433,434,435,436,437,440,444,445,446,447,448,451,453,455,457,458,461,466,469,471,474,475,477,478,481,483,484,485,486,488,489,490,492,493,496,499,500,502,503,506,507,509,510,511,512,513,514,515,520,521,522,524,527,530,532,536,537,539,540,541,542,543,544,545,547,548,549,551,552,554,555,556,558,561,563,566,567,569,573,574,576,577,582,583,588,589,590,593,594,595,596,598,599,603,605,606,607,610,611,614,615,617,621,625,626,628,629,630,635,636,637,638,640,642,645,648,649,650,651,654,655,656,663,664,666,668,671,672,673,674,675,676,677,680,684,685,688,690,693,694,697,699,703,705,707,710,711,714,716,717,718,719,720,721,722,727,728,730,731,732,735,737,738,743,744,747,750,756,759,760,761,762,763,765,766,767,768,769,770,772,774,775,776,777,779,780,781,782,785,786,787,788,790,791,794,795,796,797,798,800,806,808,809,810,815,817,818,820,821,822,828,829,833,834,835,836,837,838,839,840,843,846,847,848,849,852,855,857,859,865,866,868,870,871,877,878,880,881,883,885,886,887,891,892,893,894,898,899,900,902,903,906,910,911,913,916,917,919,920,922,923,928,930,932,935,936,937,939,940,942,944,945,949,951,953,954,955,956,958,960,961,963,966,967,969,971,972,973,975,981,983,985,986,987,988,990,991,992,993,994,996,997,998,1000,1001,1006,1008,1009,1015,1016,1020,1023,1025,1026,1029,1030,1032,1033,1034,1035,1036,1038,1039,1041,1042,1044,1045,1046,1047,1052,1053,1054,1055,1056,1057,1059,1060,1061,1066,1068,1069,1070,1071,1072,1073,1074,1075,1077,1081,1083,1086,1087,1088,1089,1090,1091,1095,1099,1100,1104,1107,1108,1109,1110,1113,1115,1118,1120,1121,1124,1125,1126,1127,1129,1132,1133,1137,1138,1139,1142,1143,1144,1146,1147,1148,1149,1150,1152,1153,1154,1155,1158,1161,1163,1165,1166,1167,1170,1171,1172,1177,1179,1180,1183,1184,1185,1186,1187,1188,1189,1191,1197,1199,1200,1202,1203,1208,1211,1212,1213,1216,1221,1222,1224,1226,1227,1228,1229,1232,1234,1235,1238,1240,1241,1242,1247,1249,1251,1254,1256,1257,1258,1260,1263,1265,1267,1268,1271,1272,1273,1274,1276,1279,1280,1285,1290,1291,1292,1294,1295,1298,1301,1303,1304,1308,1309,1310,1312,1314,1316,1317,1318,1319,1327,1329,1330,1334,1336,1338,1339,1341,1342,1343,1344,1347,1349,1350,1351,1353,1354,1358,1360,1366,1367,1369,1370,1371,1372,1375,1377,1378,1379,1381,1382,1383,1384,1385,1386,1387,1388,1396,1397,1398,1401,1406,1407,1409,1412,1414,1417,1418,1420,1421,1428,1429,1431,1434,1436,1437,1438,1439,1440,1441,1443,1445,1449,1450,1451,1454,1456,1457,1458,1460,1461,1462,1463,1464,1465,1467,1468,1472,1475,1476,1479,1483,1484,1488,1489,1490,1491,1492,1493,1496,1499,1500,1502,1503,1504,1505,1506,1507,1509,1510,1511,1513,1514,1515,1517,1518,1519,1523,1524,1525,1527,1530,1531,1532,1534,1535,1536,1537,1538,1540,1542,1543,1546,1547,1549,1550,1551,1552,1553,1554,1556,1560,1563,1564,1567,1569,1570,1572,1575,1577,1578,1579,1580,1581,1583,1584,1586,1590,1592,1595,1596,1598,1602,1605,1606,1607,1611,1613,1616,1617,1620,1624,1625,1629,1630,1632,1634,1637,1638,1639,1643,1644,1645,1646,1649,1650,1651,1652,1653,1654,1657,1658,1659,1660,1663,1664,1665,1666,1667,1668,1672,1673,1675,1676,1677,1678,1680,1681,1682,1683,1684,1686,1688,1689,1690,1691,1694,1695,1697,1698,1701,1702,1705,1707,1708,1709,1713,1716,1717,1718,1719,1721,1723,1724,1725,1726,1727,1729,1732,1733,1735,1739,1740,1741,1743,1744,1745,1746,1747,1748,1751,1752,1753,1754,1755,1760,1762,1763,1765,1767,1769,1771,1773,1777,1778,1781,1783,1784,1785,1788,1789,1791,1792,1793,1794,1795,1796,1797,1801,1802,1803,1806,1809,1813,1814,1815,1816,1819,1822,1823,1824,1826,1827,1829,1830,1836,1839,1843,1844,1845,1846,1848,1850,1851,1853,1854,1855,1856,1859,1860,1862,1865,1867,1870,1873,1874,1875,1877,1878,1881,1882,1883,1886,1888,1893,1894,1895,1896,1897,1898,1899,1900,1901,1902,1903,1905,1906,1909,1912,1916,1917,1918,1920,1921,1922,1923,1928,1929,1931,1933,1936,1937,1944,1945,1946,1947,1950,1953,1955,1957,1961,1962,1965,1967,1968,1969,1971,1973,1974,1977,1978,1980,1985,1987,1988,1989,1990,1991,1993,1994,1995,1997,2001,2003,2008,2009,2010,2011,2016,2019,2022,2023,2025,2029,2033,2034,2036,2038,2039,2040,2043,2048,2055,2057,2060,2065,2066,2067,2068,2069,2076,2081,2083,2085,2086,2089,2095,2097,2098,2099,2102,2103,2105,2106,2108,2109,2110,2113,2114,2115,2119,2120,2123,2124,2125,2127,2128,2129,2133,2134,2135,2137,2138,2139,2141,2142,2143,2144,2147,2148,2149,2150,2152,2153,2155,2156,2157,2159,2161,2162,2163,2164,2165,2169,2170,2174,2175,2178,2179,2180,2191,2192,2193,2194,2197,2199,2201,2203,2205,2206,2207,2208,2209,2210,2211,2212,2214,2216,2219,2220,2221,2222,2224,2225,2227,2228,2232,2233,2234,2237,2241,2242,2243,2245,2248,2250,2253,2254,2259,2260,2262,2265,2267,2268,2272,2274,2276,2279,2283,2285,2286,2287,2289,2290,2291,2294,2295,2297,2299,2301,2305,2306,2307,2308,2311,2314,2315,2318,2320,2322,2324,2325,2326,2327,2329,2330,2331,2333,2335,2336,2339,2341,2344,2345,2348,2353,2354,2355,2357,2358,2361,2362,2365,2366,2368,2369,2370,2374,2376,2378,2381,2384,2386,2388,2389,2392,2393,2394,2395,2396,2397,2398,2400,2401,2402,2403,2406,2407,2408,2409,2411,2413,2414,2415,2416,2417,2418,2419,2420,2421,2422,2427,2428,2430,2433,2435,2438,2441,2442,2443,2445,2446,2450,2451,2452,2453,2456,2457,2458,2459,2461,2462,2465,2466,2470,2472,2474,2478,2479,2480,2483,2484,2485,2487,2489,2492,2493,2494,2495,2496,2497,2498,2499,2501,2503,2504,2505,2506,2508,2510,2512,2513,2514,2515,2516,2517,2519,2521,2522,2524,2525,2527,2528,2531,2532,2534,2536,2539,2540,2543,2544,2545,2547,2548,2551,2553,2554,2556,2557,2558,2561,2564,2569,2570,2571,2572,2573,2576,2577,2580,2581,2582,2583,2585,2586,2587,2589,2590,2592,2593,2596,2598,2599,2602,2606,2607,2608,2611,2613,2615,2616,2617,2618,2619,2620,2622,2624,2625,2626,2632,2635,2637,2641,2642,2643,2644,2646,2647,2651,2652,2654,2655,2656,2657,2658,2662,2664,2665,2667,2669,2670,2671,2672,2675,2678,2680,2681,2682,2683,2684,2687,2688,2690,2692,2695,2697,2698],\"valid\":[10,13,16,17,18,20,21,22,23,26,34,35,36,38,40,44,49,52,59,64,70,74,83,84,85,89,95,98,104,105,106,108,110,111,112,117,118,119,124,127,129,131,133,137,138,141,142,144,147,153,156,159,160,162,167,168,170,171,178,180,190,194,195,197,198,204,205,208,214,220,221,224,225,226,227,228,229,231,237,238,239,240,242,243,246,249,250,251,252,259,260,263,265,266,288,290,294,296,298,300,309,313,320,323,329,330,337,340,342,344,348,350,351,352,354,358,369,370,374,385,388,392,397,398,406,410,412,420,422,428,439,441,442,443,449,452,456,459,460,463,464,465,467,470,472,480,482,487,491,494,495,498,501,504,505,508,517,519,525,528,533,534,535,546,559,564,570,572,575,580,584,585,586,591,597,600,602,609,612,616,619,622,623,624,627,634,639,643,644,646,657,659,661,662,670,679,687,691,695,700,701,702,704,709,726,729,733,734,736,742,745,746,748,749,751,752,753,755,758,764,773,784,792,793,799,801,802,804,807,811,813,814,816,819,823,826,827,830,842,844,845,850,854,856,858,862,864,873,875,876,882,888,889,890,896,897,901,904,908,909,915,918,921,929,931,933,941,948,950,952,959,962,964,965,974,977,978,995,999,1002,1003,1005,1011,1014,1017,1018,1021,1022,1027,1031,1048,1049,1050,1051,1082,1084,1085,1092,1094,1096,1102,1103,1106,1111,1112,1114,1123,1128,1130,1135,1136,1141,1145,1157,1159,1160,1162,1164,1169,1173,1174,1175,1176,1178,1190,1192,1195,1198,1201,1204,1205,1206,1209,1215,1220,1230,1231,1233,1236,1237,1243,1244,1246,1248,1253,1259,1261,1262,1264,1277,1284,1287,1289,1293,1296,1302,1305,1306,1307,1311,1320,1322,1324,1332,1337,1340,1345,1346,1348,1355,1356,1359,1361,1362,1363,1364,1365,1368,1376,1389,1393,1394,1402,1403,1404,1405,1408,1410,1411,1415,1424,1425,1430,1432,1433,1435,1442,1444,1446,1448,1452,1453,1455,1459,1469,1473,1477,1480,1481,1485,1494,1495,1497,1498,1508,1512,1516,1520,1521,1522,1528,1529,1539,1544,1545,1557,1562,1566,1571,1573,1585,1587,1589,1593,1594,1599,1600,1601,1604,1608,1618,1619,1621,1626,1627,1628,1635,1641,1647,1648,1655,1656,1670,1671,1674,1685,1687,1692,1696,1699,1706,1710,1711,1714,1720,1722,1728,1734,1736,1738,1749,1756,1757,1759,1761,1764,1766,1768,1772,1774,1775,1776,1780,1786,1787,1804,1805,1812,1817,1818,1820,1825,1834,1835,1837,1840,1841,1842,1857,1858,1868,1869,1880,1885,1889,1891,1892,1904,1907,1910,1911,1913,1915,1919,1924,1926,1927,1930,1932,1934,1938,1939,1942,1948,1952,1960,1964,1972,1975,1982,1986,1992,1998,2000,2002,2004,2005,2006,2007,2012,2013,2014,2015,2021,2030,2031,2032,2037,2041,2044,2046,2047,2052,2056,2058,2059,2061,2063,2064,2070,2071,2072,2073,2074,2077,2079,2080,2082,2084,2087,2088,2092,2093,2096,2100,2101,2107,2111,2112,2116,2117,2126,2131,2132,2136,2140,2151,2154,2158,2160,2166,2171,2173,2183,2184,2185,2189,2190,2195,2198,2202,2213,2215,2218,2223,2226,2229,2230,2231,2235,2236,2238,2239,2244,2247,2249,2251,2256,2257,2263,2264,2269,2270,2278,2280,2281,2282,2284,2288,2292,2303,2304,2309,2310,2312,2313,2316,2319,2328,2342,2343,2349,2350,2351,2356,2359,2364,2367,2371,2373,2375,2377,2379,2382,2385,2387,2391,2404,2405,2410,2423,2424,2426,2429,2431,2432,2434,2439,2440,2447,2448,2455,2460,2464,2467,2469,2471,2473,2475,2476,2481,2488,2490,2491,2500,2507,2509,2520,2523,2526,2530,2533,2537,2546,2549,2550,2552,2562,2565,2578,2600,2603,2604,2609,2610,2612,2621,2623,2629,2638,2645,2649,2650,2661,2663,2668,2677,2686,2693,2696]},\"outer2_inner1\":{\"fit\":[1,2,3,5,9,10,12,13,14,16,17,18,20,21,22,23,26,32,34,35,36,38,39,40,41,44,48,49,50,51,52,55,59,62,64,65,68,69,70,72,74,83,84,85,87,88,89,95,96,97,98,99,101,103,104,105,106,108,109,110,111,112,113,115,117,118,119,124,125,127,128,129,131,133,137,138,140,141,142,144,146,147,148,149,150,151,153,154,156,159,160,162,163,164,166,167,168,170,171,172,174,175,176,177,178,180,182,183,186,187,189,190,192,194,195,197,198,199,204,205,206,208,209,213,214,216,219,220,221,222,223,224,225,226,227,228,229,231,233,236,237,238,239,240,241,242,243,246,249,250,251,252,254,259,260,263,265,266,267,270,273,275,276,277,279,283,286,288,290,291,294,296,297,298,299,300,301,302,303,304,309,311,313,320,321,322,323,326,327,329,330,333,335,336,337,340,341,342,344,345,346,348,350,351,352,353,354,358,360,363,364,367,368,369,370,372,373,374,379,385,387,388,389,392,393,396,397,398,399,400,402,404,405,406,409,410,412,413,415,418,420,422,428,430,433,435,439,440,441,442,443,444,445,446,449,451,452,456,458,459,460,461,463,464,465,467,469,470,472,475,480,481,482,483,484,487,488,489,490,491,492,493,494,495,496,498,499,500,501,504,505,506,508,509,510,511,512,514,517,519,524,525,527,528,532,533,534,535,536,539,541,544,546,548,555,558,559,564,567,569,570,572,573,574,575,576,580,583,584,585,586,590,591,593,597,599,600,602,605,606,607,609,610,611,612,616,617,619,622,623,624,625,627,628,634,637,639,640,642,643,644,646,648,651,654,657,659,661,662,663,664,668,670,675,679,680,684,685,687,688,690,691,695,697,699,700,701,702,703,704,705,707,709,710,711,716,719,720,722,726,727,729,732,733,734,735,736,742,743,745,746,747,748,749,751,752,753,755,756,758,759,760,764,767,768,769,773,774,775,777,782,784,787,791,792,793,794,796,797,799,801,802,804,806,807,808,811,813,814,815,816,817,818,819,822,823,826,827,830,834,837,840,842,844,845,846,849,850,852,854,856,857,858,862,864,865,873,875,876,882,885,886,887,888,889,890,893,896,897,899,900,901,902,903,904,908,909,910,911,913,915,918,919,920,921,923,928,929,930,931,933,935,936,941,945,948,950,952,953,958,959,960,961,962,964,965,966,967,969,971,974,975,977,978,985,986,988,995,998,999,1000,1001,1002,1003,1005,1009,1011,1014,1016,1017,1018,1021,1022,1026,1027,1031,1033,1038,1039,1042,1045,1047,1048,1049,1050,1051,1053,1054,1055,1056,1060,1061,1068,1070,1071,1073,1075,1082,1083,1084,1085,1088,1092,1094,1096,1099,1102,1103,1104,1106,1107,1108,1109,1111,1112,1114,1115,1120,1121,1123,1126,1128,1129,1130,1135,1136,1137,1139,1141,1142,1145,1146,1147,1148,1149,1150,1154,1157,1159,1160,1161,1162,1164,1169,1171,1173,1174,1175,1176,1177,1178,1186,1187,1190,1192,1195,1198,1199,1201,1202,1204,1205,1206,1208,1209,1212,1215,1220,1226,1227,1228,1230,1231,1232,1233,1235,1236,1237,1238,1240,1241,1242,1243,1244,1246,1247,1248,1253,1257,1258,1259,1261,1262,1264,1268,1273,1277,1284,1285,1287,1289,1291,1293,1295,1296,1298,1302,1303,1305,1306,1307,1309,1311,1312,1314,1316,1317,1319,1320,1322,1324,1332,1337,1340,1341,1342,1344,1345,1346,1347,1348,1349,1353,1354,1355,1356,1358,1359,1361,1362,1363,1364,1365,1368,1372,1375,1376,1377,1378,1379,1381,1383,1384,1388,1389,1393,1394,1397,1401,1402,1403,1404,1405,1407,1408,1410,1411,1412,1414,1415,1417,1421,1424,1425,1429,1430,1431,1432,1433,1434,1435,1436,1437,1438,1440,1441,1442,1443,1444,1446,1448,1449,1451,1452,1453,1454,1455,1456,1458,1459,1461,1463,1465,1467,1469,1472,1473,1475,1477,1480,1481,1485,1490,1491,1492,1494,1495,1497,1498,1499,1502,1507,1508,1509,1510,1512,1513,1514,1516,1517,1518,1519,1520,1521,1522,1523,1524,1525,1527,1528,1529,1530,1531,1537,1539,1540,1543,1544,1545,1546,1547,1551,1553,1557,1560,1562,1563,1566,1569,1570,1571,1572,1573,1580,1583,1585,1587,1589,1590,1592,1593,1594,1595,1596,1598,1599,1600,1601,1604,1605,1606,1607,1608,1613,1616,1618,1619,1620,1621,1626,1627,1628,1629,1630,1635,1641,1643,1645,1646,1647,1648,1650,1652,1655,1656,1657,1658,1660,1663,1665,1670,1671,1673,1674,1675,1678,1681,1682,1684,1685,1687,1692,1694,1695,1696,1698,1699,1702,1706,1709,1710,1711,1714,1720,1721,1722,1724,1725,1728,1729,1732,1734,1735,1736,1738,1741,1743,1748,1749,1751,1752,1753,1754,1755,1756,1757,1759,1761,1763,1764,1766,1768,1769,1771,1772,1774,1775,1776,1780,1785,1786,1787,1789,1791,1792,1794,1795,1801,1802,1803,1804,1805,1809,1812,1816,1817,1818,1819,1820,1822,1825,1827,1834,1835,1836,1837,1839,1840,1841,1842,1846,1848,1850,1851,1853,1857,1858,1860,1865,1867,1868,1869,1873,1874,1875,1878,1880,1883,1885,1889,1891,1892,1893,1894,1895,1896,1897,1899,1901,1902,1903,1904,1907,1910,1911,1912,1913,1915,1917,1918,1919,1923,1924,1926,1927,1929,1930,1931,1932,1933,1934,1938,1939,1942,1944,1947,1948,1950,1952,1955,1957,1960,1961,1964,1965,1967,1968,1972,1974,1975,1982,1986,1987,1988,1991,1992,1994,1997,1998,2000,2002,2004,2005,2006,2007,2009,2010,2011,2012,2013,2014,2015,2019,2021,2022,2030,2031,2032,2033,2034,2037,2040,2041,2043,2044,2046,2047,2052,2056,2057,2058,2059,2061,2063,2064,2067,2068,2069,2070,2071,2072,2073,2074,2077,2079,2080,2081,2082,2083,2084,2085,2087,2088,2089,2092,2093,2095,2096,2099,2100,2101,2102,2105,2106,2107,2108,2110,2111,2112,2113,2114,2116,2117,2123,2124,2126,2127,2131,2132,2133,2136,2139,2140,2141,2143,2144,2147,2151,2152,2153,2154,2155,2156,2158,2160,2161,2162,2163,2165,2166,2170,2171,2173,2174,2178,2179,2180,2183,2184,2185,2189,2190,2193,2195,2198,2199,2201,2202,2203,2205,2206,2207,2208,2210,2211,2213,2215,2218,2221,2223,2225,2226,2227,2229,2230,2231,2232,2233,2234,2235,2236,2237,2238,2239,2244,2247,2248,2249,2250,2251,2254,2256,2257,2262,2263,2264,2267,2268,2269,2270,2274,2278,2279,2280,2281,2282,2284,2285,2288,2290,2291,2292,2294,2297,2301,2303,2304,2305,2306,2308,2309,2310,2311,2312,2313,2314,2316,2318,2319,2322,2324,2327,2328,2330,2331,2333,2335,2339,2341,2342,2343,2348,2349,2350,2351,2356,2357,2359,2362,2364,2367,2368,2371,2373,2374,2375,2377,2378,2379,2381,2382,2384,2385,2387,2389,2391,2393,2395,2400,2403,2404,2405,2409,2410,2415,2416,2419,2421,2422,2423,2424,2426,2428,2429,2430,2431,2432,2433,2434,2435,2438,2439,2440,2441,2445,2447,2448,2450,2451,2455,2456,2457,2458,2459,2460,2461,2464,2465,2467,2469,2470,2471,2473,2474,2475,2476,2478,2481,2483,2484,2485,2488,2489,2490,2491,2492,2495,2498,2500,2501,2503,2504,2506,2507,2509,2510,2515,2516,2520,2523,2526,2527,2530,2532,2533,2536,2537,2544,2546,2547,2549,2550,2552,2553,2554,2556,2562,2565,2569,2572,2573,2577,2578,2580,2581,2585,2586,2587,2589,2590,2592,2599,2600,2603,2604,2607,2609,2610,2611,2612,2613,2618,2620,2621,2623,2624,2625,2626,2629,2632,2635,2637,2638,2645,2649,2650,2654,2655,2657,2661,2662,2663,2664,2665,2668,2669,2671,2675,2677,2681,2682,2684,2686,2688,2690,2692,2693,2695,2696,2697],\"valid\":[4,6,7,8,11,19,27,42,43,45,54,56,57,63,66,67,71,73,76,77,79,80,81,92,100,107,122,123,132,134,136,145,152,157,158,165,173,179,185,191,196,200,203,207,210,215,217,218,230,232,234,235,244,247,248,253,257,258,268,269,272,274,280,281,282,284,287,289,293,305,306,308,314,315,317,318,319,324,325,328,339,343,347,355,357,361,362,371,375,376,377,383,386,390,391,401,403,408,411,414,417,419,421,423,425,434,436,437,447,448,453,455,457,466,471,474,477,478,485,486,502,503,507,513,515,520,521,522,530,537,540,542,543,545,547,549,551,552,554,556,561,563,566,577,582,588,589,594,595,596,598,603,614,615,621,626,629,630,635,636,638,645,649,650,655,656,666,671,672,673,674,676,677,693,694,714,717,718,721,728,730,731,737,738,744,750,761,762,763,765,766,770,772,776,779,780,781,785,786,788,790,795,798,800,809,810,820,821,828,829,833,835,836,838,839,843,847,848,855,859,866,868,870,871,877,878,880,881,883,891,892,894,898,906,916,917,922,932,937,939,940,942,944,949,951,954,955,956,963,972,973,981,983,987,990,991,992,993,994,996,997,1006,1008,1015,1020,1023,1025,1029,1030,1032,1034,1035,1036,1041,1044,1046,1052,1057,1059,1066,1069,1072,1074,1077,1081,1086,1087,1089,1090,1091,1095,1100,1110,1113,1118,1124,1125,1127,1132,1133,1138,1143,1144,1152,1153,1155,1158,1163,1165,1166,1167,1170,1172,1179,1180,1183,1184,1185,1188,1189,1191,1197,1200,1203,1211,1213,1216,1221,1222,1224,1229,1234,1249,1251,1254,1256,1260,1263,1265,1267,1271,1272,1274,1276,1279,1280,1290,1292,1294,1301,1304,1308,1310,1318,1327,1329,1330,1334,1336,1338,1339,1343,1350,1351,1360,1366,1367,1369,1370,1371,1382,1385,1386,1387,1396,1398,1406,1409,1418,1420,1428,1439,1445,1450,1457,1460,1462,1464,1468,1476,1479,1483,1484,1488,1489,1493,1496,1500,1503,1504,1505,1506,1511,1515,1532,1534,1535,1536,1538,1542,1549,1550,1552,1554,1556,1564,1567,1575,1577,1578,1579,1581,1584,1586,1602,1611,1617,1624,1625,1632,1634,1637,1638,1639,1644,1649,1651,1653,1654,1659,1664,1666,1667,1668,1672,1676,1677,1680,1683,1686,1688,1689,1690,1691,1697,1701,1705,1707,1708,1713,1716,1717,1718,1719,1723,1726,1727,1733,1739,1740,1744,1745,1746,1747,1760,1762,1765,1767,1773,1777,1778,1781,1783,1784,1788,1793,1796,1797,1806,1813,1814,1815,1823,1824,1826,1829,1830,1843,1844,1845,1854,1855,1856,1859,1862,1870,1877,1881,1882,1886,1888,1898,1900,1905,1906,1909,1916,1920,1921,1922,1928,1936,1937,1945,1946,1953,1962,1969,1971,1973,1977,1978,1980,1985,1989,1990,1993,1995,2001,2003,2008,2016,2023,2025,2029,2036,2038,2039,2048,2055,2060,2065,2066,2076,2086,2097,2098,2103,2109,2115,2119,2120,2125,2128,2129,2134,2135,2137,2138,2142,2148,2149,2150,2157,2159,2164,2169,2175,2191,2192,2194,2197,2209,2212,2214,2216,2219,2220,2222,2224,2228,2241,2242,2243,2245,2253,2259,2260,2265,2272,2276,2283,2286,2287,2289,2295,2299,2307,2315,2320,2325,2326,2329,2336,2344,2345,2353,2354,2355,2358,2361,2365,2366,2369,2370,2376,2386,2388,2392,2394,2396,2397,2398,2401,2402,2406,2407,2408,2411,2413,2414,2417,2418,2420,2427,2442,2443,2446,2452,2453,2462,2466,2472,2479,2480,2487,2493,2494,2496,2497,2499,2505,2508,2512,2513,2514,2517,2519,2521,2522,2524,2525,2528,2531,2534,2539,2540,2543,2545,2548,2551,2557,2558,2561,2564,2570,2571,2576,2582,2583,2593,2596,2598,2602,2606,2608,2615,2616,2617,2619,2622,2641,2642,2643,2644,2646,2647,2651,2652,2656,2658,2667,2670,2672,2678,2680,2683,2687,2698]},\"outer2_inner2\":{\"fit\":[4,6,7,8,10,11,13,16,17,18,19,20,21,22,23,26,27,34,35,36,38,40,42,43,44,45,49,52,54,56,57,59,63,64,66,67,70,71,73,74,76,77,79,80,81,83,84,85,89,92,95,98,100,104,105,106,107,108,110,111,112,117,118,119,122,123,124,127,129,131,132,133,134,136,137,138,141,142,144,145,147,152,153,156,157,158,159,160,162,165,167,168,170,171,173,178,179,180,185,190,191,194,195,196,197,198,200,203,204,205,207,208,210,214,215,217,218,220,221,224,225,226,227,228,229,230,231,232,234,235,237,238,239,240,242,243,244,246,247,248,249,250,251,252,253,257,258,259,260,263,265,266,268,269,272,274,280,281,282,284,287,288,289,290,293,294,296,298,300,305,306,308,309,313,314,315,317,318,319,320,323,324,325,328,329,330,337,339,340,342,343,344,347,348,350,351,352,354,355,357,358,361,362,369,370,371,374,375,376,377,383,385,386,388,390,391,392,397,398,401,403,406,408,410,411,412,414,417,419,420,421,422,423,425,428,434,436,437,439,441,442,443,447,448,449,452,453,455,456,457,459,460,463,464,465,466,467,470,471,472,474,477,478,480,482,485,486,487,491,494,495,498,501,502,503,504,505,507,508,513,515,517,519,520,521,522,525,528,530,533,534,535,537,540,542,543,545,546,547,549,551,552,554,556,559,561,563,564,566,570,572,575,577,580,582,584,585,586,588,589,591,594,595,596,597,598,600,602,603,609,612,614,615,616,619,621,622,623,624,626,627,629,630,634,635,636,638,639,643,644,645,646,649,650,655,656,657,659,661,662,666,670,671,672,673,674,676,677,679,687,691,693,694,695,700,701,702,704,709,714,717,718,721,726,728,729,730,731,733,734,736,737,738,742,744,745,746,748,749,750,751,752,753,755,758,761,762,763,764,765,766,770,772,773,776,779,780,781,784,785,786,788,790,792,793,795,798,799,800,801,802,804,807,809,810,811,813,814,816,819,820,821,823,826,827,828,829,830,833,835,836,838,839,842,843,844,845,847,848,850,854,855,856,858,859,862,864,866,868,870,871,873,875,876,877,878,880,881,882,883,888,889,890,891,892,894,896,897,898,901,904,906,908,909,915,916,917,918,921,922,929,931,932,933,937,939,940,941,942,944,948,949,950,951,952,954,955,956,959,962,963,964,965,972,973,974,977,978,981,983,987,990,991,992,993,994,995,996,997,999,1002,1003,1005,1006,1008,1011,1014,1015,1017,1018,1020,1021,1022,1023,1025,1027,1029,1030,1031,1032,1034,1035,1036,1041,1044,1046,1048,1049,1050,1051,1052,1057,1059,1066,1069,1072,1074,1077,1081,1082,1084,1085,1086,1087,1089,1090,1091,1092,1094,1095,1096,1100,1102,1103,1106,1110,1111,1112,1113,1114,1118,1123,1124,1125,1127,1128,1130,1132,1133,1135,1136,1138,1141,1143,1144,1145,1152,1153,1155,1157,1158,1159,1160,1162,1163,1164,1165,1166,1167,1169,1170,1172,1173,1174,1175,1176,1178,1179,1180,1183,1184,1185,1188,1189,1190,1191,1192,1195,1197,1198,1200,1201,1203,1204,1205,1206,1209,1211,1213,1215,1216,1220,1221,1222,1224,1229,1230,1231,1233,1234,1236,1237,1243,1244,1246,1248,1249,1251,1253,1254,1256,1259,1260,1261,1262,1263,1264,1265,1267,1271,1272,1274,1276,1277,1279,1280,1284,1287,1289,1290,1292,1293,1294,1296,1301,1302,1304,1305,1306,1307,1308,1310,1311,1318,1320,1322,1324,1327,1329,1330,1332,1334,1336,1337,1338,1339,1340,1343,1345,1346,1348,1350,1351,1355,1356,1359,1360,1361,1362,1363,1364,1365,1366,1367,1368,1369,1370,1371,1376,1382,1385,1386,1387,1389,1393,1394,1396,1398,1402,1403,1404,1405,1406,1408,1409,1410,1411,1415,1418,1420,1424,1425,1428,1430,1432,1433,1435,1439,1442,1444,1445,1446,1448,1450,1452,1453,1455,1457,1459,1460,1462,1464,1468,1469,1473,1476,1477,1479,1480,1481,1483,1484,1485,1488,1489,1493,1494,1495,1496,1497,1498,1500,1503,1504,1505,1506,1508,1511,1512,1515,1516,1520,1521,1522,1528,1529,1532,1534,1535,1536,1538,1539,1542,1544,1545,1549,1550,1552,1554,1556,1557,1562,1564,1566,1567,1571,1573,1575,1577,1578,1579,1581,1584,1585,1586,1587,1589,1593,1594,1599,1600,1601,1602,1604,1608,1611,1617,1618,1619,1621,1624,1625,1626,1627,1628,1632,1634,1635,1637,1638,1639,1641,1644,1647,1648,1649,1651,1653,1654,1655,1656,1659,1664,1666,1667,1668,1670,1671,1672,1674,1676,1677,1680,1683,1685,1686,1687,1688,1689,1690,1691,1692,1696,1697,1699,1701,1705,1706,1707,1708,1710,1711,1713,1714,1716,1717,1718,1719,1720,1722,1723,1726,1727,1728,1733,1734,1736,1738,1739,1740,1744,1745,1746,1747,1749,1756,1757,1759,1760,1761,1762,1764,1765,1766,1767,1768,1772,1773,1774,1775,1776,1777,1778,1780,1781,1783,1784,1786,1787,1788,1793,1796,1797,1804,1805,1806,1812,1813,1814,1815,1817,1818,1820,1823,1824,1825,1826,1829,1830,1834,1835,1837,1840,1841,1842,1843,1844,1845,1854,1855,1856,1857,1858,1859,1862,1868,1869,1870,1877,1880,1881,1882,1885,1886,1888,1889,1891,1892,1898,1900,1904,1905,1906,1907,1909,1910,1911,1913,1915,1916,1919,1920,1921,1922,1924,1926,1927,1928,1930,1932,1934,1936,1937,1938,1939,1942,1945,1946,1948,1952,1953,1960,1962,1964,1969,1971,1972,1973,1975,1977,1978,1980,1982,1985,1986,1989,1990,1992,1993,1995,1998,2000,2001,2002,2003,2004,2005,2006,2007,2008,2012,2013,2014,2015,2016,2021,2023,2025,2029,2030,2031,2032,2036,2037,2038,2039,2041,2044,2046,2047,2048,2052,2055,2056,2058,2059,2060,2061,2063,2064,2065,2066,2070,2071,2072,2073,2074,2076,2077,2079,2080,2082,2084,2086,2087,2088,2092,2093,2096,2097,2098,2100,2101,2103,2107,2109,2111,2112,2115,2116,2117,2119,2120,2125,2126,2128,2129,2131,2132,2134,2135,2136,2137,2138,2140,2142,2148,2149,2150,2151,2154,2157,2158,2159,2160,2164,2166,2169,2171,2173,2175,2183,2184,2185,2189,2190,2191,2192,2194,2195,2197,2198,2202,2209,2212,2213,2214,2215,2216,2218,2219,2220,2222,2223,2224,2226,2228,2229,2230,2231,2235,2236,2238,2239,2241,2242,2243,2244,2245,2247,2249,2251,2253,2256,2257,2259,2260,2263,2264,2265,2269,2270,2272,2276,2278,2280,2281,2282,2283,2284,2286,2287,2288,2289,2292,2295,2299,2303,2304,2307,2309,2310,2312,2313,2315,2316,2319,2320,2325,2326,2328,2329,2336,2342,2343,2344,2345,2349,2350,2351,2353,2354,2355,2356,2358,2359,2361,2364,2365,2366,2367,2369,2370,2371,2373,2375,2376,2377,2379,2382,2385,2386,2387,2388,2391,2392,2394,2396,2397,2398,2401,2402,2404,2405,2406,2407,2408,2410,2411,2413,2414,2417,2418,2420,2423,2424,2426,2427,2429,2431,2432,2434,2439,2440,2442,2443,2446,2447,2448,2452,2453,2455,2460,2462,2464,2466,2467,2469,2471,2472,2473,2475,2476,2479,2480,2481,2487,2488,2490,2491,2493,2494,2496,2497,2499,2500,2505,2507,2508,2509,2512,2513,2514,2517,2519,2520,2521,2522,2523,2524,2525,2526,2528,2530,2531,2533,2534,2537,2539,2540,2543,2545,2546,2548,2549,2550,2551,2552,2557,2558,2561,2562,2564,2565,2570,2571,2576,2578,2582,2583,2593,2596,2598,2600,2602,2603,2604,2606,2608,2609,2610,2612,2615,2616,2617,2619,2621,2622,2623,2629,2638,2641,2642,2643,2644,2645,2646,2647,2649,2650,2651,2652,2656,2658,2661,2663,2667,2668,2670,2672,2677,2678,2680,2683,2686,2687,2693,2696,2698],\"valid\":[1,2,3,5,9,12,14,32,39,41,48,50,51,55,62,65,68,69,72,87,88,96,97,99,101,103,109,113,115,125,128,140,146,148,149,150,151,154,163,164,166,172,174,175,176,177,182,183,186,187,189,192,199,206,209,213,216,219,222,223,233,236,241,254,267,270,273,275,276,277,279,283,286,291,297,299,301,302,303,304,311,321,322,326,327,333,335,336,341,345,346,353,360,363,364,367,368,372,373,379,387,389,393,396,399,400,402,404,405,409,413,415,418,430,433,435,440,444,445,446,451,458,461,469,475,481,483,484,488,489,490,492,493,496,499,500,506,509,510,511,512,514,524,527,532,536,539,541,544,548,555,558,567,569,573,574,576,583,590,593,599,605,606,607,610,611,617,625,628,637,640,642,648,651,654,663,664,668,675,680,684,685,688,690,697,699,703,705,707,710,711,716,719,720,722,727,732,735,743,747,756,759,760,767,768,769,774,775,777,782,787,791,794,796,797,806,808,815,817,818,822,834,837,840,846,849,852,857,865,885,886,887,893,899,900,902,903,910,911,913,919,920,923,928,930,935,936,945,953,958,960,961,966,967,969,971,975,985,986,988,998,1000,1001,1009,1016,1026,1033,1038,1039,1042,1045,1047,1053,1054,1055,1056,1060,1061,1068,1070,1071,1073,1075,1083,1088,1099,1104,1107,1108,1109,1115,1120,1121,1126,1129,1137,1139,1142,1146,1147,1148,1149,1150,1154,1161,1171,1177,1186,1187,1199,1202,1208,1212,1226,1227,1228,1232,1235,1238,1240,1241,1242,1247,1257,1258,1268,1273,1285,1291,1295,1298,1303,1309,1312,1314,1316,1317,1319,1341,1342,1344,1347,1349,1353,1354,1358,1372,1375,1377,1378,1379,1381,1383,1384,1388,1397,1401,1407,1412,1414,1417,1421,1429,1431,1434,1436,1437,1438,1440,1441,1443,1449,1451,1454,1456,1458,1461,1463,1465,1467,1472,1475,1490,1491,1492,1499,1502,1507,1509,1510,1513,1514,1517,1518,1519,1523,1524,1525,1527,1530,1531,1537,1540,1543,1546,1547,1551,1553,1560,1563,1569,1570,1572,1580,1583,1590,1592,1595,1596,1598,1605,1606,1607,1613,1616,1620,1629,1630,1643,1645,1646,1650,1652,1657,1658,1660,1663,1665,1673,1675,1678,1681,1682,1684,1694,1695,1698,1702,1709,1721,1724,1725,1729,1732,1735,1741,1743,1748,1751,1752,1753,1754,1755,1763,1769,1771,1785,1789,1791,1792,1794,1795,1801,1802,1803,1809,1816,1819,1822,1827,1836,1839,1846,1848,1850,1851,1853,1860,1865,1867,1873,1874,1875,1878,1883,1893,1894,1895,1896,1897,1899,1901,1902,1903,1912,1917,1918,1923,1929,1931,1933,1944,1947,1950,1955,1957,1961,1965,1967,1968,1974,1987,1988,1991,1994,1997,2009,2010,2011,2019,2022,2033,2034,2040,2043,2057,2067,2068,2069,2081,2083,2085,2089,2095,2099,2102,2105,2106,2108,2110,2113,2114,2123,2124,2127,2133,2139,2141,2143,2144,2147,2152,2153,2155,2156,2161,2162,2163,2165,2170,2174,2178,2179,2180,2193,2199,2201,2203,2205,2206,2207,2208,2210,2211,2221,2225,2227,2232,2233,2234,2237,2248,2250,2254,2262,2267,2268,2274,2279,2285,2290,2291,2294,2297,2301,2305,2306,2308,2311,2314,2318,2322,2324,2327,2330,2331,2333,2335,2339,2341,2348,2357,2362,2368,2374,2378,2381,2384,2389,2393,2395,2400,2403,2409,2415,2416,2419,2421,2422,2428,2430,2433,2435,2438,2441,2445,2450,2451,2456,2457,2458,2459,2461,2465,2470,2474,2478,2483,2484,2485,2489,2492,2495,2498,2501,2503,2504,2506,2510,2515,2516,2527,2532,2536,2544,2547,2553,2554,2556,2569,2572,2573,2577,2580,2581,2585,2586,2587,2589,2590,2592,2599,2607,2611,2613,2618,2620,2624,2625,2626,2632,2635,2637,2654,2655,2657,2662,2664,2665,2669,2671,2675,2681,2682,2684,2688,2690,2692,2695,2697]},\"outer3\":{\"fit\":[0,6,7,8,10,11,12,13,14,15,16,17,18,19,20,21,23,24,25,26,28,29,30,31,32,33,35,36,37,38,41,42,43,45,46,47,49,50,51,52,53,54,55,57,58,59,60,61,62,64,65,66,67,69,73,75,77,78,81,82,83,84,86,87,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,105,106,107,108,109,110,111,112,114,115,116,117,118,119,120,121,122,123,124,125,126,129,130,131,133,134,135,136,137,138,139,140,141,142,143,146,147,148,149,150,151,152,153,154,155,156,157,158,160,161,162,163,164,166,167,168,169,170,172,174,176,177,178,179,180,181,182,183,184,185,186,187,188,189,192,193,194,195,196,197,198,199,201,202,205,207,209,211,212,213,214,215,216,219,220,221,222,223,224,225,227,228,229,230,231,232,233,234,235,236,237,239,241,242,243,245,247,248,249,250,252,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,274,275,276,277,278,279,280,281,282,283,285,286,287,288,290,291,292,293,294,295,296,297,298,299,300,301,302,303,305,307,308,309,310,311,312,313,314,315,316,317,319,320,324,325,326,327,330,331,332,333,334,338,339,340,341,342,343,344,345,346,348,349,350,351,352,353,354,355,356,358,359,360,361,362,363,364,365,366,367,368,370,372,373,375,376,377,378,379,380,381,382,383,384,385,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,403,404,405,406,407,409,410,411,414,415,416,417,419,420,421,422,424,426,427,428,429,430,431,432,433,435,436,438,439,440,441,442,443,444,445,446,448,449,450,452,453,454,456,458,460,462,463,464,468,469,470,472,473,474,475,476,478,479,480,481,482,483,484,485,486,488,489,492,493,494,495,497,498,499,500,501,506,507,508,511,512,514,515,516,518,521,522,523,524,526,527,529,530,531,533,534,535,537,538,539,541,542,543,544,546,548,549,550,551,552,553,554,557,558,559,560,561,562,563,564,565,566,568,569,570,571,572,573,574,575,576,578,579,580,581,582,583,584,586,587,589,590,591,592,594,596,597,598,599,600,601,602,603,604,605,606,607,608,609,611,613,614,616,618,619,620,622,623,625,626,627,628,631,632,633,638,639,640,641,642,643,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,665,666,667,669,670,672,673,674,676,677,678,679,680,681,682,683,684,686,687,689,690,692,693,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,715,716,719,720,721,722,723,724,725,726,727,728,729,731,732,733,734,735,736,738,739,740,741,742,743,744,745,746,748,750,751,752,754,755,756,757,758,759,760,763,764,765,767,768,769,770,771,772,773,774,775,776,778,779,780,782,783,785,786,788,789,790,792,793,795,796,797,798,799,800,801,803,804,805,806,807,808,810,811,812,813,814,815,817,819,820,821,822,823,824,825,826,827,829,830,831,832,833,834,835,836,837,839,840,841,842,843,844,845,846,848,849,850,851,853,854,855,857,858,860,861,862,863,864,865,867,868,869,870,871,872,873,874,875,877,879,880,883,884,885,886,887,888,889,890,892,893,894,895,896,897,898,899,900,901,902,904,905,906,907,908,909,910,911,912,913,914,915,916,917,919,920,921,923,924,925,926,927,928,929,930,931,932,934,935,937,938,939,940,941,942,943,944,945,946,947,948,949,951,952,953,954,955,957,959,960,961,962,964,965,966,968,969,970,971,972,973,974,975,976,977,978,979,980,981,982,983,984,987,989,990,992,993,994,995,996,997,1000,1001,1003,1004,1005,1006,1007,1009,1010,1012,1013,1015,1016,1019,1020,1022,1023,1024,1025,1027,1028,1029,1031,1032,1033,1034,1035,1036,1037,1038,1039,1040,1041,1042,1043,1044,1045,1046,1047,1048,1049,1050,1051,1052,1053,1054,1055,1056,1057,1058,1059,1060,1061,1062,1063,1064,1065,1066,1067,1068,1069,1071,1072,1073,1075,1076,1078,1079,1080,1081,1083,1085,1086,1087,1088,1089,1090,1091,1092,1093,1094,1096,1097,1098,1099,1100,1101,1102,1103,1104,1105,1106,1107,1108,1109,1111,1113,1114,1115,1116,1117,1118,1119,1120,1122,1123,1124,1125,1126,1127,1130,1131,1132,1134,1135,1136,1137,1139,1140,1142,1143,1144,1145,1146,1148,1149,1150,1151,1152,1153,1154,1155,1156,1157,1158,1159,1161,1162,1165,1167,1168,1170,1171,1173,1175,1176,1177,1178,1179,1180,1181,1182,1183,1185,1186,1187,1190,1191,1192,1193,1194,1195,1196,1198,1199,1200,1201,1202,1204,1205,1207,1209,1210,1211,1212,1213,1214,1215,1217,1218,1219,1223,1224,1225,1226,1228,1229,1230,1231,1232,1233,1234,1235,1236,1237,1238,1239,1240,1242,1243,1244,1245,1247,1248,1249,1250,1251,1252,1253,1254,1255,1256,1257,1258,1259,1260,1261,1262,1264,1266,1269,1270,1271,1273,1274,1275,1276,1277,1278,1280,1281,1282,1283,1284,1286,1287,1288,1289,1291,1293,1294,1295,1296,1297,1299,1300,1302,1303,1305,1306,1307,1309,1310,1311,1312,1313,1314,1315,1316,1317,1318,1320,1321,1323,1324,1325,1326,1327,1328,1329,1331,1332,1333,1334,1335,1337,1338,1339,1341,1342,1344,1345,1347,1348,1349,1350,1351,1352,1353,1354,1356,1357,1358,1359,1360,1361,1364,1365,1366,1368,1369,1370,1371,1372,1373,1374,1376,1377,1378,1379,1380,1381,1382,1385,1388,1390,1391,1392,1394,1395,1397,1399,1400,1401,1403,1404,1405,1406,1407,1409,1410,1412,1413,1414,1415,1416,1417,1418,1419,1421,1422,1423,1424,1425,1426,1427,1428,1430,1433,1435,1437,1441,1442,1443,1444,1445,1446,1447,1449,1451,1452,1453,1454,1455,1457,1458,1459,1460,1461,1462,1463,1464,1465,1466,1467,1468,1469,1470,1471,1472,1473,1474,1475,1476,1477,1478,1479,1480,1481,1482,1483,1484,1485,1486,1487,1488,1489,1490,1491,1493,1494,1495,1496,1498,1499,1500,1501,1504,1505,1507,1508,1510,1511,1512,1514,1515,1516,1517,1518,1519,1520,1522,1523,1524,1525,1526,1527,1529,1530,1531,1532,1533,1534,1535,1536,1537,1539,1540,1541,1542,1543,1544,1545,1546,1547,1548,1549,1551,1552,1553,1555,1556,1557,1558,1559,1560,1561,1562,1564,1565,1566,1568,1569,1570,1571,1572,1573,1574,1576,1577,1578,1579,1580,1581,1582,1583,1584,1585,1586,1587,1588,1589,1591,1593,1594,1597,1598,1599,1600,1601,1602,1603,1606,1607,1608,1609,1610,1611,1612,1613,1614,1615,1617,1618,1619,1620,1622,1623,1624,1627,1628,1629,1630,1631,1633,1634,1635,1636,1637,1638,1639,1640,1641,1642,1643,1644,1645,1647,1650,1651,1652,1653,1655,1656,1657,1658,1659,1660,1661,1662,1664,1666,1668,1669,1670,1671,1672,1674,1675,1676,1677,1678,1679,1680,1681,1682,1683,1685,1686,1687,1688,1689,1690,1691,1693,1694,1695,1696,1697,1699,1700,1701,1702,1703,1704,1706,1708,1709,1710,1711,1712,1713,1714,1715,1716,1717,1718,1719,1720,1721,1722,1723,1725,1726,1727,1728,1729,1730,1731,1732,1734,1736,1737,1738,1739,1741,1742,1743,1745,1746,1747,1748,1749,1750,1751,1752,1753,1754,1755,1757,1758,1759,1760,1761,1763,1764,1765,1766,1767,1770,1771,1773,1774,1775,1776,1777,1779,1780,1782,1783,1784,1785,1786,1788,1790,1791,1793,1795,1797,1798,1799,1800,1801,1802,1806,1807,1808,1810,1811,1812,1813,1815,1816,1817,1821,1822,1823,1824,1825,1827,1828,1830,1831,1832,1833,1835,1837,1838,1839,1842,1843,1844,1847,1848,1849,1850,1852,1853,1857,1861,1863,1864,1865,1866,1868,1869,1870,1871,1872,1873,1874,1876,1879,1880,1881,1883,1884,1885,1886,1887,1888,1889,1890,1892,1895,1896,1897,1899,1902,1903,1904,1907,1908,1909,1910,1912,1914,1915,1916,1918,1919,1920,1922,1923,1924,1925,1926,1927,1928,1930,1931,1932,1934,1935,1936,1937,1938,1939,1940,1941,1943,1945,1946,1948,1949,1950,1951,1952,1954,1956,1957,1958,1959,1960,1961,1963,1964,1965,1966,1968,1969,1970,1971,1973,1974,1976,1978,1979,1981,1982,1983,1984,1985,1986,1987,1988,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2002,2003,2004,2006,2007,2008,2010,2011,2012,2013,2014,2015,2016,2017,2018,2020,2021,2022,2023,2024,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2041,2042,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057,2058,2059,2060,2061,2062,2063,2067,2069,2071,2072,2073,2075,2076,2077,2078,2079,2080,2081,2082,2083,2084,2085,2086,2087,2088,2089,2090,2091,2092,2093,2094,2095,2096,2099,2100,2101,2102,2103,2104,2105,2106,2107,2108,2110,2111,2112,2114,2115,2116,2117,2118,2119,2120,2121,2122,2123,2124,2125,2126,2127,2128,2129,2130,2133,2135,2136,2138,2140,2142,2143,2144,2145,2146,2148,2150,2152,2153,2154,2155,2156,2157,2158,2159,2160,2161,2164,2165,2166,2167,2168,2170,2171,2172,2173,2174,2175,2176,2177,2180,2181,2182,2183,2184,2185,2186,2187,2188,2189,2190,2191,2192,2193,2194,2195,2196,2197,2198,2200,2201,2202,2203,2204,2205,2207,2211,2212,2214,2215,2217,2218,2219,2220,2221,2222,2223,2225,2226,2227,2228,2231,2232,2233,2235,2236,2238,2239,2240,2241,2243,2245,2246,2247,2249,2250,2251,2252,2254,2255,2257,2258,2259,2261,2263,2265,2266,2267,2268,2269,2270,2271,2272,2273,2274,2275,2276,2277,2278,2279,2283,2284,2287,2288,2289,2293,2295,2296,2297,2298,2299,2300,2301,2302,2304,2305,2306,2308,2309,2310,2311,2312,2313,2314,2315,2316,2317,2318,2319,2320,2321,2323,2324,2325,2326,2327,2328,2330,2332,2333,2334,2336,2337,2338,2339,2340,2341,2342,2343,2344,2346,2347,2348,2349,2350,2351,2352,2353,2354,2355,2356,2357,2358,2359,2360,2361,2362,2363,2364,2365,2367,2368,2369,2370,2371,2372,2373,2374,2375,2376,2377,2378,2379,2380,2381,2382,2383,2387,2388,2390,2393,2395,2396,2397,2398,2399,2400,2401,2402,2403,2404,2405,2407,2409,2411,2412,2414,2415,2416,2418,2419,2420,2422,2423,2424,2425,2428,2429,2430,2431,2432,2433,2434,2435,2436,2437,2438,2439,2440,2442,2444,2445,2446,2447,2448,2449,2451,2452,2453,2454,2455,2456,2457,2459,2460,2462,2463,2464,2465,2467,2468,2470,2471,2472,2473,2474,2475,2477,2478,2480,2481,2482,2483,2485,2486,2487,2488,2489,2490,2491,2492,2493,2494,2495,2496,2497,2498,2499,2500,2502,2503,2505,2506,2507,2509,2510,2511,2512,2517,2518,2519,2520,2521,2522,2523,2526,2527,2529,2530,2531,2532,2533,2534,2535,2536,2538,2540,2541,2542,2543,2544,2545,2546,2547,2548,2549,2551,2552,2553,2554,2555,2557,2559,2560,2561,2562,2563,2565,2566,2567,2568,2569,2571,2573,2574,2575,2576,2577,2578,2579,2580,2582,2583,2584,2586,2587,2588,2589,2591,2592,2594,2595,2596,2597,2598,2600,2601,2603,2604,2605,2608,2612,2614,2615,2616,2617,2618,2619,2620,2622,2623,2624,2626,2627,2628,2630,2631,2633,2634,2636,2638,2639,2640,2643,2645,2646,2647,2648,2649,2650,2652,2653,2654,2655,2656,2657,2658,2659,2660,2661,2662,2663,2664,2665,2666,2667,2668,2670,2671,2673,2674,2675,2676,2677,2679,2680,2681,2683,2684,2685,2687,2688,2689,2690,2691,2692,2694,2696,2698],\"valid\":[1,2,3,4,5,9,22,27,34,39,40,44,48,56,63,68,70,71,72,74,76,79,80,85,88,104,113,127,128,132,144,145,159,165,171,173,175,190,191,200,203,204,206,208,210,217,218,226,238,240,244,246,251,253,273,284,289,304,306,318,321,322,323,328,329,335,336,337,347,357,369,371,374,386,402,408,412,413,418,423,425,434,437,447,451,455,457,459,461,465,466,467,471,477,487,490,491,496,502,503,504,505,509,510,513,517,519,520,525,528,532,536,540,545,547,555,556,567,577,585,588,593,595,610,612,615,617,621,624,629,630,634,635,636,637,644,645,663,664,668,671,675,685,688,691,694,714,717,718,730,737,747,749,753,761,762,766,777,781,784,787,791,794,802,809,816,818,828,838,847,852,856,859,866,876,878,881,882,891,903,918,922,933,936,950,956,958,963,967,985,986,988,991,998,999,1002,1008,1011,1014,1017,1018,1021,1026,1030,1070,1074,1077,1082,1084,1095,1110,1112,1121,1128,1129,1133,1138,1141,1147,1160,1163,1164,1166,1169,1172,1174,1184,1188,1189,1197,1203,1206,1208,1216,1220,1221,1222,1227,1241,1246,1263,1265,1267,1268,1272,1279,1285,1290,1292,1298,1301,1304,1308,1319,1322,1330,1336,1340,1343,1346,1355,1362,1363,1367,1375,1383,1384,1386,1387,1389,1393,1396,1398,1402,1408,1411,1420,1429,1431,1432,1434,1436,1438,1439,1440,1448,1450,1456,1492,1497,1502,1503,1506,1509,1513,1521,1528,1538,1550,1554,1563,1567,1575,1590,1592,1595,1596,1604,1605,1616,1621,1625,1626,1632,1646,1648,1649,1654,1663,1665,1667,1673,1684,1692,1698,1705,1707,1724,1733,1735,1740,1744,1756,1762,1768,1769,1772,1778,1781,1787,1789,1792,1794,1796,1803,1804,1805,1809,1814,1818,1819,1820,1826,1829,1834,1836,1840,1841,1845,1846,1851,1854,1855,1856,1858,1859,1860,1862,1867,1875,1877,1878,1882,1891,1893,1894,1898,1900,1901,1905,1906,1911,1913,1917,1921,1929,1933,1942,1944,1947,1953,1955,1962,1967,1972,1975,1977,1980,1989,2001,2005,2009,2019,2025,2040,2043,2064,2065,2066,2068,2070,2074,2097,2098,2109,2113,2131,2132,2134,2137,2139,2141,2147,2149,2151,2162,2163,2169,2178,2179,2199,2206,2208,2209,2210,2213,2216,2224,2229,2230,2234,2237,2242,2244,2248,2253,2256,2260,2262,2264,2280,2281,2282,2285,2286,2290,2291,2292,2294,2303,2307,2322,2329,2331,2335,2345,2366,2384,2385,2386,2389,2391,2392,2394,2406,2408,2410,2413,2417,2421,2426,2427,2441,2443,2450,2458,2461,2466,2469,2476,2479,2484,2501,2504,2508,2513,2514,2515,2516,2524,2525,2528,2537,2539,2550,2556,2558,2564,2570,2572,2581,2585,2590,2593,2599,2602,2606,2607,2609,2610,2611,2613,2621,2625,2629,2632,2635,2637,2641,2642,2644,2651,2669,2672,2678,2682,2686,2693,2695,2697]},\"outer3_inner0\":{\"fit\":[6,8,10,11,12,14,15,17,18,19,23,24,25,26,28,29,30,31,32,33,35,36,37,41,43,45,47,49,51,53,54,58,59,61,62,64,65,66,67,69,77,78,81,82,83,84,86,90,91,92,94,95,100,102,103,105,107,110,111,112,116,117,118,119,122,124,126,129,131,134,135,136,137,138,139,141,142,143,146,148,149,151,152,153,154,157,158,160,161,162,164,168,169,170,174,177,180,182,184,187,188,192,193,195,196,197,199,202,205,209,211,212,213,214,215,216,219,220,221,223,224,227,228,230,231,232,233,234,235,236,237,239,243,245,248,250,252,255,257,258,260,262,263,265,266,267,269,270,271,272,274,275,277,278,279,281,282,283,285,287,288,291,292,294,295,296,298,300,308,309,310,312,313,314,315,317,320,324,330,331,332,338,340,342,343,345,346,350,351,353,355,356,358,359,361,362,364,365,370,372,373,376,377,378,380,383,384,387,388,390,391,392,393,395,397,398,399,400,401,403,405,406,407,410,411,417,420,421,426,427,428,430,431,433,438,440,442,448,450,452,453,454,456,458,460,462,463,468,469,472,473,474,475,476,478,479,482,483,484,485,486,488,489,494,495,499,500,506,507,508,511,515,516,518,521,523,526,527,529,530,531,534,535,537,538,539,541,542,543,546,548,549,550,552,554,557,560,561,562,563,565,566,568,569,571,572,573,574,575,576,580,582,584,586,587,589,592,594,596,598,601,602,605,607,608,609,611,613,614,616,618,622,623,626,628,631,632,633,638,639,640,641,642,646,648,649,651,652,653,654,658,659,660,662,666,667,672,673,676,678,679,680,681,684,686,687,690,692,693,695,696,697,699,701,702,703,706,707,709,710,711,712,713,715,719,720,721,723,724,726,729,732,734,738,740,741,742,743,744,745,748,751,752,754,756,757,758,759,760,763,764,765,768,769,770,772,774,775,778,779,780,785,786,788,789,790,792,795,796,797,798,799,800,801,805,807,808,810,811,812,813,815,817,820,822,824,825,827,829,830,831,832,833,834,837,839,840,841,842,845,848,849,850,851,853,855,857,858,860,861,865,867,868,869,870,871,872,874,877,885,886,887,888,889,890,892,893,894,895,897,898,899,900,901,902,907,908,909,910,911,912,913,917,919,921,923,924,926,928,929,931,934,935,937,939,941,942,944,945,946,947,948,949,951,953,955,957,960,961,965,966,969,970,972,973,974,975,976,977,978,980,982,983,987,989,992,994,995,997,1001,1003,1004,1005,1006,1010,1013,1015,1016,1019,1020,1023,1027,1028,1029,1031,1034,1036,1037,1043,1044,1045,1048,1050,1051,1052,1053,1054,1055,1056,1057,1058,1059,1060,1062,1063,1064,1065,1066,1069,1071,1072,1073,1079,1080,1083,1085,1087,1088,1089,1090,1091,1092,1094,1097,1098,1099,1100,1103,1105,1106,1107,1108,1113,1114,1115,1116,1118,1120,1122,1126,1127,1130,1131,1132,1134,1135,1136,1137,1142,1144,1146,1149,1150,1151,1152,1153,1154,1155,1156,1158,1159,1162,1165,1168,1170,1171,1173,1176,1177,1178,1179,1181,1182,1183,1185,1186,1187,1193,1194,1195,1200,1201,1202,1204,1205,1207,1211,1212,1214,1215,1219,1223,1224,1226,1228,1230,1231,1232,1233,1235,1236,1237,1239,1242,1243,1244,1247,1248,1249,1250,1252,1253,1254,1255,1256,1259,1262,1266,1269,1271,1273,1277,1281,1286,1289,1291,1295,1296,1297,1299,1300,1302,1303,1305,1309,1310,1311,1312,1313,1314,1315,1316,1317,1323,1325,1326,1327,1328,1331,1332,1333,1334,1335,1337,1338,1339,1348,1349,1350,1351,1352,1356,1358,1359,1360,1366,1369,1370,1373,1377,1378,1379,1380,1381,1382,1385,1388,1391,1392,1395,1399,1400,1401,1403,1404,1405,1406,1409,1410,1413,1418,1419,1422,1423,1425,1426,1427,1428,1430,1433,1437,1442,1445,1446,1447,1449,1451,1452,1454,1455,1457,1458,1459,1460,1461,1462,1466,1467,1468,1469,1471,1473,1474,1475,1476,1478,1480,1483,1484,1485,1486,1487,1488,1493,1496,1498,1500,1501,1504,1507,1508,1511,1512,1514,1515,1516,1517,1518,1523,1524,1525,1527,1529,1531,1533,1534,1535,1536,1539,1542,1543,1544,1545,1546,1549,1551,1552,1553,1556,1557,1560,1565,1566,1568,1570,1571,1572,1574,1576,1578,1580,1581,1584,1586,1587,1589,1597,1598,1601,1602,1606,1608,1609,1610,1611,1613,1614,1618,1619,1624,1627,1628,1629,1630,1631,1635,1637,1638,1639,1641,1643,1644,1645,1647,1650,1652,1653,1656,1657,1658,1659,1661,1662,1664,1666,1668,1669,1670,1671,1672,1674,1675,1676,1677,1679,1680,1683,1685,1686,1689,1690,1693,1694,1695,1696,1697,1699,1701,1702,1704,1706,1708,1710,1712,1713,1716,1717,1718,1720,1721,1723,1725,1726,1727,1728,1729,1730,1731,1732,1734,1736,1737,1738,1741,1743,1745,1747,1748,1749,1750,1751,1752,1753,1754,1755,1757,1758,1759,1761,1764,1766,1767,1770,1771,1774,1775,1776,1777,1779,1783,1788,1790,1791,1793,1795,1798,1801,1802,1806,1807,1810,1811,1812,1813,1815,1817,1821,1823,1824,1825,1828,1830,1831,1835,1837,1838,1839,1842,1843,1847,1848,1850,1853,1861,1863,1864,1868,1869,1871,1872,1876,1879,1883,1885,1892,1895,1896,1899,1902,1903,1907,1909,1910,1916,1922,1924,1925,1926,1927,1928,1930,1931,1932,1934,1935,1936,1937,1940,1941,1943,1946,1950,1956,1957,1958,1961,1963,1966,1968,1970,1976,1978,1979,1981,1983,1984,1985,1987,1990,1992,1995,1996,1998,1999,2004,2006,2007,2008,2010,2011,2014,2015,2017,2018,2021,2022,2023,2024,2028,2030,2032,2034,2035,2036,2037,2038,2042,2044,2045,2046,2047,2049,2050,2051,2053,2057,2058,2059,2062,2067,2071,2073,2076,2077,2078,2080,2081,2082,2083,2085,2086,2087,2092,2093,2094,2096,2099,2100,2101,2102,2104,2105,2106,2107,2108,2111,2114,2115,2116,2117,2120,2121,2122,2123,2126,2127,2128,2129,2135,2140,2143,2144,2145,2148,2152,2153,2154,2155,2156,2157,2158,2159,2160,2161,2164,2166,2167,2168,2171,2172,2173,2174,2176,2180,2182,2183,2185,2186,2187,2188,2190,2192,2193,2194,2195,2196,2197,2198,2200,2201,2202,2203,2205,2212,2214,2215,2217,2218,2220,2221,2225,2233,2236,2238,2241,2246,2247,2251,2252,2255,2257,2258,2259,2263,2265,2266,2267,2268,2271,2272,2273,2274,2276,2277,2278,2279,2288,2295,2296,2297,2299,2300,2301,2304,2305,2308,2309,2310,2311,2312,2313,2315,2316,2317,2318,2319,2320,2321,2324,2325,2327,2332,2333,2334,2336,2338,2340,2341,2342,2343,2344,2346,2347,2349,2352,2353,2355,2356,2357,2360,2364,2368,2369,2375,2376,2377,2379,2380,2381,2382,2383,2388,2390,2396,2397,2398,2399,2401,2403,2404,2405,2407,2409,2411,2412,2414,2416,2418,2419,2422,2428,2431,2432,2433,2434,2435,2436,2439,2442,2444,2445,2446,2447,2448,2451,2452,2453,2454,2455,2456,2457,2459,2460,2462,2463,2467,2468,2471,2474,2477,2482,2483,2488,2489,2490,2491,2493,2494,2495,2496,2497,2499,2502,2503,2505,2511,2512,2518,2519,2520,2522,2523,2526,2532,2534,2536,2540,2541,2543,2544,2545,2546,2547,2548,2549,2551,2552,2553,2554,2557,2559,2562,2563,2565,2567,2571,2574,2576,2577,2578,2583,2584,2587,2588,2589,2594,2596,2597,2598,2600,2603,2604,2605,2608,2612,2615,2617,2618,2619,2622,2623,2624,2626,2627,2628,2630,2633,2638,2639,2643,2645,2646,2647,2648,2652,2653,2654,2655,2656,2658,2660,2662,2664,2666,2667,2668,2670,2673,2674,2675,2676,2679,2680,2683,2687,2690,2691,2696,2698],\"valid\":[0,7,13,16,20,21,38,42,46,50,52,55,57,60,73,75,87,89,93,96,97,98,99,101,106,108,109,114,115,120,121,123,125,130,133,140,147,150,155,156,163,166,167,172,176,178,179,181,183,185,186,189,194,198,201,207,222,225,229,241,242,247,249,254,256,259,261,264,268,276,280,286,290,293,297,299,301,302,303,305,307,311,316,319,325,326,327,333,334,339,341,344,348,349,352,354,360,363,366,367,368,375,379,381,382,385,389,394,396,404,409,414,415,416,419,422,424,429,432,435,436,439,441,443,444,445,446,449,464,470,480,481,492,493,497,498,501,512,514,522,524,533,544,551,553,558,559,564,570,578,579,581,583,590,591,597,599,600,603,604,606,619,620,625,627,643,647,650,655,656,657,661,665,669,670,674,677,682,683,689,698,700,704,705,708,716,722,725,727,728,731,733,735,736,739,746,750,755,767,771,773,776,782,783,793,803,804,806,814,819,821,823,826,835,836,843,844,846,854,862,863,864,873,875,879,880,883,884,896,904,905,906,914,915,916,920,925,927,930,932,938,940,943,952,954,959,962,964,968,971,979,981,984,990,993,996,1000,1007,1009,1012,1022,1024,1025,1032,1033,1035,1038,1039,1040,1041,1042,1046,1047,1049,1061,1067,1068,1075,1076,1078,1081,1086,1093,1096,1101,1102,1104,1109,1111,1117,1119,1123,1124,1125,1139,1140,1143,1145,1148,1157,1161,1167,1175,1180,1190,1191,1192,1196,1198,1199,1209,1210,1213,1217,1218,1225,1229,1234,1238,1240,1245,1251,1257,1258,1260,1261,1264,1270,1274,1275,1276,1278,1280,1282,1283,1284,1287,1288,1293,1294,1306,1307,1318,1320,1321,1324,1329,1341,1342,1344,1345,1347,1353,1354,1357,1361,1364,1365,1368,1371,1372,1374,1376,1390,1394,1397,1407,1412,1414,1415,1416,1417,1421,1424,1435,1441,1443,1444,1453,1463,1464,1465,1470,1472,1477,1479,1481,1482,1489,1490,1491,1494,1495,1499,1505,1510,1519,1520,1522,1526,1530,1532,1537,1540,1541,1547,1548,1555,1558,1559,1561,1562,1564,1569,1573,1577,1579,1582,1583,1585,1588,1591,1593,1594,1599,1600,1603,1607,1612,1615,1617,1620,1622,1623,1633,1634,1636,1640,1642,1651,1655,1660,1678,1681,1682,1687,1688,1691,1700,1703,1709,1711,1714,1715,1719,1722,1739,1742,1746,1760,1763,1765,1773,1780,1782,1784,1785,1786,1797,1799,1800,1808,1816,1822,1827,1832,1833,1844,1849,1852,1857,1865,1866,1870,1873,1874,1880,1881,1884,1886,1887,1888,1889,1890,1897,1904,1908,1912,1914,1915,1918,1919,1920,1923,1938,1939,1945,1948,1949,1951,1952,1954,1959,1960,1964,1965,1969,1971,1973,1974,1982,1986,1988,1991,1993,1994,1997,2000,2002,2003,2012,2013,2016,2020,2026,2027,2029,2031,2033,2039,2041,2048,2052,2054,2055,2056,2060,2061,2063,2069,2072,2075,2079,2084,2088,2089,2090,2091,2095,2103,2110,2112,2118,2119,2124,2125,2130,2133,2136,2138,2142,2146,2150,2165,2170,2175,2177,2181,2184,2189,2191,2204,2207,2211,2219,2222,2223,2226,2227,2228,2231,2232,2235,2239,2240,2243,2245,2249,2250,2254,2261,2269,2270,2275,2283,2284,2287,2289,2293,2298,2302,2306,2314,2323,2326,2328,2330,2337,2339,2348,2350,2351,2354,2358,2359,2361,2362,2363,2365,2367,2370,2371,2372,2373,2374,2378,2387,2393,2395,2400,2402,2415,2420,2423,2424,2425,2429,2430,2437,2438,2440,2449,2464,2465,2470,2472,2473,2475,2478,2480,2481,2485,2486,2487,2492,2498,2500,2506,2507,2509,2510,2517,2521,2527,2529,2530,2531,2533,2535,2538,2542,2555,2560,2561,2566,2568,2569,2573,2575,2579,2580,2582,2586,2591,2592,2595,2601,2614,2616,2620,2631,2634,2636,2640,2649,2650,2657,2659,2661,2663,2665,2671,2677,2681,2684,2685,2688,2689,2692,2694]},\"outer3_inner1\":{\"fit\":[0,7,8,11,12,13,15,16,18,19,20,21,23,25,36,38,42,46,50,51,52,53,55,57,59,60,61,62,64,65,67,69,73,75,81,86,87,89,91,93,95,96,97,98,99,101,103,105,106,107,108,109,114,115,116,118,119,120,121,122,123,125,129,130,131,133,137,138,140,141,142,143,146,147,149,150,151,152,153,154,155,156,157,160,162,163,166,167,169,170,172,176,178,179,180,181,183,185,186,187,189,193,194,195,196,198,199,201,202,207,209,211,214,216,221,222,224,225,228,229,230,231,234,237,239,241,242,245,247,248,249,250,252,254,255,256,258,259,261,263,264,265,268,269,270,271,276,280,281,282,286,288,290,293,294,297,299,301,302,303,305,307,309,310,311,314,315,316,317,319,320,325,326,327,330,332,333,334,339,340,341,343,344,348,349,351,352,353,354,356,358,359,360,362,363,365,366,367,368,370,372,373,375,376,379,381,382,383,384,385,388,389,390,391,394,396,398,399,400,403,404,405,406,407,409,411,414,415,416,419,421,422,424,429,430,432,433,435,436,439,441,443,444,445,446,448,449,452,453,456,463,464,470,472,480,481,483,486,489,492,493,497,498,501,511,512,514,521,522,524,526,529,530,531,533,534,537,541,544,549,551,552,553,554,558,559,560,561,564,565,569,570,572,573,575,578,579,581,583,584,589,590,591,592,597,599,600,601,602,603,604,606,611,613,614,616,619,620,622,623,625,626,627,628,632,639,642,643,647,650,652,653,654,655,656,657,658,659,661,662,665,666,667,669,670,674,676,677,681,682,683,684,687,689,693,695,696,698,700,701,702,703,704,705,706,707,708,709,713,716,719,722,725,726,727,728,729,731,733,734,735,736,739,742,744,746,748,750,751,755,758,759,763,764,765,767,770,771,772,773,776,778,779,780,782,783,786,788,792,793,795,797,801,803,804,805,806,807,808,812,814,815,819,821,823,824,825,826,827,829,833,835,836,841,842,843,844,845,846,848,849,850,854,861,862,863,864,865,867,868,870,871,872,873,874,875,877,879,880,883,884,885,886,887,888,892,896,897,900,901,902,904,905,906,908,912,913,914,915,916,917,920,925,926,927,928,930,931,932,934,935,937,938,939,940,943,948,949,951,952,953,954,955,959,962,964,965,968,970,971,972,978,979,981,982,983,984,987,989,990,992,993,996,997,1000,1003,1004,1005,1006,1007,1009,1012,1013,1015,1019,1020,1022,1023,1024,1025,1027,1031,1032,1033,1035,1036,1037,1038,1039,1040,1041,1042,1044,1045,1046,1047,1048,1049,1050,1052,1057,1059,1060,1061,1062,1064,1067,1068,1069,1072,1075,1076,1078,1080,1081,1083,1085,1086,1089,1092,1093,1094,1096,1097,1098,1100,1101,1102,1103,1104,1106,1109,1111,1114,1117,1119,1122,1123,1124,1125,1127,1130,1132,1136,1139,1140,1142,1143,1145,1148,1150,1153,1155,1157,1158,1161,1162,1165,1167,1168,1170,1171,1173,1175,1180,1187,1190,1191,1192,1193,1196,1198,1199,1200,1201,1202,1207,1209,1210,1212,1213,1215,1217,1218,1224,1225,1229,1230,1233,1234,1235,1236,1238,1239,1240,1242,1243,1244,1245,1250,1251,1253,1254,1255,1256,1257,1258,1259,1260,1261,1262,1264,1266,1270,1271,1274,1275,1276,1277,1278,1280,1281,1282,1283,1284,1287,1288,1289,1293,1294,1295,1296,1299,1300,1302,1303,1305,1306,1307,1309,1311,1312,1314,1317,1318,1320,1321,1324,1325,1329,1334,1337,1341,1342,1344,1345,1347,1348,1350,1353,1354,1357,1361,1364,1365,1366,1368,1370,1371,1372,1374,1376,1379,1380,1385,1390,1394,1397,1403,1405,1407,1409,1412,1413,1414,1415,1416,1417,1419,1421,1422,1424,1425,1428,1430,1435,1437,1441,1442,1443,1444,1445,1451,1453,1455,1457,1458,1460,1461,1462,1463,1464,1465,1467,1470,1471,1472,1473,1476,1477,1478,1479,1481,1482,1483,1485,1486,1488,1489,1490,1491,1494,1495,1499,1501,1505,1508,1510,1511,1514,1515,1519,1520,1522,1523,1526,1527,1529,1530,1531,1532,1533,1537,1539,1540,1541,1543,1544,1545,1546,1547,1548,1549,1551,1553,1555,1556,1558,1559,1560,1561,1562,1564,1566,1569,1571,1572,1573,1576,1577,1578,1579,1581,1582,1583,1584,1585,1586,1588,1589,1591,1593,1594,1598,1599,1600,1601,1602,1603,1607,1609,1611,1612,1615,1617,1618,1619,1620,1622,1623,1627,1633,1634,1635,1636,1637,1640,1641,1642,1644,1647,1651,1653,1655,1657,1658,1660,1664,1666,1670,1671,1672,1674,1676,1677,1678,1680,1681,1682,1686,1687,1688,1690,1691,1695,1697,1699,1700,1701,1703,1706,1708,1709,1711,1713,1714,1715,1716,1719,1720,1721,1722,1723,1727,1729,1732,1736,1738,1739,1742,1743,1745,1746,1747,1748,1749,1750,1757,1758,1760,1761,1763,1765,1766,1771,1773,1774,1775,1777,1779,1780,1782,1784,1785,1786,1790,1791,1793,1797,1799,1800,1806,1807,1808,1811,1812,1816,1817,1822,1825,1827,1831,1832,1833,1835,1837,1842,1843,1844,1847,1849,1852,1853,1857,1863,1865,1866,1869,1870,1871,1872,1873,1874,1879,1880,1881,1884,1885,1886,1887,1888,1889,1890,1897,1899,1902,1903,1904,1907,1908,1910,1912,1914,1915,1918,1919,1920,1923,1926,1927,1928,1935,1937,1938,1939,1940,1941,1945,1946,1948,1949,1950,1951,1952,1954,1957,1958,1959,1960,1963,1964,1965,1969,1971,1973,1974,1976,1981,1982,1983,1985,1986,1988,1990,1991,1992,1993,1994,1995,1996,1997,1999,2000,2002,2003,2006,2008,2011,2012,2013,2014,2015,2016,2018,2020,2021,2022,2023,2024,2026,2027,2029,2030,2031,2033,2034,2035,2036,2037,2038,2039,2041,2044,2045,2046,2047,2048,2049,2050,2052,2054,2055,2056,2060,2061,2063,2069,2072,2073,2075,2077,2079,2081,2082,2083,2084,2086,2087,2088,2089,2090,2091,2092,2093,2095,2096,2101,2102,2103,2105,2106,2108,2110,2111,2112,2114,2115,2116,2117,2118,2119,2120,2121,2122,2123,2124,2125,2126,2128,2129,2130,2133,2135,2136,2138,2142,2143,2144,2146,2150,2153,2154,2155,2157,2160,2165,2170,2172,2173,2175,2176,2177,2181,2183,2184,2188,2189,2191,2192,2194,2195,2196,2198,2200,2202,2203,2204,2207,2211,2219,2222,2223,2226,2227,2228,2231,2232,2235,2236,2239,2240,2243,2245,2249,2250,2252,2254,2257,2261,2266,2267,2268,2269,2270,2271,2273,2275,2276,2277,2278,2279,2283,2284,2287,2288,2289,2293,2295,2296,2298,2300,2302,2304,2305,2306,2309,2310,2312,2313,2314,2315,2316,2318,2319,2323,2325,2326,2328,2330,2332,2333,2334,2336,2337,2339,2341,2342,2343,2346,2347,2348,2350,2351,2353,2354,2356,2358,2359,2361,2362,2363,2364,2365,2367,2369,2370,2371,2372,2373,2374,2376,2377,2378,2380,2381,2382,2383,2387,2390,2393,2395,2400,2402,2403,2404,2405,2411,2415,2419,2420,2422,2423,2424,2425,2428,2429,2430,2433,2435,2436,2437,2438,2440,2442,2444,2446,2447,2448,2449,2452,2453,2457,2464,2465,2467,2468,2470,2472,2473,2475,2478,2480,2481,2483,2485,2486,2487,2488,2489,2490,2491,2492,2493,2494,2495,2497,2498,2500,2503,2505,2506,2507,2509,2510,2511,2517,2520,2521,2522,2523,2526,2527,2529,2530,2531,2532,2533,2534,2535,2536,2538,2542,2545,2546,2549,2551,2553,2555,2560,2561,2565,2566,2568,2569,2571,2573,2574,2575,2576,2579,2580,2582,2586,2587,2588,2589,2591,2592,2595,2598,2601,2612,2614,2616,2619,2620,2622,2623,2631,2633,2634,2636,2640,2645,2646,2647,2649,2650,2652,2655,2657,2659,2661,2663,2664,2665,2668,2670,2671,2677,2679,2681,2684,2685,2688,2689,2692,2694,2698],\"valid\":[6,10,14,17,24,26,28,29,30,31,32,33,35,37,41,43,45,47,49,54,58,66,77,78,82,83,84,90,92,94,100,102,110,111,112,117,124,126,134,135,136,139,148,158,161,164,168,174,177,182,184,188,192,197,205,212,213,215,219,220,223,227,232,233,235,236,243,257,260,262,266,267,272,274,275,277,278,279,283,285,287,291,292,295,296,298,300,308,312,313,324,331,338,342,345,346,350,355,361,364,377,378,380,387,392,393,395,397,401,410,417,420,426,427,428,431,438,440,442,450,454,458,460,462,468,469,473,474,475,476,478,479,482,484,485,488,494,495,499,500,506,507,508,515,516,518,523,527,535,538,539,542,543,546,548,550,557,562,563,566,568,571,574,576,580,582,586,587,594,596,598,605,607,608,609,618,631,633,638,640,641,646,648,649,651,660,672,673,678,679,680,686,690,692,697,699,710,711,712,715,720,721,723,724,732,738,740,741,743,745,752,754,756,757,760,768,769,774,775,785,789,790,796,798,799,800,810,811,813,817,820,822,830,831,832,834,837,839,840,851,853,855,857,858,860,869,889,890,893,894,895,898,899,907,909,910,911,919,921,923,924,929,941,942,944,945,946,947,957,960,961,966,969,973,974,975,976,977,980,994,995,1001,1010,1016,1028,1029,1034,1043,1051,1053,1054,1055,1056,1058,1063,1065,1066,1071,1073,1079,1087,1088,1090,1091,1099,1105,1107,1108,1113,1115,1116,1118,1120,1126,1131,1134,1135,1137,1144,1146,1149,1151,1152,1154,1156,1159,1176,1177,1178,1179,1181,1182,1183,1185,1186,1194,1195,1204,1205,1211,1214,1219,1223,1226,1228,1231,1232,1237,1247,1248,1249,1252,1269,1273,1286,1291,1297,1310,1313,1315,1316,1323,1326,1327,1328,1331,1332,1333,1335,1338,1339,1349,1351,1352,1356,1358,1359,1360,1369,1373,1377,1378,1381,1382,1388,1391,1392,1395,1399,1400,1401,1404,1406,1410,1418,1423,1426,1427,1433,1446,1447,1449,1452,1454,1459,1466,1468,1469,1474,1475,1480,1484,1487,1493,1496,1498,1500,1504,1507,1512,1516,1517,1518,1524,1525,1534,1535,1536,1542,1552,1557,1565,1568,1570,1574,1580,1587,1597,1606,1608,1610,1613,1614,1624,1628,1629,1630,1631,1638,1639,1643,1645,1650,1652,1656,1659,1661,1662,1668,1669,1675,1679,1683,1685,1689,1693,1694,1696,1702,1704,1710,1712,1717,1718,1725,1726,1728,1730,1731,1734,1737,1741,1751,1752,1753,1754,1755,1759,1764,1767,1770,1776,1783,1788,1795,1798,1801,1802,1810,1813,1815,1821,1823,1824,1828,1830,1838,1839,1848,1850,1861,1864,1868,1876,1883,1892,1895,1896,1909,1916,1922,1924,1925,1930,1931,1932,1934,1936,1943,1956,1961,1966,1968,1970,1978,1979,1984,1987,1998,2004,2007,2010,2017,2028,2032,2042,2051,2053,2057,2058,2059,2062,2067,2071,2076,2078,2080,2085,2094,2099,2100,2104,2107,2127,2140,2145,2148,2152,2156,2158,2159,2161,2164,2166,2167,2168,2171,2174,2180,2182,2185,2186,2187,2190,2193,2197,2201,2205,2212,2214,2215,2217,2218,2220,2221,2225,2233,2238,2241,2246,2247,2251,2255,2258,2259,2263,2265,2272,2274,2297,2299,2301,2308,2311,2317,2320,2321,2324,2327,2338,2340,2344,2349,2352,2355,2357,2360,2368,2375,2379,2388,2396,2397,2398,2399,2401,2407,2409,2412,2414,2416,2418,2431,2432,2434,2439,2445,2451,2454,2455,2456,2459,2460,2462,2463,2471,2474,2477,2482,2496,2499,2502,2512,2518,2519,2540,2541,2543,2544,2547,2548,2552,2554,2557,2559,2562,2563,2567,2577,2578,2583,2584,2594,2596,2597,2600,2603,2604,2605,2608,2615,2617,2618,2624,2626,2627,2628,2630,2638,2639,2643,2648,2653,2654,2656,2658,2660,2662,2666,2667,2673,2674,2675,2676,2680,2683,2687,2690,2691,2696]},\"outer3_inner2\":{\"fit\":[0,6,7,10,13,14,16,17,20,21,24,26,28,29,30,31,32,33,35,37,38,41,42,43,45,46,47,49,50,52,54,55,57,58,60,66,73,75,77,78,82,83,84,87,89,90,92,93,94,96,97,98,99,100,101,102,106,108,109,110,111,112,114,115,117,120,121,123,124,125,126,130,133,134,135,136,139,140,147,148,150,155,156,158,161,163,164,166,167,168,172,174,176,177,178,179,181,182,183,184,185,186,188,189,192,194,197,198,201,205,207,212,213,215,219,220,222,223,225,227,229,232,233,235,236,241,242,243,247,249,254,256,257,259,260,261,262,264,266,267,268,272,274,275,276,277,278,279,280,283,285,286,287,290,291,292,293,295,296,297,298,299,300,301,302,303,305,307,308,311,312,313,316,319,324,325,326,327,331,333,334,338,339,341,342,344,345,346,348,349,350,352,354,355,360,361,363,364,366,367,368,375,377,378,379,380,381,382,385,387,389,392,393,394,395,396,397,401,404,409,410,414,415,416,417,419,420,422,424,426,427,428,429,431,432,435,436,438,439,440,441,442,443,444,445,446,449,450,454,458,460,462,464,468,469,470,473,474,475,476,478,479,480,481,482,484,485,488,492,493,494,495,497,498,499,500,501,506,507,508,512,514,515,516,518,522,523,524,527,533,535,538,539,542,543,544,546,548,550,551,553,557,558,559,562,563,564,566,568,570,571,574,576,578,579,580,581,582,583,586,587,590,591,594,596,597,598,599,600,603,604,605,606,607,608,609,618,619,620,625,627,631,633,638,640,641,643,646,647,648,649,650,651,655,656,657,660,661,665,669,670,672,673,674,677,678,679,680,682,683,686,689,690,692,697,698,699,700,704,705,708,710,711,712,715,716,720,721,722,723,724,725,727,728,731,732,733,735,736,738,739,740,741,743,745,746,750,752,754,755,756,757,760,767,768,769,771,773,774,775,776,782,783,785,789,790,793,796,798,799,800,803,804,806,810,811,813,814,817,819,820,821,822,823,826,830,831,832,834,835,836,837,839,840,843,844,846,851,853,854,855,857,858,860,862,863,864,869,873,875,879,880,883,884,889,890,893,894,895,896,898,899,904,905,906,907,909,910,911,914,915,916,919,920,921,923,924,925,927,929,930,932,938,940,941,942,943,944,945,946,947,952,954,957,959,960,961,962,964,966,968,969,971,973,974,975,976,977,979,980,981,984,990,993,994,995,996,1000,1001,1007,1009,1010,1012,1016,1022,1024,1025,1028,1029,1032,1033,1034,1035,1038,1039,1040,1041,1042,1043,1046,1047,1049,1051,1053,1054,1055,1056,1058,1061,1063,1065,1066,1067,1068,1071,1073,1075,1076,1078,1079,1081,1086,1087,1088,1090,1091,1093,1096,1099,1101,1102,1104,1105,1107,1108,1109,1111,1113,1115,1116,1117,1118,1119,1120,1123,1124,1125,1126,1131,1134,1135,1137,1139,1140,1143,1144,1145,1146,1148,1149,1151,1152,1154,1156,1157,1159,1161,1167,1175,1176,1177,1178,1179,1180,1181,1182,1183,1185,1186,1190,1191,1192,1194,1195,1196,1198,1199,1204,1205,1209,1210,1211,1213,1214,1217,1218,1219,1223,1225,1226,1228,1229,1231,1232,1234,1237,1238,1240,1245,1247,1248,1249,1251,1252,1257,1258,1260,1261,1264,1269,1270,1273,1274,1275,1276,1278,1280,1282,1283,1284,1286,1287,1288,1291,1293,1294,1297,1306,1307,1310,1313,1315,1316,1318,1320,1321,1323,1324,1326,1327,1328,1329,1331,1332,1333,1335,1338,1339,1341,1342,1344,1345,1347,1349,1351,1352,1353,1354,1356,1357,1358,1359,1360,1361,1364,1365,1368,1369,1371,1372,1373,1374,1376,1377,1378,1381,1382,1388,1390,1391,1392,1394,1395,1397,1399,1400,1401,1404,1406,1407,1410,1412,1414,1415,1416,1417,1418,1421,1423,1424,1426,1427,1433,1435,1441,1443,1444,1446,1447,1449,1452,1453,1454,1459,1463,1464,1465,1466,1468,1469,1470,1472,1474,1475,1477,1479,1480,1481,1482,1484,1487,1489,1490,1491,1493,1494,1495,1496,1498,1499,1500,1504,1505,1507,1510,1512,1516,1517,1518,1519,1520,1522,1524,1525,1526,1530,1532,1534,1535,1536,1537,1540,1541,1542,1547,1548,1552,1555,1557,1558,1559,1561,1562,1564,1565,1568,1569,1570,1573,1574,1577,1579,1580,1582,1583,1585,1587,1588,1591,1593,1594,1597,1599,1600,1603,1606,1607,1608,1610,1612,1613,1614,1615,1617,1620,1622,1623,1624,1628,1629,1630,1631,1633,1634,1636,1638,1639,1640,1642,1643,1645,1650,1651,1652,1655,1656,1659,1660,1661,1662,1668,1669,1675,1678,1679,1681,1682,1683,1685,1687,1688,1689,1691,1693,1694,1696,1700,1702,1703,1704,1709,1710,1711,1712,1714,1715,1717,1718,1719,1722,1725,1726,1728,1730,1731,1734,1737,1739,1741,1742,1746,1751,1752,1753,1754,1755,1759,1760,1763,1764,1765,1767,1770,1773,1776,1780,1782,1783,1784,1785,1786,1788,1795,1797,1798,1799,1800,1801,1802,1808,1810,1813,1815,1816,1821,1822,1823,1824,1827,1828,1830,1832,1833,1838,1839,1844,1848,1849,1850,1852,1857,1861,1864,1865,1866,1868,1870,1873,1874,1876,1880,1881,1883,1884,1886,1887,1888,1889,1890,1892,1895,1896,1897,1904,1908,1909,1912,1914,1915,1916,1918,1919,1920,1922,1923,1924,1925,1930,1931,1932,1934,1936,1938,1939,1943,1945,1948,1949,1951,1952,1954,1956,1959,1960,1961,1964,1965,1966,1968,1969,1970,1971,1973,1974,1978,1979,1982,1984,1986,1987,1988,1991,1993,1994,1997,1998,2000,2002,2003,2004,2007,2010,2012,2013,2016,2017,2020,2026,2027,2028,2029,2031,2032,2033,2039,2041,2042,2048,2051,2052,2053,2054,2055,2056,2057,2058,2059,2060,2061,2062,2063,2067,2069,2071,2072,2075,2076,2078,2079,2080,2084,2085,2088,2089,2090,2091,2094,2095,2099,2100,2103,2104,2107,2110,2112,2118,2119,2124,2125,2127,2130,2133,2136,2138,2140,2142,2145,2146,2148,2150,2152,2156,2158,2159,2161,2164,2165,2166,2167,2168,2170,2171,2174,2175,2177,2180,2181,2182,2184,2185,2186,2187,2189,2190,2191,2193,2197,2201,2204,2205,2207,2211,2212,2214,2215,2217,2218,2219,2220,2221,2222,2223,2225,2226,2227,2228,2231,2232,2233,2235,2238,2239,2240,2241,2243,2245,2246,2247,2249,2250,2251,2254,2255,2258,2259,2261,2263,2265,2269,2270,2272,2274,2275,2283,2284,2287,2289,2293,2297,2298,2299,2301,2302,2306,2308,2311,2314,2317,2320,2321,2323,2324,2326,2327,2328,2330,2337,2338,2339,2340,2344,2348,2349,2350,2351,2352,2354,2355,2357,2358,2359,2360,2361,2362,2363,2365,2367,2368,2370,2371,2372,2373,2374,2375,2378,2379,2387,2388,2393,2395,2396,2397,2398,2399,2400,2401,2402,2407,2409,2412,2414,2415,2416,2418,2420,2423,2424,2425,2429,2430,2431,2432,2434,2437,2438,2439,2440,2445,2449,2451,2454,2455,2456,2459,2460,2462,2463,2464,2465,2470,2471,2472,2473,2474,2475,2477,2478,2480,2481,2482,2485,2486,2487,2492,2496,2498,2499,2500,2502,2506,2507,2509,2510,2512,2517,2518,2519,2521,2527,2529,2530,2531,2533,2535,2538,2540,2541,2542,2543,2544,2547,2548,2552,2554,2555,2557,2559,2560,2561,2562,2563,2566,2567,2568,2569,2573,2575,2577,2578,2579,2580,2582,2583,2584,2586,2591,2592,2594,2595,2596,2597,2600,2601,2603,2604,2605,2608,2614,2615,2616,2617,2618,2620,2624,2626,2627,2628,2630,2631,2634,2636,2638,2639,2640,2643,2648,2649,2650,2653,2654,2656,2657,2658,2659,2660,2661,2662,2663,2665,2666,2667,2671,2673,2674,2675,2676,2677,2680,2681,2683,2684,2685,2687,2688,2689,2690,2691,2692,2694,2696],\"valid\":[8,11,12,15,18,19,23,25,36,51,53,59,61,62,64,65,67,69,81,86,91,95,103,105,107,116,118,119,122,129,131,137,138,141,142,143,146,149,151,152,153,154,157,160,162,169,170,180,187,193,195,196,199,202,209,211,214,216,221,224,228,230,231,234,237,239,245,248,250,252,255,258,263,265,269,270,271,281,282,288,294,309,310,314,315,317,320,330,332,340,343,351,353,356,358,359,362,365,370,372,373,376,383,384,388,390,391,398,399,400,403,405,406,407,411,421,430,433,448,452,453,456,463,472,483,486,489,511,521,526,529,530,531,534,537,541,549,552,554,560,561,565,569,572,573,575,584,589,592,601,602,611,613,614,616,622,623,626,628,632,639,642,652,653,654,658,659,662,666,667,676,681,684,687,693,695,696,701,702,703,706,707,709,713,719,726,729,734,742,744,748,751,758,759,763,764,765,770,772,778,779,780,786,788,792,795,797,801,805,807,808,812,815,824,825,827,829,833,841,842,845,848,849,850,861,865,867,868,870,871,872,874,877,885,886,887,888,892,897,900,901,902,908,912,913,917,926,928,931,934,935,937,939,948,949,951,953,955,965,970,972,978,982,983,987,989,992,997,1003,1004,1005,1006,1013,1015,1019,1020,1023,1027,1031,1036,1037,1044,1045,1048,1050,1052,1057,1059,1060,1062,1064,1069,1072,1080,1083,1085,1089,1092,1094,1097,1098,1100,1103,1106,1114,1122,1127,1130,1132,1136,1142,1150,1153,1155,1158,1162,1165,1168,1170,1171,1173,1187,1193,1200,1201,1202,1207,1212,1215,1224,1230,1233,1235,1236,1239,1242,1243,1244,1250,1253,1254,1255,1256,1259,1262,1266,1271,1277,1281,1289,1295,1296,1299,1300,1302,1303,1305,1309,1311,1312,1314,1317,1325,1334,1337,1348,1350,1366,1370,1379,1380,1385,1403,1405,1409,1413,1419,1422,1425,1428,1430,1437,1442,1445,1451,1455,1457,1458,1460,1461,1462,1467,1471,1473,1476,1478,1483,1485,1486,1488,1501,1508,1511,1514,1515,1523,1527,1529,1531,1533,1539,1543,1544,1545,1546,1549,1551,1553,1556,1560,1566,1571,1572,1576,1578,1581,1584,1586,1589,1598,1601,1602,1609,1611,1618,1619,1627,1635,1637,1641,1644,1647,1653,1657,1658,1664,1666,1670,1671,1672,1674,1676,1677,1680,1686,1690,1695,1697,1699,1701,1706,1708,1713,1716,1720,1721,1723,1727,1729,1732,1736,1738,1743,1745,1747,1748,1749,1750,1757,1758,1761,1766,1771,1774,1775,1777,1779,1790,1791,1793,1806,1807,1811,1812,1817,1825,1831,1835,1837,1842,1843,1847,1853,1863,1869,1871,1872,1879,1885,1899,1902,1903,1907,1910,1926,1927,1928,1935,1937,1940,1941,1946,1950,1957,1958,1963,1976,1981,1983,1985,1990,1992,1995,1996,1999,2006,2008,2011,2014,2015,2018,2021,2022,2023,2024,2030,2034,2035,2036,2037,2038,2044,2045,2046,2047,2049,2050,2073,2077,2081,2082,2083,2086,2087,2092,2093,2096,2101,2102,2105,2106,2108,2111,2114,2115,2116,2117,2120,2121,2122,2123,2126,2128,2129,2135,2143,2144,2153,2154,2155,2157,2160,2172,2173,2176,2183,2188,2192,2194,2195,2196,2198,2200,2202,2203,2236,2252,2257,2266,2267,2268,2271,2273,2276,2277,2278,2279,2288,2295,2296,2300,2304,2305,2309,2310,2312,2313,2315,2316,2318,2319,2325,2332,2333,2334,2336,2341,2342,2343,2346,2347,2353,2356,2364,2369,2376,2377,2380,2381,2382,2383,2390,2403,2404,2405,2411,2419,2422,2428,2433,2435,2436,2442,2444,2446,2447,2448,2452,2453,2457,2467,2468,2483,2488,2489,2490,2491,2493,2494,2495,2497,2503,2505,2511,2520,2522,2523,2526,2532,2534,2536,2545,2546,2549,2551,2553,2565,2571,2574,2576,2587,2588,2589,2598,2612,2619,2622,2623,2633,2645,2646,2647,2652,2655,2664,2668,2670,2679,2698]},\"outer4\":{\"fit\":[0,1,2,3,4,5,8,9,10,11,12,13,14,15,16,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,43,44,46,47,48,49,53,56,57,58,59,60,61,63,64,66,67,68,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,88,89,90,91,92,93,94,95,98,99,100,101,102,103,104,106,107,108,110,111,112,113,114,115,116,117,118,120,121,122,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,148,150,152,153,155,156,157,158,159,160,161,162,163,165,167,168,169,170,171,172,173,174,175,176,178,180,181,182,184,185,186,187,188,189,190,191,192,193,194,195,196,197,200,201,202,203,204,205,206,208,209,210,211,212,214,215,217,218,220,221,222,224,226,227,228,229,230,232,234,235,236,238,239,240,243,244,245,246,248,249,250,251,252,253,254,255,256,257,258,260,261,262,263,264,265,266,268,269,271,273,274,276,278,282,283,284,285,286,287,289,290,292,294,295,296,297,298,299,300,301,304,306,307,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,327,328,329,330,331,332,333,334,335,336,337,338,340,341,343,344,346,347,349,350,351,353,354,356,357,358,359,360,361,362,363,364,365,366,369,370,371,373,374,375,376,378,380,381,382,383,384,385,386,388,390,391,394,395,397,400,402,403,404,407,408,409,410,411,412,413,415,416,417,418,419,420,421,423,424,425,426,427,428,429,431,432,433,434,437,438,439,441,442,443,447,450,451,452,453,454,455,456,457,459,460,461,462,463,464,465,466,467,468,470,471,473,476,477,478,479,482,483,484,486,487,490,491,492,493,494,495,496,497,498,501,502,503,504,505,509,510,511,513,514,515,516,517,518,519,520,521,523,524,525,526,528,529,531,532,533,534,536,538,540,542,543,545,546,547,548,549,550,551,552,553,555,556,557,558,560,561,562,563,564,565,566,567,568,570,571,572,573,575,577,578,579,581,582,583,584,585,586,587,588,589,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,608,609,610,612,613,615,616,617,618,619,620,621,622,624,625,626,627,629,630,631,632,633,634,635,636,637,638,640,641,644,645,646,647,649,652,653,654,655,657,658,659,660,662,663,664,665,667,668,669,670,671,672,673,674,675,676,677,678,679,681,682,683,685,686,687,688,689,690,691,692,693,694,695,696,698,700,702,706,708,709,710,711,712,713,714,715,717,718,721,722,723,724,725,726,727,729,730,732,733,734,735,737,738,739,740,741,742,743,744,745,746,747,748,749,752,753,754,755,756,757,758,760,761,762,764,765,766,768,769,770,771,772,773,774,775,777,778,779,780,781,783,784,786,787,788,789,791,794,795,798,799,801,802,803,804,805,806,809,810,812,813,815,816,818,819,823,824,825,826,827,828,829,831,832,833,836,837,838,839,840,841,842,844,845,846,847,848,850,851,852,853,854,856,858,859,860,861,862,863,865,866,867,869,870,872,873,874,876,878,879,881,882,884,885,887,888,889,890,891,893,894,895,897,898,899,900,903,904,905,906,907,908,909,911,912,914,915,916,917,918,919,920,922,924,925,926,927,929,931,933,934,936,937,938,939,942,943,944,946,947,948,949,950,951,952,955,956,957,958,959,960,962,963,965,966,967,968,969,970,971,972,974,976,977,978,979,980,982,983,984,985,986,988,989,991,992,993,994,995,997,998,999,1000,1002,1003,1004,1006,1007,1008,1010,1011,1012,1013,1014,1015,1017,1018,1019,1020,1021,1022,1023,1024,1026,1027,1028,1029,1030,1031,1033,1034,1035,1036,1037,1038,1039,1040,1041,1042,1043,1044,1048,1049,1050,1051,1052,1057,1058,1062,1063,1064,1065,1066,1067,1070,1072,1074,1075,1076,1077,1078,1079,1080,1082,1084,1085,1086,1087,1088,1089,1091,1092,1093,1094,1095,1097,1098,1100,1101,1102,1103,1104,1105,1106,1107,1109,1110,1112,1113,1114,1116,1117,1119,1120,1121,1122,1128,1129,1130,1131,1132,1133,1134,1136,1137,1138,1140,1141,1142,1143,1144,1145,1147,1149,1150,1151,1152,1153,1155,1156,1157,1158,1159,1160,1161,1162,1163,1164,1165,1166,1168,1169,1170,1171,1172,1173,1174,1176,1177,1178,1181,1182,1183,1184,1187,1188,1189,1190,1192,1193,1194,1195,1196,1197,1198,1200,1202,1203,1205,1206,1207,1208,1209,1210,1211,1214,1215,1216,1217,1218,1219,1220,1221,1222,1223,1224,1225,1226,1227,1230,1231,1232,1233,1239,1240,1241,1242,1243,1244,1245,1246,1248,1249,1250,1252,1253,1254,1255,1256,1257,1258,1259,1261,1262,1263,1264,1265,1266,1267,1268,1269,1270,1272,1275,1277,1278,1279,1280,1281,1282,1283,1285,1286,1288,1289,1290,1291,1292,1293,1294,1295,1296,1297,1298,1299,1300,1301,1302,1304,1305,1308,1311,1313,1315,1319,1321,1322,1323,1324,1325,1326,1327,1328,1329,1330,1331,1332,1333,1334,1335,1336,1337,1338,1339,1340,1341,1343,1344,1346,1349,1350,1351,1352,1353,1355,1356,1357,1359,1361,1362,1363,1364,1365,1366,1367,1368,1370,1372,1373,1374,1375,1377,1380,1382,1383,1384,1385,1386,1387,1389,1390,1391,1392,1393,1394,1395,1396,1397,1398,1399,1400,1402,1403,1404,1405,1406,1407,1408,1409,1410,1411,1412,1413,1414,1416,1418,1419,1420,1421,1422,1423,1424,1425,1426,1427,1428,1429,1430,1431,1432,1433,1434,1435,1436,1438,1439,1440,1441,1442,1444,1445,1446,1447,1448,1449,1450,1453,1454,1455,1456,1457,1458,1459,1460,1463,1464,1466,1468,1469,1470,1471,1472,1473,1474,1476,1477,1478,1479,1480,1481,1482,1483,1484,1485,1486,1487,1488,1490,1491,1492,1493,1495,1496,1497,1498,1499,1500,1501,1502,1503,1506,1508,1509,1512,1513,1515,1516,1518,1519,1520,1521,1522,1525,1526,1528,1529,1531,1533,1534,1535,1536,1537,1538,1540,1541,1543,1544,1545,1547,1548,1549,1550,1553,1554,1555,1556,1557,1558,1559,1560,1561,1562,1563,1565,1566,1567,1568,1571,1574,1575,1576,1578,1581,1582,1584,1585,1586,1587,1588,1589,1590,1591,1592,1593,1594,1595,1596,1597,1598,1599,1600,1601,1602,1603,1604,1605,1606,1607,1608,1609,1610,1612,1613,1614,1615,1616,1618,1619,1621,1622,1623,1624,1625,1626,1628,1629,1631,1632,1633,1635,1636,1637,1638,1640,1641,1642,1644,1645,1646,1647,1648,1649,1652,1653,1654,1655,1656,1658,1659,1661,1662,1663,1664,1665,1666,1667,1668,1669,1671,1673,1674,1675,1676,1677,1679,1680,1681,1683,1684,1685,1686,1689,1690,1691,1692,1693,1696,1698,1699,1700,1701,1702,1703,1704,1705,1706,1707,1708,1709,1710,1711,1712,1713,1714,1715,1716,1718,1719,1720,1723,1724,1726,1727,1728,1730,1731,1733,1734,1735,1736,1737,1738,1739,1740,1742,1744,1749,1750,1751,1752,1755,1756,1757,1758,1759,1761,1762,1763,1764,1765,1766,1767,1768,1769,1770,1772,1773,1774,1775,1777,1778,1779,1780,1781,1782,1783,1785,1786,1787,1788,1789,1790,1791,1792,1794,1796,1798,1799,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,1810,1811,1812,1813,1814,1815,1816,1817,1818,1819,1820,1821,1822,1824,1825,1826,1827,1828,1829,1830,1831,1832,1833,1834,1835,1836,1837,1838,1840,1841,1842,1843,1844,1845,1846,1847,1849,1851,1852,1854,1855,1856,1857,1858,1859,1860,1861,1862,1863,1864,1865,1866,1867,1868,1869,1870,1871,1872,1873,1875,1876,1877,1878,1879,1880,1881,1882,1883,1884,1887,1889,1890,1891,1892,1893,1894,1895,1897,1898,1900,1901,1903,1905,1906,1907,1908,1909,1910,1911,1912,1913,1914,1916,1917,1919,1921,1922,1924,1925,1926,1927,1929,1930,1932,1933,1934,1935,1936,1939,1940,1941,1942,1943,1944,1947,1948,1949,1951,1952,1953,1954,1955,1956,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1989,1990,1993,1994,1995,1996,1998,1999,2000,2001,2003,2004,2005,2007,2008,2009,2013,2014,2015,2016,2017,2018,2019,2020,2021,2023,2024,2025,2026,2027,2028,2031,2032,2034,2035,2036,2037,2038,2040,2041,2042,2043,2044,2045,2049,2050,2051,2052,2053,2054,2057,2058,2059,2060,2061,2062,2063,2064,2065,2066,2067,2068,2070,2071,2072,2073,2074,2075,2077,2078,2079,2080,2084,2086,2088,2089,2090,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100,2101,2104,2107,2109,2110,2111,2112,2113,2114,2115,2116,2117,2118,2119,2120,2121,2122,2123,2124,2126,2128,2130,2131,2132,2133,2134,2135,2137,2139,2140,2141,2142,2145,2146,2147,2148,2149,2151,2154,2155,2156,2157,2159,2161,2162,2163,2164,2166,2167,2168,2169,2170,2171,2172,2174,2176,2177,2178,2179,2181,2182,2185,2186,2187,2188,2189,2192,2193,2194,2195,2196,2197,2198,2199,2200,2201,2202,2203,2204,2206,2208,2209,2210,2211,2212,2213,2214,2215,2216,2217,2218,2220,2221,2223,2224,2226,2228,2229,2230,2231,2232,2233,2234,2235,2237,2238,2239,2240,2241,2242,2243,2244,2246,2247,2248,2249,2251,2252,2253,2254,2255,2256,2257,2258,2259,2260,2261,2262,2264,2265,2266,2267,2268,2269,2270,2271,2272,2273,2275,2276,2277,2278,2280,2281,2282,2285,2286,2289,2290,2291,2292,2293,2294,2295,2296,2297,2298,2299,2300,2301,2302,2303,2304,2305,2307,2308,2309,2310,2311,2312,2313,2314,2315,2316,2317,2319,2321,2322,2323,2325,2326,2327,2328,2329,2330,2331,2332,2334,2335,2337,2338,2339,2340,2342,2343,2344,2345,2346,2347,2348,2349,2351,2352,2353,2355,2356,2357,2358,2359,2360,2362,2363,2364,2366,2367,2368,2369,2370,2371,2372,2373,2374,2375,2376,2377,2378,2379,2380,2382,2383,2384,2385,2386,2387,2388,2389,2390,2391,2392,2393,2394,2397,2398,2399,2400,2401,2402,2404,2406,2407,2408,2409,2410,2411,2412,2413,2414,2415,2417,2418,2419,2421,2424,2425,2426,2427,2429,2430,2431,2432,2433,2434,2435,2436,2437,2438,2439,2441,2442,2443,2444,2445,2447,2448,2449,2450,2451,2452,2453,2454,2455,2457,2458,2459,2460,2461,2462,2463,2464,2465,2466,2467,2468,2469,2471,2473,2475,2476,2477,2479,2480,2482,2484,2485,2486,2488,2489,2490,2491,2492,2493,2494,2495,2496,2498,2499,2501,2502,2503,2504,2505,2506,2508,2509,2510,2511,2512,2513,2514,2515,2516,2518,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2531,2533,2534,2535,2537,2538,2539,2540,2541,2542,2543,2545,2546,2547,2548,2549,2550,2551,2552,2553,2555,2556,2557,2558,2559,2560,2562,2563,2564,2565,2566,2567,2568,2569,2570,2572,2573,2574,2575,2576,2578,2579,2580,2581,2582,2583,2584,2585,2586,2588,2589,2590,2591,2592,2593,2594,2595,2596,2597,2598,2599,2600,2601,2602,2603,2604,2605,2606,2607,2608,2609,2610,2611,2612,2613,2614,2615,2616,2617,2618,2620,2621,2622,2623,2625,2626,2627,2628,2629,2630,2631,2632,2633,2634,2635,2636,2637,2638,2639,2640,2641,2642,2643,2644,2645,2646,2648,2649,2650,2651,2652,2653,2655,2657,2658,2659,2660,2661,2663,2664,2666,2668,2669,2670,2671,2672,2673,2674,2676,2678,2679,2680,2681,2682,2683,2684,2685,2686,2687,2688,2689,2691,2693,2694,2695,2696,2697],\"valid\":[6,7,17,41,42,45,50,51,52,54,55,62,65,69,87,96,97,105,109,119,123,147,149,151,154,164,166,177,179,183,198,199,207,213,216,219,223,225,231,233,237,241,242,247,259,267,270,272,275,277,279,280,281,288,291,293,302,303,305,308,326,339,342,345,348,352,355,367,368,372,377,379,387,389,392,393,396,398,399,401,405,406,414,422,430,435,436,440,444,445,446,448,449,458,469,472,474,475,480,481,485,488,489,499,500,506,507,508,512,522,527,530,535,537,539,541,544,554,559,569,574,576,580,590,606,607,611,614,623,628,639,642,643,648,650,651,656,661,666,680,684,697,699,701,703,704,705,707,716,719,720,728,731,736,750,751,759,763,767,776,782,785,790,792,793,796,797,800,807,808,811,814,817,820,821,822,830,834,835,843,849,855,857,864,868,871,875,877,880,883,886,892,896,901,902,910,913,921,923,928,930,932,935,940,941,945,953,954,961,964,973,975,981,987,990,996,1001,1005,1009,1016,1025,1032,1045,1046,1047,1053,1054,1055,1056,1059,1060,1061,1068,1069,1071,1073,1081,1083,1090,1096,1099,1108,1111,1115,1118,1123,1124,1125,1126,1127,1135,1139,1146,1148,1154,1167,1175,1179,1180,1185,1186,1191,1199,1201,1204,1212,1213,1228,1229,1234,1235,1236,1237,1238,1247,1251,1260,1271,1273,1274,1276,1284,1287,1303,1306,1307,1309,1310,1312,1314,1316,1317,1318,1320,1342,1345,1347,1348,1354,1358,1360,1369,1371,1376,1378,1379,1381,1388,1401,1415,1417,1437,1443,1451,1452,1461,1462,1465,1467,1475,1489,1494,1504,1505,1507,1510,1511,1514,1517,1523,1524,1527,1530,1532,1539,1542,1546,1551,1552,1564,1569,1570,1572,1573,1577,1579,1580,1583,1611,1617,1620,1627,1630,1634,1639,1643,1650,1651,1657,1660,1670,1672,1678,1682,1687,1688,1694,1695,1697,1717,1721,1722,1725,1729,1732,1741,1743,1745,1746,1747,1748,1753,1754,1760,1771,1776,1784,1793,1795,1797,1823,1839,1848,1850,1853,1874,1885,1886,1888,1896,1899,1902,1904,1915,1918,1920,1923,1928,1931,1937,1938,1945,1946,1950,1973,1974,1988,1991,1992,1997,2002,2006,2010,2011,2012,2022,2029,2030,2033,2039,2046,2047,2048,2055,2056,2069,2076,2081,2082,2083,2085,2087,2102,2103,2105,2106,2108,2125,2127,2129,2136,2138,2143,2144,2150,2152,2153,2158,2160,2165,2173,2175,2180,2183,2184,2190,2191,2205,2207,2219,2222,2225,2227,2236,2245,2250,2263,2274,2279,2283,2284,2287,2288,2306,2318,2320,2324,2333,2336,2341,2350,2354,2361,2365,2381,2395,2396,2403,2405,2416,2420,2422,2423,2428,2440,2446,2456,2470,2472,2474,2478,2481,2483,2487,2497,2500,2507,2517,2519,2532,2536,2544,2554,2561,2571,2577,2587,2619,2624,2647,2654,2656,2662,2665,2667,2675,2677,2690,2692,2698]},\"outer4_inner0\":{\"fit\":[0,1,2,3,4,5,10,12,13,16,18,19,20,21,22,24,30,31,32,36,37,38,40,44,46,47,53,56,58,59,60,61,64,66,67,68,72,73,74,75,76,77,78,81,82,86,89,90,92,94,95,98,99,100,101,103,106,107,108,113,114,115,116,120,122,124,125,126,127,128,129,131,133,134,139,140,142,144,145,146,150,152,155,156,157,158,159,160,161,162,163,167,169,170,171,172,174,175,176,178,181,182,184,185,186,189,190,192,193,194,195,196,201,202,203,204,205,206,210,211,215,222,224,226,228,229,230,232,235,238,239,243,245,248,249,251,253,254,256,257,258,262,264,269,271,273,274,276,278,282,285,286,287,290,292,294,297,299,301,304,307,309,310,311,313,314,315,316,317,319,322,323,324,325,327,328,329,330,332,333,335,336,341,343,344,349,350,351,353,354,356,359,360,361,363,364,365,369,370,373,374,378,381,382,383,384,385,386,388,390,394,395,402,403,404,407,408,409,411,412,413,415,417,421,425,426,429,433,434,437,438,439,441,443,450,451,453,454,455,459,461,462,463,464,465,467,468,470,473,478,479,483,486,487,490,492,493,495,496,497,498,501,502,503,504,505,510,511,514,515,516,517,520,524,525,526,528,529,531,533,536,538,540,542,543,545,547,548,549,550,552,558,560,562,564,565,566,567,568,571,572,575,578,581,582,583,584,585,587,589,591,593,594,596,597,598,599,600,602,603,604,610,612,613,615,616,617,619,620,624,625,627,629,630,631,632,633,634,638,640,641,644,649,652,653,654,657,658,659,663,664,665,668,669,670,671,672,673,674,676,677,678,681,683,685,687,688,689,693,694,696,698,700,706,711,712,713,715,722,724,725,726,727,733,734,735,737,738,739,741,742,746,747,748,749,754,756,758,761,762,765,769,770,771,772,773,775,777,778,780,783,784,786,787,791,794,798,799,802,803,804,805,806,812,813,815,816,818,819,823,825,826,827,828,829,832,833,837,840,841,844,845,846,848,850,851,853,854,858,860,861,862,863,867,870,872,873,874,879,882,891,894,895,897,898,899,900,903,904,905,906,907,911,912,914,915,916,920,925,926,929,931,936,937,938,939,942,944,946,948,949,950,951,952,955,956,957,958,959,960,962,963,965,967,968,969,970,971,972,976,979,980,986,988,989,991,994,995,998,1000,1002,1003,1004,1006,1007,1011,1012,1015,1017,1018,1022,1023,1026,1028,1029,1030,1031,1033,1034,1035,1036,1037,1038,1039,1040,1041,1042,1043,1044,1050,1052,1057,1058,1062,1063,1064,1066,1067,1072,1074,1075,1076,1078,1082,1086,1087,1088,1089,1091,1093,1094,1097,1098,1100,1101,1102,1104,1106,1107,1109,1110,1112,1113,1114,1117,1119,1120,1121,1122,1128,1130,1131,1132,1134,1136,1137,1141,1142,1143,1144,1145,1150,1151,1152,1155,1156,1157,1158,1161,1162,1163,1164,1165,1166,1168,1169,1170,1172,1174,1176,1177,1181,1182,1183,1187,1188,1190,1192,1193,1194,1195,1198,1202,1208,1209,1210,1211,1214,1216,1219,1221,1223,1224,1225,1232,1233,1240,1241,1242,1244,1245,1249,1250,1252,1254,1255,1256,1257,1258,1259,1262,1265,1266,1267,1268,1269,1270,1277,1278,1280,1281,1282,1283,1285,1286,1288,1291,1293,1294,1295,1296,1297,1298,1299,1305,1311,1313,1321,1322,1323,1324,1325,1326,1327,1333,1336,1337,1338,1339,1341,1343,1344,1346,1349,1351,1352,1353,1357,1362,1364,1365,1368,1372,1373,1374,1377,1380,1382,1384,1385,1386,1387,1391,1392,1393,1394,1396,1397,1399,1400,1406,1407,1408,1409,1411,1412,1414,1416,1418,1420,1421,1423,1424,1425,1426,1427,1428,1429,1431,1432,1433,1434,1436,1438,1440,1441,1444,1447,1449,1456,1458,1459,1460,1463,1464,1470,1471,1472,1473,1474,1477,1479,1482,1484,1486,1487,1488,1490,1491,1492,1495,1496,1498,1499,1500,1501,1502,1506,1508,1513,1515,1518,1519,1521,1522,1525,1528,1531,1535,1536,1537,1538,1540,1541,1543,1544,1547,1548,1549,1553,1554,1555,1556,1558,1559,1560,1563,1566,1567,1568,1575,1578,1581,1582,1584,1585,1586,1588,1589,1590,1591,1592,1593,1595,1597,1598,1599,1600,1601,1603,1605,1607,1608,1609,1610,1612,1613,1614,1615,1616,1618,1619,1621,1622,1623,1628,1629,1631,1632,1635,1636,1640,1641,1642,1645,1646,1648,1652,1655,1658,1661,1663,1664,1665,1668,1671,1673,1675,1679,1680,1681,1683,1684,1689,1690,1692,1693,1699,1701,1702,1704,1706,1708,1709,1710,1711,1712,1714,1715,1716,1718,1723,1726,1728,1735,1737,1740,1742,1744,1749,1751,1752,1755,1756,1757,1758,1759,1761,1763,1766,1767,1769,1770,1772,1773,1774,1778,1779,1780,1782,1783,1785,1786,1788,1790,1792,1794,1798,1799,1802,1803,1804,1805,1806,1807,1809,1813,1816,1817,1818,1819,1821,1822,1824,1827,1832,1833,1834,1835,1836,1838,1841,1842,1843,1844,1846,1847,1851,1852,1855,1856,1857,1860,1863,1865,1866,1867,1869,1870,1871,1872,1873,1875,1876,1878,1879,1880,1881,1884,1887,1889,1890,1891,1893,1894,1897,1898,1907,1909,1911,1912,1913,1914,1916,1917,1919,1922,1924,1926,1929,1930,1933,1935,1936,1939,1940,1941,1944,1948,1951,1953,1954,1955,1956,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1971,1972,1976,1977,1978,1979,1982,1983,1985,1986,1989,1990,1993,1994,1995,1996,1999,2003,2004,2005,2007,2008,2009,2013,2014,2015,2017,2019,2020,2021,2023,2024,2026,2028,2031,2034,2035,2036,2037,2038,2040,2042,2044,2045,2049,2051,2052,2053,2057,2060,2061,2062,2063,2064,2065,2068,2070,2071,2072,2073,2075,2077,2078,2079,2084,2086,2089,2090,2091,2095,2099,2100,2110,2112,2115,2119,2121,2122,2123,2124,2126,2131,2132,2133,2135,2137,2139,2140,2141,2142,2147,2148,2149,2151,2154,2157,2159,2161,2162,2163,2170,2176,2179,2182,2185,2186,2187,2188,2189,2193,2194,2196,2197,2199,2201,2202,2203,2204,2206,2208,2210,2211,2212,2214,2216,2217,2220,2221,2223,2226,2228,2229,2231,2232,2233,2234,2235,2238,2239,2241,2242,2243,2248,2249,2251,2253,2254,2255,2256,2257,2258,2259,2261,2264,2267,2269,2270,2271,2272,2273,2277,2280,2285,2291,2292,2293,2294,2297,2299,2302,2303,2305,2307,2308,2314,2316,2321,2322,2323,2328,2330,2331,2332,2334,2335,2338,2339,2340,2343,2344,2346,2347,2348,2351,2353,2355,2356,2357,2358,2359,2360,2362,2363,2364,2366,2367,2370,2371,2373,2374,2376,2378,2380,2382,2383,2384,2385,2386,2387,2388,2389,2390,2391,2392,2393,2394,2397,2398,2399,2400,2401,2402,2407,2408,2411,2412,2413,2414,2415,2417,2418,2419,2421,2424,2427,2429,2430,2433,2434,2436,2437,2438,2441,2444,2445,2448,2449,2450,2452,2453,2458,2461,2462,2463,2465,2466,2467,2468,2471,2473,2477,2480,2484,2485,2486,2490,2491,2492,2493,2495,2498,2499,2501,2502,2503,2505,2506,2509,2510,2511,2512,2514,2515,2518,2522,2524,2525,2527,2529,2530,2531,2533,2534,2535,2537,2539,2540,2541,2542,2543,2545,2547,2548,2549,2550,2551,2552,2556,2557,2559,2560,2563,2566,2567,2569,2572,2573,2574,2579,2580,2581,2582,2583,2585,2586,2588,2589,2590,2591,2592,2595,2596,2599,2601,2603,2605,2606,2607,2608,2609,2611,2612,2613,2614,2615,2617,2620,2623,2625,2626,2627,2630,2631,2633,2634,2635,2637,2639,2641,2642,2645,2646,2649,2650,2651,2653,2655,2657,2658,2659,2660,2661,2663,2669,2671,2673,2674,2676,2680,2681,2682,2683,2684,2685,2686,2687,2688,2689,2691,2694,2695,2697],\"valid\":[8,9,11,14,15,23,25,26,27,28,29,33,34,35,39,43,48,49,57,63,70,71,79,80,83,84,85,88,91,93,102,104,110,111,112,117,118,121,130,132,135,136,137,138,141,143,148,153,165,168,173,180,187,188,191,197,200,208,209,212,214,217,218,220,221,227,234,236,240,244,246,250,252,255,260,261,263,265,266,268,283,284,289,295,296,298,300,306,312,318,320,321,331,334,337,338,340,346,347,357,358,362,366,371,375,376,380,391,397,400,410,416,418,419,420,423,424,427,428,431,432,442,447,452,456,457,460,466,471,476,477,482,484,491,494,509,513,518,519,521,523,532,534,546,551,553,555,556,557,561,563,570,573,577,579,586,588,592,595,601,605,608,609,618,621,622,626,635,636,637,645,646,647,655,660,662,667,675,679,682,686,690,691,692,695,702,708,709,710,714,717,718,721,723,729,730,732,740,743,744,745,752,753,755,757,760,764,766,768,774,779,781,788,789,795,801,809,810,824,831,836,838,839,842,847,852,856,859,865,866,869,876,878,881,884,885,887,888,889,890,893,908,909,917,918,919,922,924,927,933,934,943,947,966,974,977,978,982,983,984,985,992,993,997,999,1008,1010,1013,1014,1019,1020,1021,1024,1027,1048,1049,1051,1065,1070,1077,1079,1080,1084,1085,1092,1095,1103,1105,1116,1129,1133,1138,1140,1147,1149,1153,1159,1160,1171,1173,1178,1184,1189,1196,1197,1200,1203,1205,1206,1207,1215,1217,1218,1220,1222,1226,1227,1230,1231,1239,1243,1246,1248,1253,1261,1263,1264,1272,1275,1279,1289,1290,1292,1300,1301,1302,1304,1308,1315,1319,1328,1329,1330,1331,1332,1334,1335,1340,1350,1355,1356,1359,1361,1363,1366,1367,1370,1375,1383,1389,1390,1395,1398,1402,1403,1404,1405,1410,1413,1419,1422,1430,1435,1439,1442,1445,1446,1448,1450,1453,1454,1455,1457,1466,1468,1469,1476,1478,1480,1481,1483,1485,1493,1497,1503,1509,1512,1516,1520,1526,1529,1533,1534,1545,1550,1557,1561,1562,1565,1571,1574,1576,1587,1594,1596,1602,1604,1606,1624,1625,1626,1633,1637,1638,1644,1647,1649,1653,1654,1656,1659,1662,1666,1667,1669,1674,1676,1677,1685,1686,1691,1696,1698,1700,1703,1705,1707,1713,1719,1720,1724,1727,1730,1731,1733,1734,1736,1738,1739,1750,1762,1764,1765,1768,1775,1777,1781,1787,1789,1791,1796,1800,1801,1808,1810,1811,1812,1814,1815,1820,1825,1826,1828,1829,1830,1831,1837,1840,1845,1849,1854,1858,1859,1861,1862,1864,1868,1877,1882,1883,1892,1895,1900,1901,1903,1905,1906,1908,1910,1921,1925,1927,1932,1934,1942,1943,1947,1949,1952,1970,1975,1980,1981,1984,1987,1998,2000,2001,2016,2018,2025,2027,2032,2041,2043,2050,2054,2058,2059,2066,2067,2074,2080,2088,2092,2093,2094,2096,2097,2098,2101,2104,2107,2109,2111,2113,2114,2116,2117,2118,2120,2128,2130,2134,2145,2146,2155,2156,2164,2166,2167,2168,2169,2171,2172,2174,2177,2178,2181,2192,2195,2198,2200,2209,2213,2215,2218,2224,2230,2237,2240,2244,2246,2247,2252,2260,2262,2265,2266,2268,2275,2276,2278,2281,2282,2286,2289,2290,2295,2296,2298,2300,2301,2304,2309,2310,2311,2312,2313,2315,2317,2319,2325,2326,2327,2329,2337,2342,2345,2349,2352,2368,2369,2372,2375,2377,2379,2404,2406,2409,2410,2425,2426,2431,2432,2435,2439,2442,2443,2447,2451,2454,2455,2457,2459,2460,2464,2469,2475,2476,2479,2482,2488,2489,2494,2496,2504,2508,2513,2516,2520,2521,2523,2526,2528,2538,2546,2553,2555,2558,2562,2564,2565,2568,2570,2575,2576,2578,2584,2593,2594,2597,2598,2600,2602,2604,2610,2616,2618,2621,2622,2628,2629,2632,2636,2638,2640,2643,2644,2648,2652,2664,2666,2668,2670,2672,2678,2679,2693,2696]},\"outer4_inner1\":{\"fit\":[0,4,8,9,10,11,12,13,14,15,18,22,23,24,25,26,27,28,29,30,31,33,34,35,37,39,43,44,46,48,49,57,58,63,67,70,71,72,75,76,78,79,80,82,83,84,85,88,89,91,93,94,95,99,102,103,104,107,110,111,112,113,114,116,117,118,120,121,124,126,130,132,135,136,137,138,139,141,143,144,146,148,152,153,155,157,159,160,161,165,168,169,171,173,175,178,180,184,187,188,191,197,200,201,203,204,205,208,209,210,211,212,214,217,218,220,221,226,227,232,234,235,236,240,243,244,246,250,251,252,253,255,256,260,261,263,265,266,268,269,271,278,283,284,285,286,289,292,295,296,298,300,306,307,311,312,313,314,315,316,318,319,320,321,323,325,328,330,331,334,336,337,338,340,341,343,346,347,349,350,353,357,358,361,362,365,366,370,371,373,375,376,378,380,382,384,386,391,395,397,400,402,403,407,409,410,411,412,413,416,418,419,420,421,423,424,425,426,427,428,429,431,432,433,434,437,442,443,447,450,451,452,453,454,456,457,459,460,462,465,466,471,473,476,477,479,482,483,484,486,487,490,491,492,494,495,496,505,509,510,511,513,518,519,520,521,523,524,525,526,528,531,532,534,540,545,546,547,549,550,551,553,555,556,557,558,560,561,563,565,568,570,572,573,577,579,582,586,588,589,591,592,593,594,595,596,598,600,601,603,605,608,609,612,613,618,621,622,626,627,629,630,631,632,633,634,635,636,637,638,645,646,647,652,654,655,657,658,660,662,664,665,667,668,669,671,673,675,676,677,678,679,681,682,685,686,689,690,691,692,693,695,696,698,702,706,708,709,710,714,715,717,718,721,723,724,725,726,729,730,732,733,737,739,740,742,743,744,745,747,749,752,753,755,757,760,761,762,764,766,768,770,771,773,774,779,780,781,783,784,788,789,795,799,801,802,804,805,809,810,813,815,816,818,824,825,828,831,832,836,838,839,842,846,847,852,854,856,858,859,860,863,865,866,867,869,870,872,874,876,878,881,882,884,885,887,888,889,890,891,893,895,897,898,900,905,907,908,909,912,917,918,919,922,924,925,927,929,933,934,936,937,938,939,943,947,949,951,956,963,966,967,970,971,972,974,976,977,978,980,982,983,984,985,986,989,991,992,993,994,995,997,999,1002,1004,1006,1007,1008,1010,1011,1012,1013,1014,1017,1018,1019,1020,1021,1024,1027,1028,1031,1036,1038,1039,1040,1041,1042,1048,1049,1051,1063,1065,1066,1067,1070,1074,1076,1077,1079,1080,1084,1085,1092,1094,1095,1097,1098,1100,1101,1103,1104,1105,1110,1112,1114,1116,1122,1129,1131,1133,1138,1140,1141,1142,1143,1147,1149,1150,1151,1152,1153,1155,1157,1158,1159,1160,1161,1163,1164,1171,1172,1173,1174,1176,1178,1181,1184,1187,1189,1193,1194,1195,1196,1197,1200,1202,1203,1205,1206,1207,1209,1210,1214,1215,1216,1217,1218,1219,1220,1221,1222,1223,1224,1225,1226,1227,1230,1231,1239,1241,1242,1243,1246,1248,1250,1252,1253,1255,1257,1258,1261,1263,1264,1265,1266,1267,1270,1272,1275,1278,1279,1281,1282,1285,1286,1289,1290,1292,1293,1295,1297,1298,1299,1300,1301,1302,1304,1308,1315,1319,1321,1322,1323,1325,1327,1328,1329,1330,1331,1332,1334,1335,1336,1337,1340,1343,1350,1351,1353,1355,1356,1359,1361,1362,1363,1364,1365,1366,1367,1370,1373,1374,1375,1380,1383,1384,1387,1389,1390,1391,1393,1395,1396,1397,1398,1399,1400,1402,1403,1404,1405,1407,1408,1409,1410,1412,1413,1416,1418,1419,1420,1421,1422,1426,1427,1429,1430,1431,1433,1435,1438,1439,1442,1444,1445,1446,1448,1450,1453,1454,1455,1457,1458,1459,1464,1466,1468,1469,1470,1471,1476,1477,1478,1480,1481,1483,1485,1486,1487,1488,1493,1497,1498,1501,1503,1506,1509,1512,1515,1516,1520,1521,1526,1529,1531,1533,1534,1535,1536,1537,1538,1540,1541,1543,1544,1545,1550,1553,1554,1556,1557,1558,1560,1561,1562,1563,1565,1566,1567,1571,1574,1576,1581,1582,1587,1588,1589,1594,1595,1596,1598,1599,1602,1604,1606,1608,1609,1610,1612,1614,1615,1619,1624,1625,1626,1628,1632,1633,1637,1638,1644,1646,1647,1649,1653,1654,1656,1658,1659,1661,1662,1664,1665,1666,1667,1668,1669,1673,1674,1676,1677,1679,1680,1683,1684,1685,1686,1689,1691,1692,1696,1698,1700,1701,1703,1705,1706,1707,1708,1709,1710,1712,1713,1715,1716,1719,1720,1723,1724,1727,1728,1730,1731,1733,1734,1735,1736,1737,1738,1739,1740,1750,1756,1757,1758,1759,1761,1762,1764,1765,1766,1768,1769,1770,1773,1775,1777,1778,1779,1781,1782,1785,1786,1787,1788,1789,1790,1791,1792,1794,1796,1800,1801,1805,1807,1808,1810,1811,1812,1814,1815,1816,1818,1820,1821,1824,1825,1826,1828,1829,1830,1831,1832,1833,1834,1836,1837,1838,1840,1841,1843,1844,1845,1846,1847,1849,1852,1854,1855,1856,1857,1858,1859,1861,1862,1864,1867,1868,1870,1872,1877,1878,1879,1882,1883,1884,1890,1892,1893,1895,1898,1900,1901,1903,1905,1906,1908,1910,1911,1912,1913,1914,1917,1921,1922,1924,1925,1927,1929,1930,1932,1933,1934,1940,1941,1942,1943,1944,1947,1948,1949,1951,1952,1956,1957,1958,1962,1964,1966,1967,1969,1970,1971,1975,1976,1977,1979,1980,1981,1984,1987,1989,1994,1995,1998,1999,2000,2001,2003,2004,2005,2007,2009,2014,2016,2017,2018,2019,2020,2024,2025,2027,2032,2034,2035,2036,2037,2038,2040,2041,2043,2044,2045,2049,2050,2053,2054,2058,2059,2062,2065,2066,2067,2070,2071,2072,2073,2074,2077,2078,2080,2086,2088,2090,2091,2092,2093,2094,2096,2097,2098,2100,2101,2104,2107,2109,2111,2112,2113,2114,2115,2116,2117,2118,2119,2120,2122,2123,2128,2130,2132,2134,2137,2139,2140,2141,2145,2146,2147,2148,2149,2151,2154,2155,2156,2157,2159,2162,2164,2166,2167,2168,2169,2171,2172,2174,2177,2178,2181,2182,2185,2186,2187,2188,2192,2194,2195,2198,2200,2203,2206,2208,2209,2210,2213,2214,2215,2216,2218,2224,2229,2230,2231,2237,2238,2240,2241,2242,2243,2244,2246,2247,2248,2249,2251,2252,2253,2254,2255,2258,2260,2262,2264,2265,2266,2267,2268,2269,2271,2273,2275,2276,2278,2281,2282,2286,2289,2290,2291,2292,2293,2295,2296,2298,2300,2301,2304,2305,2307,2309,2310,2311,2312,2313,2315,2317,2319,2321,2323,2325,2326,2327,2329,2331,2332,2334,2337,2340,2342,2343,2345,2347,2348,2349,2351,2352,2356,2363,2364,2366,2368,2369,2370,2372,2373,2374,2375,2377,2379,2384,2385,2386,2387,2390,2392,2394,2399,2402,2404,2406,2407,2408,2409,2410,2413,2417,2419,2425,2426,2429,2431,2432,2433,2434,2435,2436,2439,2442,2443,2444,2447,2448,2450,2451,2452,2453,2454,2455,2457,2459,2460,2461,2462,2463,2464,2466,2468,2469,2471,2473,2475,2476,2477,2479,2482,2486,2488,2489,2491,2492,2494,2495,2496,2501,2502,2503,2504,2508,2509,2511,2513,2514,2516,2520,2521,2523,2526,2528,2531,2535,2538,2539,2541,2542,2543,2546,2549,2550,2551,2552,2553,2555,2556,2558,2559,2562,2563,2564,2565,2566,2568,2570,2572,2575,2576,2578,2579,2580,2583,2584,2585,2588,2589,2590,2591,2593,2594,2597,2598,2599,2600,2601,2602,2603,2604,2605,2606,2609,2610,2612,2614,2615,2616,2618,2621,2622,2623,2627,2628,2629,2630,2631,2632,2633,2634,2635,2636,2638,2640,2641,2642,2643,2644,2645,2646,2648,2649,2651,2652,2655,2664,2666,2668,2669,2670,2672,2674,2678,2679,2680,2681,2682,2683,2684,2687,2689,2693,2695,2696],\"valid\":[1,2,3,5,16,19,20,21,32,36,38,40,47,53,56,59,60,61,64,66,68,73,74,77,81,86,90,92,98,100,101,106,108,115,122,125,127,128,129,131,133,134,140,142,145,150,156,158,162,163,167,170,172,174,176,181,182,185,186,189,190,192,193,194,195,196,202,206,215,222,224,228,229,230,238,239,245,248,249,254,257,258,262,264,273,274,276,282,287,290,294,297,299,301,304,309,310,317,322,324,327,329,332,333,335,344,351,354,356,359,360,363,364,369,374,381,383,385,388,390,394,404,408,415,417,438,439,441,455,461,463,464,467,468,470,478,493,497,498,501,502,503,504,514,515,516,517,529,533,536,538,542,543,548,552,562,564,566,567,571,575,578,581,583,584,585,587,597,599,602,604,610,615,616,617,619,620,624,625,640,641,644,649,653,659,663,670,672,674,683,687,688,694,700,711,712,713,722,727,734,735,738,741,746,748,754,756,758,765,769,772,775,777,778,786,787,791,794,798,803,806,812,819,823,826,827,829,833,837,840,841,844,845,848,850,851,853,861,862,873,879,894,899,903,904,906,911,914,915,916,920,926,931,942,944,946,948,950,952,955,957,958,959,960,962,965,968,969,979,988,998,1000,1003,1015,1022,1023,1026,1029,1030,1033,1034,1035,1037,1043,1044,1050,1052,1057,1058,1062,1064,1072,1075,1078,1082,1086,1087,1088,1089,1091,1093,1102,1106,1107,1109,1113,1117,1119,1120,1121,1128,1130,1132,1134,1136,1137,1144,1145,1156,1162,1165,1166,1168,1169,1170,1177,1182,1183,1188,1190,1192,1198,1208,1211,1232,1233,1240,1244,1245,1249,1254,1256,1259,1262,1268,1269,1277,1280,1283,1288,1291,1294,1296,1305,1311,1313,1324,1326,1333,1338,1339,1341,1344,1346,1349,1352,1357,1368,1372,1377,1382,1385,1386,1392,1394,1406,1411,1414,1423,1424,1425,1428,1432,1434,1436,1440,1441,1447,1449,1456,1460,1463,1472,1473,1474,1479,1482,1484,1490,1491,1492,1495,1496,1499,1500,1502,1508,1513,1518,1519,1522,1525,1528,1547,1548,1549,1555,1559,1568,1575,1578,1584,1585,1586,1590,1591,1592,1593,1597,1600,1601,1603,1605,1607,1613,1616,1618,1621,1622,1623,1629,1631,1635,1636,1640,1641,1642,1645,1648,1652,1655,1663,1671,1675,1681,1690,1693,1699,1702,1704,1711,1714,1718,1726,1742,1744,1749,1751,1752,1755,1763,1767,1772,1774,1780,1783,1798,1799,1802,1803,1804,1806,1809,1813,1817,1819,1822,1827,1835,1842,1851,1860,1863,1865,1866,1869,1871,1873,1875,1876,1880,1881,1887,1889,1891,1894,1897,1907,1909,1916,1919,1926,1935,1936,1939,1953,1954,1955,1959,1960,1961,1963,1965,1968,1972,1978,1982,1983,1985,1986,1990,1993,1996,2008,2013,2015,2021,2023,2026,2028,2031,2042,2051,2052,2057,2060,2061,2063,2064,2068,2075,2079,2084,2089,2095,2099,2110,2121,2124,2126,2131,2133,2135,2142,2161,2163,2170,2176,2179,2189,2193,2196,2197,2199,2201,2202,2204,2211,2212,2217,2220,2221,2223,2226,2228,2232,2233,2234,2235,2239,2256,2257,2259,2261,2270,2272,2277,2280,2285,2294,2297,2299,2302,2303,2308,2314,2316,2322,2328,2330,2335,2338,2339,2344,2346,2353,2355,2357,2358,2359,2360,2362,2367,2371,2376,2378,2380,2382,2383,2388,2389,2391,2393,2397,2398,2400,2401,2411,2412,2414,2415,2418,2421,2424,2427,2430,2437,2438,2441,2445,2449,2458,2465,2467,2480,2484,2485,2490,2493,2498,2499,2505,2506,2510,2512,2515,2518,2522,2524,2525,2527,2529,2530,2533,2534,2537,2540,2545,2547,2548,2557,2560,2567,2569,2573,2574,2581,2582,2586,2592,2595,2596,2607,2608,2611,2613,2617,2620,2625,2626,2637,2639,2650,2653,2657,2658,2659,2660,2661,2663,2671,2673,2676,2685,2686,2688,2691,2694,2697]},\"outer4_inner2\":{\"fit\":[1,2,3,5,8,9,11,14,15,16,19,20,21,23,25,26,27,28,29,32,33,34,35,36,38,39,40,43,47,48,49,53,56,57,59,60,61,63,64,66,68,70,71,73,74,77,79,80,81,83,84,85,86,88,90,91,92,93,98,100,101,102,104,106,108,110,111,112,115,117,118,121,122,125,127,128,129,130,131,132,133,134,135,136,137,138,140,141,142,143,145,148,150,153,156,158,162,163,165,167,168,170,172,173,174,176,180,181,182,185,186,187,188,189,190,191,192,193,194,195,196,197,200,202,206,208,209,212,214,215,217,218,220,221,222,224,227,228,229,230,234,236,238,239,240,244,245,246,248,249,250,252,254,255,257,258,260,261,262,263,264,265,266,268,273,274,276,282,283,284,287,289,290,294,295,296,297,298,299,300,301,304,306,309,310,312,317,318,320,321,322,324,327,329,331,332,333,334,335,337,338,340,344,346,347,351,354,356,357,358,359,360,362,363,364,366,369,371,374,375,376,380,381,383,385,388,390,391,394,397,400,404,408,410,415,416,417,418,419,420,423,424,427,428,431,432,438,439,441,442,447,452,455,456,457,460,461,463,464,466,467,468,470,471,476,477,478,482,484,491,493,494,497,498,501,502,503,504,509,513,514,515,516,517,518,519,521,523,529,532,533,534,536,538,542,543,546,548,551,552,553,555,556,557,561,562,563,564,566,567,570,571,573,575,577,578,579,581,583,584,585,586,587,588,592,595,597,599,601,602,604,605,608,609,610,615,616,617,618,619,620,621,622,624,625,626,635,636,637,640,641,644,645,646,647,649,653,655,659,660,662,663,667,670,672,674,675,679,682,683,686,687,688,690,691,692,694,695,700,702,708,709,710,711,712,713,714,717,718,721,722,723,727,729,730,732,734,735,738,740,741,743,744,745,746,748,752,753,754,755,756,757,758,760,764,765,766,768,769,772,774,775,777,778,779,781,786,787,788,789,791,794,795,798,801,803,806,809,810,812,819,823,824,826,827,829,831,833,836,837,838,839,840,841,842,844,845,847,848,850,851,852,853,856,859,861,862,865,866,869,873,876,878,879,881,884,885,887,888,889,890,893,894,899,903,904,906,908,909,911,914,915,916,917,918,919,920,922,924,926,927,931,933,934,942,943,944,946,947,948,950,952,955,957,958,959,960,962,965,966,968,969,974,977,978,979,982,983,984,985,988,992,993,997,998,999,1000,1003,1008,1010,1013,1014,1015,1019,1020,1021,1022,1023,1024,1026,1027,1029,1030,1033,1034,1035,1037,1043,1044,1048,1049,1050,1051,1052,1057,1058,1062,1064,1065,1070,1072,1075,1077,1078,1079,1080,1082,1084,1085,1086,1087,1088,1089,1091,1092,1093,1095,1102,1103,1105,1106,1107,1109,1113,1116,1117,1119,1120,1121,1128,1129,1130,1132,1133,1134,1136,1137,1138,1140,1144,1145,1147,1149,1153,1156,1159,1160,1162,1165,1166,1168,1169,1170,1171,1173,1177,1178,1182,1183,1184,1188,1189,1190,1192,1196,1197,1198,1200,1203,1205,1206,1207,1208,1211,1215,1217,1218,1220,1222,1226,1227,1230,1231,1232,1233,1239,1240,1243,1244,1245,1246,1248,1249,1253,1254,1256,1259,1261,1262,1263,1264,1268,1269,1272,1275,1277,1279,1280,1283,1288,1289,1290,1291,1292,1294,1296,1300,1301,1302,1304,1305,1308,1311,1313,1315,1319,1324,1326,1328,1329,1330,1331,1332,1333,1334,1335,1338,1339,1340,1341,1344,1346,1349,1350,1352,1355,1356,1357,1359,1361,1363,1366,1367,1368,1370,1372,1375,1377,1382,1383,1385,1386,1389,1390,1392,1394,1395,1398,1402,1403,1404,1405,1406,1410,1411,1413,1414,1419,1422,1423,1424,1425,1428,1430,1432,1434,1435,1436,1439,1440,1441,1442,1445,1446,1447,1448,1449,1450,1453,1454,1455,1456,1457,1460,1463,1466,1468,1469,1472,1473,1474,1476,1478,1479,1480,1481,1482,1483,1484,1485,1490,1491,1492,1493,1495,1496,1497,1499,1500,1502,1503,1508,1509,1512,1513,1516,1518,1519,1520,1522,1525,1526,1528,1529,1533,1534,1545,1547,1548,1549,1550,1555,1557,1559,1561,1562,1565,1568,1571,1574,1575,1576,1578,1584,1585,1586,1587,1590,1591,1592,1593,1594,1596,1597,1600,1601,1602,1603,1604,1605,1606,1607,1613,1616,1618,1621,1622,1623,1624,1625,1626,1629,1631,1633,1635,1636,1637,1638,1640,1641,1642,1644,1645,1647,1648,1649,1652,1653,1654,1655,1656,1659,1662,1663,1666,1667,1669,1671,1674,1675,1676,1677,1681,1685,1686,1690,1691,1693,1696,1698,1699,1700,1702,1703,1704,1705,1707,1711,1713,1714,1718,1719,1720,1724,1726,1727,1730,1731,1733,1734,1736,1738,1739,1742,1744,1749,1750,1751,1752,1755,1762,1763,1764,1765,1767,1768,1772,1774,1775,1777,1780,1781,1783,1787,1789,1791,1796,1798,1799,1800,1801,1802,1803,1804,1806,1808,1809,1810,1811,1812,1813,1814,1815,1817,1819,1820,1822,1825,1826,1827,1828,1829,1830,1831,1835,1837,1840,1842,1845,1849,1851,1854,1858,1859,1860,1861,1862,1863,1864,1865,1866,1868,1869,1871,1873,1875,1876,1877,1880,1881,1882,1883,1887,1889,1891,1892,1894,1895,1897,1900,1901,1903,1905,1906,1907,1908,1909,1910,1916,1919,1921,1925,1926,1927,1932,1934,1935,1936,1939,1942,1943,1947,1949,1952,1953,1954,1955,1959,1960,1961,1963,1965,1968,1970,1972,1975,1978,1980,1981,1982,1983,1984,1985,1986,1987,1990,1993,1996,1998,2000,2001,2008,2013,2015,2016,2018,2021,2023,2025,2026,2027,2028,2031,2032,2041,2042,2043,2050,2051,2052,2054,2057,2058,2059,2060,2061,2063,2064,2066,2067,2068,2074,2075,2079,2080,2084,2088,2089,2092,2093,2094,2095,2096,2097,2098,2099,2101,2104,2107,2109,2110,2111,2113,2114,2116,2117,2118,2120,2121,2124,2126,2128,2130,2131,2133,2134,2135,2142,2145,2146,2155,2156,2161,2163,2164,2166,2167,2168,2169,2170,2171,2172,2174,2176,2177,2178,2179,2181,2189,2192,2193,2195,2196,2197,2198,2199,2200,2201,2202,2204,2209,2211,2212,2213,2215,2217,2218,2220,2221,2223,2224,2226,2228,2230,2232,2233,2234,2235,2237,2239,2240,2244,2246,2247,2252,2256,2257,2259,2260,2261,2262,2265,2266,2268,2270,2272,2275,2276,2277,2278,2280,2281,2282,2285,2286,2289,2290,2294,2295,2296,2297,2298,2299,2300,2301,2302,2303,2304,2308,2309,2310,2311,2312,2313,2314,2315,2316,2317,2319,2322,2325,2326,2327,2328,2329,2330,2335,2337,2338,2339,2342,2344,2345,2346,2349,2352,2353,2355,2357,2358,2359,2360,2362,2367,2368,2369,2371,2372,2375,2376,2377,2378,2379,2380,2382,2383,2388,2389,2391,2393,2397,2398,2400,2401,2404,2406,2409,2410,2411,2412,2414,2415,2418,2421,2424,2425,2426,2427,2430,2431,2432,2435,2437,2438,2439,2441,2442,2443,2445,2447,2449,2451,2454,2455,2457,2458,2459,2460,2464,2465,2467,2469,2475,2476,2479,2480,2482,2484,2485,2488,2489,2490,2493,2494,2496,2498,2499,2504,2505,2506,2508,2510,2512,2513,2515,2516,2518,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2533,2534,2537,2538,2540,2545,2546,2547,2548,2553,2555,2557,2558,2560,2562,2564,2565,2567,2568,2569,2570,2573,2574,2575,2576,2578,2581,2582,2584,2586,2592,2593,2594,2595,2596,2597,2598,2600,2602,2604,2607,2608,2610,2611,2613,2616,2617,2618,2620,2621,2622,2625,2626,2628,2629,2632,2636,2637,2638,2639,2640,2643,2644,2648,2650,2652,2653,2657,2658,2659,2660,2661,2663,2664,2666,2668,2670,2671,2672,2673,2676,2678,2679,2685,2686,2688,2691,2693,2694,2696,2697],\"valid\":[0,4,10,12,13,18,22,24,30,31,37,44,46,58,67,72,75,76,78,82,89,94,95,99,103,107,113,114,116,120,124,126,139,144,146,152,155,157,159,160,161,169,171,175,178,184,201,203,204,205,210,211,226,232,235,243,251,253,256,269,271,278,285,286,292,307,311,313,314,315,316,319,323,325,328,330,336,341,343,349,350,353,361,365,370,373,378,382,384,386,395,402,403,407,409,411,412,413,421,425,426,429,433,434,437,443,450,451,453,454,459,462,465,473,479,483,486,487,490,492,495,496,505,510,511,520,524,525,526,528,531,540,545,547,549,550,558,560,565,568,572,582,589,591,593,594,596,598,600,603,612,613,627,629,630,631,632,633,634,638,652,654,657,658,664,665,668,669,671,673,676,677,678,681,685,689,693,696,698,706,715,724,725,726,733,737,739,742,747,749,761,762,770,771,773,780,783,784,799,802,804,805,813,815,816,818,825,828,832,846,854,858,860,863,867,870,872,874,882,891,895,897,898,900,905,907,912,925,929,936,937,938,939,949,951,956,963,967,970,971,972,976,980,986,989,991,994,995,1002,1004,1006,1007,1011,1012,1017,1018,1028,1031,1036,1038,1039,1040,1041,1042,1063,1066,1067,1074,1076,1094,1097,1098,1100,1101,1104,1110,1112,1114,1122,1131,1141,1142,1143,1150,1151,1152,1155,1157,1158,1161,1163,1164,1172,1174,1176,1181,1187,1193,1194,1195,1202,1209,1210,1214,1216,1219,1221,1223,1224,1225,1241,1242,1250,1252,1255,1257,1258,1265,1266,1267,1270,1278,1281,1282,1285,1286,1293,1295,1297,1298,1299,1321,1322,1323,1325,1327,1336,1337,1343,1351,1353,1362,1364,1365,1373,1374,1380,1384,1387,1391,1393,1396,1397,1399,1400,1407,1408,1409,1412,1416,1418,1420,1421,1426,1427,1429,1431,1433,1438,1444,1458,1459,1464,1470,1471,1477,1486,1487,1488,1498,1501,1506,1515,1521,1531,1535,1536,1537,1538,1540,1541,1543,1544,1553,1554,1556,1558,1560,1563,1566,1567,1581,1582,1588,1589,1595,1598,1599,1608,1609,1610,1612,1614,1615,1619,1628,1632,1646,1658,1661,1664,1665,1668,1673,1679,1680,1683,1684,1689,1692,1701,1706,1708,1709,1710,1712,1715,1716,1723,1728,1735,1737,1740,1756,1757,1758,1759,1761,1766,1769,1770,1773,1778,1779,1782,1785,1786,1788,1790,1792,1794,1805,1807,1816,1818,1821,1824,1832,1833,1834,1836,1838,1841,1843,1844,1846,1847,1852,1855,1856,1857,1867,1870,1872,1878,1879,1884,1890,1893,1898,1911,1912,1913,1914,1917,1922,1924,1929,1930,1933,1940,1941,1944,1948,1951,1956,1957,1958,1962,1964,1966,1967,1969,1971,1976,1977,1979,1989,1994,1995,1999,2003,2004,2005,2007,2009,2014,2017,2019,2020,2024,2034,2035,2036,2037,2038,2040,2044,2045,2049,2053,2062,2065,2070,2071,2072,2073,2077,2078,2086,2090,2091,2100,2112,2115,2119,2122,2123,2132,2137,2139,2140,2141,2147,2148,2149,2151,2154,2157,2159,2162,2182,2185,2186,2187,2188,2194,2203,2206,2208,2210,2214,2216,2229,2231,2238,2241,2242,2243,2248,2249,2251,2253,2254,2255,2258,2264,2267,2269,2271,2273,2291,2292,2293,2305,2307,2321,2323,2331,2332,2334,2340,2343,2347,2348,2351,2356,2363,2364,2366,2370,2373,2374,2384,2385,2386,2387,2390,2392,2394,2399,2402,2407,2408,2413,2417,2419,2429,2433,2434,2436,2444,2448,2450,2452,2453,2461,2462,2463,2466,2468,2471,2473,2477,2486,2491,2492,2495,2501,2502,2503,2509,2511,2514,2531,2535,2539,2541,2542,2543,2549,2550,2551,2552,2556,2559,2563,2566,2572,2579,2580,2583,2585,2588,2589,2590,2591,2599,2601,2603,2605,2606,2609,2612,2614,2615,2623,2627,2630,2631,2633,2634,2635,2641,2642,2645,2646,2649,2651,2655,2669,2674,2680,2681,2682,2683,2684,2687,2689,2695]},\"final\":{\"fit\":[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,726,727,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767,768,769,770,771,772,773,774,775,776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793,794,795,796,797,798,799,800,801,802,803,804,805,806,807,808,809,810,811,812,813,814,815,816,817,818,819,820,821,822,823,824,825,826,827,828,829,830,831,832,833,834,835,836,837,838,839,840,841,842,843,844,845,846,847,848,849,850,851,852,853,854,855,856,857,858,859,860,861,862,863,864,865,866,867,868,869,870,871,872,873,874,875,876,877,878,879,880,881,882,883,884,885,886,887,888,889,890,891,892,893,894,895,896,897,898,899,900,901,902,903,904,905,906,907,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,925,926,927,928,929,930,931,932,933,934,935,936,937,938,939,940,941,942,943,944,945,946,947,948,949,950,951,952,953,954,955,956,957,958,959,960,961,962,963,964,965,966,967,968,969,970,971,972,973,974,975,976,977,978,979,980,981,982,983,984,985,986,987,988,989,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039,1040,1041,1042,1043,1044,1045,1046,1047,1048,1049,1050,1051,1052,1053,1054,1055,1056,1057,1058,1059,1060,1061,1062,1063,1064,1065,1066,1067,1068,1069,1070,1071,1072,1073,1074,1075,1076,1077,1078,1079,1080,1081,1082,1083,1084,1085,1086,1087,1088,1089,1090,1091,1092,1093,1094,1095,1096,1097,1098,1099,1100,1101,1102,1103,1104,1105,1106,1107,1108,1109,1110,1111,1112,1113,1114,1115,1116,1117,1118,1119,1120,1121,1122,1123,1124,1125,1126,1127,1128,1129,1130,1131,1132,1133,1134,1135,1136,1137,1138,1139,1140,1141,1142,1143,1144,1145,1146,1147,1148,1149,1150,1151,1152,1153,1154,1155,1156,1157,1158,1159,1160,1161,1162,1163,1164,1165,1166,1167,1168,1169,1170,1171,1172,1173,1174,1175,1176,1177,1178,1179,1180,1181,1182,1183,1184,1185,1186,1187,1188,1189,1190,1191,1192,1193,1194,1195,1196,1197,1198,1199,1200,1201,1202,1203,1204,1205,1206,1207,1208,1209,1210,1211,1212,1213,1214,1215,1216,1217,1218,1219,1220,1221,1222,1223,1224,1225,1226,1227,1228,1229,1230,1231,1232,1233,1234,1235,1236,1237,1238,1239,1240,1241,1242,1243,1244,1245,1246,1247,1248,1249,1250,1251,1252,1253,1254,1255,1256,1257,1258,1259,1260,1261,1262,1263,1264,1265,1266,1267,1268,1269,1270,1271,1272,1273,1274,1275,1276,1277,1278,1279,1280,1281,1282,1283,1284,1285,1286,1287,1288,1289,1290,1291,1292,1293,1294,1295,1296,1297,1298,1299,1300,1301,1302,1303,1304,1305,1306,1307,1308,1309,1310,1311,1312,1313,1314,1315,1316,1317,1318,1319,1320,1321,1322,1323,1324,1325,1326,1327,1328,1329,1330,1331,1332,1333,1334,1335,1336,1337,1338,1339,1340,1341,1342,1343,1344,1345,1346,1347,1348,1349,1350,1351,1352,1353,1354,1355,1356,1357,1358,1359,1360,1361,1362,1363,1364,1365,1366,1367,1368,1369,1370,1371,1372,1373,1374,1375,1376,1377,1378,1379,1380,1381,1382,1383,1384,1385,1386,1387,1388,1389,1390,1391,1392,1393,1394,1395,1396,1397,1398,1399,1400,1401,1402,1403,1404,1405,1406,1407,1408,1409,1410,1411,1412,1413,1414,1415,1416,1417,1418,1419,1420,1421,1422,1423,1424,1425,1426,1427,1428,1429,1430,1431,1432,1433,1434,1435,1436,1437,1438,1439,1440,1441,1442,1443,1444,1445,1446,1447,1448,1449,1450,1451,1452,1453,1454,1455,1456,1457,1458,1459,1460,1461,1462,1463,1464,1465,1466,1467,1468,1469,1470,1471,1472,1473,1474,1475,1476,1477,1478,1479,1480,1481,1482,1483,1484,1485,1486,1487,1488,1489,1490,1491,1492,1493,1494,1495,1496,1497,1498,1499,1500,1501,1502,1503,1504,1505,1506,1507,1508,1509,1510,1511,1512,1513,1514,1515,1516,1517,1518,1519,1520,1521,1522,1523,1524,1525,1526,1527,1528,1529,1530,1531,1532,1533,1534,1535,1536,1537,1538,1539,1540,1541,1542,1543,1544,1545,1546,1547,1548,1549,1550,1551,1552,1553,1554,1555,1556,1557,1558,1559,1560,1561,1562,1563,1564,1565,1566,1567,1568,1569,1570,1571,1572,1573,1574,1575,1576,1577,1578,1579,1580,1581,1582,1583,1584,1585,1586,1587,1588,1589,1590,1591,1592,1593,1594,1595,1596,1597,1598,1599,1600,1601,1602,1603,1604,1605,1606,1607,1608,1609,1610,1611,1612,1613,1614,1615,1616,1617,1618,1619,1620,1621,1622,1623,1624,1625,1626,1627,1628,1629,1630,1631,1632,1633,1634,1635,1636,1637,1638,1639,1640,1641,1642,1643,1644,1645,1646,1647,1648,1649,1650,1651,1652,1653,1654,1655,1656,1657,1658,1659,1660,1661,1662,1663,1664,1665,1666,1667,1668,1669,1670,1671,1672,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682,1683,1684,1685,1686,1687,1688,1689,1690,1691,1692,1693,1694,1695,1696,1697,1698,1699,1700,1701,1702,1703,1704,1705,1706,1707,1708,1709,1710,1711,1712,1713,1714,1715,1716,1717,1718,1719,1720,1721,1722,1723,1724,1725,1726,1727,1728,1729,1730,1731,1732,1733,1734,1735,1736,1737,1738,1739,1740,1741,1742,1743,1744,1745,1746,1747,1748,1749,1750,1751,1752,1753,1754,1755,1756,1757,1758,1759,1760,1761,1762,1763,1764,1765,1766,1767,1768,1769,1770,1771,1772,1773,1774,1775,1776,1777,1778,1779,1780,1781,1782,1783,1784,1785,1786,1787,1788,1789,1790,1791,1792,1793,1794,1795,1796,1797,1798,1799,1800,1801,1802,1803,1804,1805,1806,1807,1808,1809,1810,1811,1812,1813,1814,1815,1816,1817,1818,1819,1820,1821,1822,1823,1824,1825,1826,1827,1828,1829,1830,1831,1832,1833,1834,1835,1836,1837,1838,1839,1840,1841,1842,1843,1844,1845,1846,1847,1848,1849,1850,1851,1852,1853,1854,1855,1856,1857,1858,1859,1860,1861,1862,1863,1864,1865,1866,1867,1868,1869,1870,1871,1872,1873,1874,1875,1876,1877,1878,1879,1880,1881,1882,1883,1884,1885,1886,1887,1888,1889,1890,1891,1892,1893,1894,1895,1896,1897,1898,1899,1900,1901,1902,1903,1904,1905,1906,1907,1908,1909,1910,1911,1912,1913,1914,1915,1916,1917,1918,1919,1920,1921,1922,1923,1924,1925,1926,1927,1928,1929,1930,1931,1932,1933,1934,1935,1936,1937,1938,1939,1940,1941,1942,1943,1944,1945,1946,1947,1948,1949,1950,1951,1952,1953,1954,1955,1956,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1977,1978,1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057,2058,2059,2060,2061,2062,2063,2064,2065,2066,2067,2068,2069,2070,2071,2072,2073,2074,2075,2076,2077,2078,2079,2080,2081,2082,2083,2084,2085,2086,2087,2088,2089,2090,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100,2101,2102,2103,2104,2105,2106,2107,2108,2109,2110,2111,2112,2113,2114,2115,2116,2117,2118,2119,2120,2121,2122,2123,2124,2125,2126,2127,2128,2129,2130,2131,2132,2133,2134,2135,2136,2137,2138,2139,2140,2141,2142,2143,2144,2145,2146,2147,2148,2149,2150,2151,2152,2153,2154,2155,2156,2157,2158,2159,2160,2161,2162,2163,2164,2165,2166,2167,2168,2169,2170,2171,2172,2173,2174,2175,2176,2177,2178,2179,2180,2181,2182,2183,2184,2185,2186,2187,2188,2189,2190,2191,2192,2193,2194,2195,2196,2197,2198,2199,2200,2201,2202,2203,2204,2205,2206,2207,2208,2209,2210,2211,2212,2213,2214,2215,2216,2217,2218,2219,2220,2221,2222,2223,2224,2225,2226,2227,2228,2229,2230,2231,2232,2233,2234,2235,2236,2237,2238,2239,2240,2241,2242,2243,2244,2245,2246,2247,2248,2249,2250,2251,2252,2253,2254,2255,2256,2257,2258,2259,2260,2261,2262,2263,2264,2265,2266,2267,2268,2269,2270,2271,2272,2273,2274,2275,2276,2277,2278,2279,2280,2281,2282,2283,2284,2285,2286,2287,2288,2289,2290,2291,2292,2293,2294,2295,2296,2297,2298,2299,2300,2301,2302,2303,2304,2305,2306,2307,2308,2309,2310,2311,2312,2313,2314,2315,2316,2317,2318,2319,2320,2321,2322,2323,2324,2325,2326,2327,2328,2329,2330,2331,2332,2333,2334,2335,2336,2337,2338,2339,2340,2341,2342,2343,2344,2345,2346,2347,2348,2349,2350,2351,2352,2353,2354,2355,2356,2357,2358,2359,2360,2361,2362,2363,2364,2365,2366,2367,2368,2369,2370,2371,2372,2373,2374,2375,2376,2377,2378,2379,2380,2381,2382,2383,2384,2385,2386,2387,2388,2389,2390,2391,2392,2393,2394,2395,2396,2397,2398,2399,2400,2401,2402,2403,2404,2405,2406,2407,2408,2409,2410,2411,2412,2413,2414,2415,2416,2417,2418,2419,2420,2421,2422,2423,2424,2425,2426,2427,2428,2429,2430,2431,2432,2433,2434,2435,2436,2437,2438,2439,2440,2441,2442,2443,2444,2445,2446,2447,2448,2449,2450,2451,2452,2453,2454,2455,2456,2457,2458,2459,2460,2461,2462,2463,2464,2465,2466,2467,2468,2469,2470,2471,2472,2473,2474,2475,2476,2477,2478,2479,2480,2481,2482,2483,2484,2485,2486,2487,2488,2489,2490,2491,2492,2493,2494,2495,2496,2497,2498,2499,2500,2501,2502,2503,2504,2505,2506,2507,2508,2509,2510,2511,2512,2513,2514,2515,2516,2517,2518,2519,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2531,2532,2533,2534,2535,2536,2537,2538,2539,2540,2541,2542,2543,2544,2545,2546,2547,2548,2549,2550,2551,2552,2553,2554,2555,2556,2557,2558,2559,2560,2561,2562,2563,2564,2565,2566,2567,2568,2569,2570,2571,2572,2573,2574,2575,2576,2577,2578,2579,2580,2581,2582,2583,2584,2585,2586,2587,2588,2589,2590,2591,2592,2593,2594,2595,2596,2597,2598,2599,2600,2601,2602,2603,2604,2605,2606,2607,2608,2609,2610,2611,2612,2613,2614,2615,2616,2617,2618,2619,2620,2621,2622,2623,2624,2625,2626,2627,2628,2629,2630,2631,2632,2633,2634,2635,2636,2637,2638,2639,2640,2641,2642,2643,2644,2645,2646,2647,2648,2649,2650,2651,2652,2653,2654,2655,2656,2657,2658,2659,2660,2661,2662,2663,2664,2665,2666,2667,2668,2669,2670,2671,2672,2673,2674,2675,2676,2677,2678,2679,2680,2681,2682,2683,2684,2685,2686,2687,2688,2689,2690,2691,2692,2693,2694,2695,2696,2697,2698],\"valid\":[]}},\"outer_inner\":{\"outer0\":[\"outer0_inner0\",\"outer0_inner1\",\"outer0_inner2\"],\"outer1\":[\"outer1_inner0\",\"outer1_inner1\",\"outer1_inner2\"],\"outer2\":[\"outer2_inner0\",\"outer2_inner1\",\"outer2_inner2\"],\"outer3\":[\"outer3_inner0\",\"outer3_inner1\",\"outer3_inner2\"],\"outer4\":[\"outer4_inner0\",\"outer4_inner1\",\"outer4_inner2\"]}},\"jobs\":[\"outer0\",\"outer1\",\"outer2\",\"outer3\",\"outer4\",\"final\"],\"configuration\":{\"effective_batch\":18,\"gradient_clip\":1.0,\"pool_layers\":[2,4,6,8,12,16,20,24],\"alpha\":32,\"r\":16,\"public_training_reproduction\":false,\"dtype\":\"Frozen FP16 base, FP32 trainables, CUDA AMP FP16 with GradScaler\",\"head_dropout\":0.1,\"checkpoint_for_submission\":true,\"epochs_for_future_timing_projection\":3,\"objective\":\"CE(plain(z),y)+CE(centered(z-mean(z over this3answer prompt group)),y)\",\"accumulation\":6,\"microbatch\":3,\"base\":\"qwen-lm/qwen-3/Transformers/8b-base/1\",\"sampler\":\"Each epoch shuffles fitting answers within each prompt, covers all rows through distinct-answer triples, pads incomplete triples and the final accumulation batch; no validation labels.\",\"head_lr\":0.001,\"optimizer\":\"AdamW\",\"adapter_lr\":0.0001,\"gradient_checkpointing\":true,\"weight_decay\":0.01,\"lora_dropout\":0.05,\"layers\":24,\"max_length\":448},\"amp\":{\"growth_interval\":2000,\"minimum_scale\":1.0,\"max_overflows_per_update\":3,\"accumulation\":6,\"initial_scale\":256.0},\"dispatch_ready\":true,\"authorized_diagnostic_smoke_required\":true,\"smoke_evidence\":{\"base_config_sha256\":\"3bd01d7ad7a2e203ecbbe84e24087a51c6d2a108ee4bcc42d0016bf49564983a\",\"successful_updates\":12,\"total_retry_seconds\":0,\"worker_source_sha256\":\"da4e896784e9452b7289386649dd9e03fe2477371cf3c1072b96d773ff2eaca2\",\"amp_source_sha256\":\"0af0a6f7c139e02eac9c8e8c087d81fb34093c76c09758f34033c683575fa32d\",\"report_sha256\":\"a5b701b2dd25722aa14eab912bf931e5c607ab5acbfb1f41b1827948e7a28bd3\",\"script_version_id\":358830800,\"core_versions\":{\"torch\":\"2.11.0+cu128\",\"transformers\":\"5.16.1\",\"peft\":\"0.20.0\",\"accelerate\":\"1.14.0\",\"safetensors\":\"0.8.0\",\"numpy\":\"2.1.3\",\"pandas\":\"2.3.3\"},\"optional_audit_sha256\":\"a7eb23fae319bb71517016eeb99d8a811d44cba8198a84154c853133ed871f99\",\"estimated_two_worker_training_hours\":3.224296659674993,\"median_successful_update_seconds\":9.920912798999979,\"verified_run_audit_sha256\":\"ce510845983862690def24f4d78fdbab3ab76adb1795128807a5761117421eb6\",\"notebook_ref\":\"lightningv08/ntl-lora24-feasibility-smoke\",\"optional_helper_sha256\":\"4e743a3f9f4b11391431a5e92fa3cc842438ee717468855096d1bb41abb26f00\",\"version\":2},\"reference_hashes\":{\"gemma_tail_half_current_calibrated_recipe_report.json\":\"0570c4e2e1abdec011ab5f943cf40284236b4c9b25464128212fc29f678060c8\",\"gemma_tail_half_current_calibrated_recipe_oof.npz\":\"20e415de0ce4fb9746468ea567e0983bbcd8aad088610972cdab786608fc01ec\"},\"fixed_mixture_weights\":[0.05,0.1,0.2],\"lora_plain_center_log_mix\":0.5,\"recovered_contexts\":[1,3,6],\"grouping\":\"Lexical TF-IDF average linkage cosine distance.9; singleton plain fallback\",\"schedule\":\"Three fixed epochs; prompt-wise triples cover all fitting answers, pad incomplete triples/accumulation;10%warmup then cosine decay; scheduler advances only successful updates.\",\"maximum_monitor_hours\":10,\"public_adapters_used\":false,\"inner_oof_weight_tuning\":false,\"submission_created\":false}")
Path('/kaggle/working/training_spec.json').write_text(json.dumps(spec,indent=2))
Path('/kaggle/working/source_audit.json').write_text(json.dumps({name:hashlib.sha256(source.encode()).hexdigest() for name,source in sources.items()},indent=2))

In [ ]:
import subprocess,sys
assert spec['dispatch_ready'],'Rebuild from the actual successful v2 smoke and both verified evidence audits before dispatch.'
assert hashlib.sha256(Path('optional_torchao.py').read_bytes()).hexdigest()==spec['smoke_evidence']['optional_helper_sha256']
preflight=subprocess.run([sys.executable,'optional_torchao.py'])
assert preflight.returncode==0
compat=json.loads(Path('optional_torchao_audit.json').read_text())
assert compat['status'] in ('REMOVED_INCOMPATIBLE_OPTIONAL_TORCHAO','NO_CHANGE_REQUIRED')
assert compat['core_versions_unchanged'] and compat['core_versions_after']==spec['smoke_evidence']['core_versions']
assert compat['packages_installed']==[] and set(compat['packages_removed'])<=set(['torchao'])

In [ ]:
import os,sys,time,subprocess
import torch,peft,transformers
assert spec['dispatch_ready'],'Rebuild from the successful measured smoke report before dispatch.'
assert torch.cuda.device_count()==2 and not spec['public_adapters_used']
assert spec['smoke_evidence']['amp_source_sha256']==hashlib.sha256(Path('amp_update.py').read_bytes()).hexdigest()
processes=[]
for worker in (0,1):
    env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']=str(worker)
    handle=Path(f'lora24_worker{worker}.log').open('w')
    process=subprocess.Popen([sys.executable,'train_lora24.py','--worker',str(worker)],env=env,stdout=handle,stderr=subprocess.STDOUT)
    processes.append((process,handle))
seen=[0,0]
def monitor_workers(seconds=1800):
    deadline=time.monotonic()+seconds
    while any(p.poll() is None for p,_ in processes) and time.monotonic()<deadline:
        for worker,(process,_) in enumerate(processes):
            text=Path(f'lora24_worker{worker}.log').read_text(errors='replace')
            lines=text.splitlines();fresh=lines[seen[worker]:];seen[worker]=len(lines)
            for line in fresh:
                if line.startswith('{') or line.startswith('JOB COMPLETE') or 'AMP' in line:print(line,flush=True)
            if process.poll() is not None and process.returncode:
                for other,_ in processes:
                    if other.poll() is None:other.terminate()
                print(text[-16000:],flush=True)
                raise RuntimeError(f'Worker{worker} failed with code{process.returncode}')
        time.sleep(10)
    print('Worker states:',[p.poll() for p,_ in processes],flush=True)

## Independent training workers

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

In [ ]:
monitor_workers(1800)

## Fixed-mixture held-out comparisons

In [ ]:
if any(p.poll() is None for p,_ in processes):
    for p,_ in processes:
        if p.poll() is None:p.terminate()
    raise RuntimeError('Ten-hour monitoring budget exceeded; preserve partial artifacts and revise the measured estimate.')
for worker,(p,handle) in enumerate(processes):
    handle.close()
    if p.returncode:print(Path(f'lora24_worker{worker}.log').read_text(errors='replace')[-16000:])
    assert p.returncode==0
    status=json.loads(Path(f'lora24/worker{worker}_status.json').read_text())
    assert status['status']=='COMPLETE'
assert all(Path('lora24',name,'COMPLETE.json').exists() for name in spec['jobs'])
subprocess.run([sys.executable,'evaluate_lora24.py'],check=True)